# KAMBA++ Standalone Experiments
This notebook contains the complete finalized implementation. It does not require `kambapp_experiment.py`. Run cells in order.


## 1. Environment and GPU verification


In [1]:
from pathlib import Path
import platform
print('Project:', Path.cwd())
print('Python:', platform.python_version())


Project: <PROJECT_ROOT>
Python: 3.9.23


## 2. Complete KAMBA++ implementation
Run this cell once. It includes automatic recovery of the known headerless DAPT2020 private-Thursday CSV. The cell is collapsed for readability.


In [2]:
"""Clean KAMBA++ experiment pipeline.

Implements the frozen architecture:
input projection -> EDyT -> parallel SKAN/SwiGLU/DSSSM -> adaptive fusion.
No code or results from earlier KAMBA++ experiments are reused.
"""
from __future__ import annotations

import argparse
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset


@dataclass
class Config:
    dataset_name: str = "DAPT2020"
    data_path: str = "./data/DAPT2020"
    output_dir: str = "./outputs/dapt2020"
    label_column: str = "Stage"
    benign_values: Tuple[str, ...] = ("Benign", "BENIGN", "Normal", "normal", "0")
    group_column: Optional[str] = "Flow ID"
    timestamp_column: Optional[str] = "Timestamp"
    drop_columns: Tuple[str, ...] = (
        "Flow ID", "Src IP", "Dst IP", "Source IP", "Destination IP",
        "Src Port", "Dst Port", "Timestamp", "Activity", "Stage",
    )
    sequence_length: int = 8
    stride: int = 8
    min_group_size: int = 2
    train_fraction: float = 0.70
    val_fraction: float = 0.15
    test_fraction: float = 0.15
    hidden_dim: int = 128
    skan_dim: int = 96
    swiglu_dim: int = 96
    state_dim: int = 96
    fusion_dim: int = 96
    num_basis: int = 8
    basis_min: float = -3.0
    basis_max: float = 3.0
    gate_temperature: float = 0.20
    dropout: float = 0.20
    batch_size: int = 256
    epochs: int = 60
    patience: int = 10
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    sparse_regularizer: float = 1e-5
    gradient_clip: float = 1.0
    num_workers: int = 0
    seeds: Tuple[int, ...] = (13, 27, 41, 55, 69)
    threshold_metric: str = "f1"
    use_amp: bool = True
    deterministic: bool = True
    device: str = "auto"
    tsne_max_samples: int = 5000
    tsne_perplexity: float = 30.0


def load_config(path: str) -> Config:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    tuple_fields = {"benign_values", "drop_columns", "seeds"}
    for key in tuple_fields:
        if key in raw:
            raw[key] = tuple(raw[key])
    return Config(**raw)


def save_json(obj: object, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def journal_style() -> None:
    plt.rcParams.update({
        "font.family": "serif", "font.size": 11, "axes.labelsize": 11,
        "axes.titlesize": 12, "axes.linewidth": 0.9, "legend.fontsize": 10,
        "figure.dpi": 160, "savefig.dpi": 400,
    })


def stratified_sample_indices(y: np.ndarray, maximum: int, seed: int) -> np.ndarray:
    if len(y) <= maximum:
        return np.arange(len(y))
    rng = np.random.default_rng(seed)
    selected = []
    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        quota = max(2, int(round(maximum * len(cls_idx) / len(y))))
        selected.extend(rng.choice(cls_idx, min(quota, len(cls_idx)), replace=False).tolist())
    selected = np.asarray(selected, dtype=np.int64)
    if len(selected) > maximum:
        selected = rng.choice(selected, maximum, replace=False)
    rng.shuffle(selected)
    return selected


def plot_tsne(representation: np.ndarray, labels: np.ndarray, path: Path,
              title: str, cfg: Config, seed: int) -> pd.DataFrame:
    idx = stratified_sample_indices(labels, cfg.tsne_max_samples, seed)
    x, y = representation[idx], labels[idx]
    if len(x) < 4 or len(np.unique(y)) < 2:
        print(f"Skipping t-SNE '{title}': insufficient samples/classes.")
        return pd.DataFrame(columns=["sample_index", "label", "tsne_x", "tsne_y"])
    perplexity = min(cfg.tsne_perplexity, max(2.0, (len(x) - 1) / 3.0))
    points = TSNE(n_components=2, perplexity=perplexity, init="pca",
                  learning_rate="auto", max_iter=1000,
                  random_state=seed).fit_transform(x)
    journal_style()
    fig, ax = plt.subplots(figsize=(6.2, 5.0))
    for cls, color, name in [(0, "#2C7BB6", "Benign"), (1, "#D7191C", "APT/Attack")]:
        mask = y == cls
        ax.scatter(points[mask, 0], points[mask, 1], s=12, alpha=0.68,
                   c=color, label=name, edgecolors="none")
    ax.set(title=title, xlabel="t-SNE dimension 1", ylabel="t-SNE dimension 2")
    ax.grid(True, linestyle="--", linewidth=0.45, alpha=0.35)
    ax.legend(frameon=True)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return pd.DataFrame({
        "sample_index": idx,
        "label": y,
        "tsne_x": points[:, 0],
        "tsne_y": points[:, 1],
    })


def plot_confusion(y: np.ndarray, p: np.ndarray, threshold: float,
                   path: Path, title: str) -> None:
    pred = (p >= threshold).astype(np.int64)
    cm = confusion_matrix(y, pred, labels=[0, 1])
    pct = 100.0 * cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    journal_style()
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    image = ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            color = "white" if pct[i, j] > 55 else "black"
            ax.text(j, i, f"{cm[i, j]:,}\n({pct[i, j]:.2f}%)",
                    ha="center", va="center", color=color, fontweight="bold")
    ax.set_xticks([0, 1], ["Benign", "APT/Attack"])
    ax.set_yticks([0, 1], ["Benign", "APT/Attack"])
    ax.set(xlabel="Predicted class", ylabel="True class", title=title)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Row percentage (%)")
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def plot_training_curves(history: pd.DataFrame, path: Path, title: str) -> None:
    journal_style()
    fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.8))
    axes[0].plot(history["epoch"], history["train_loss"], color="#1F77B4", linewidth=2)
    axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Loss")
    axes[1].plot(history["epoch"], history["auprc"], label="Validation AUPRC", color="#D95F02", linewidth=2)
    axes[1].plot(history["epoch"], history["f1"], label="Validation F1", color="#1B9E77", linewidth=2)
    axes[1].set(title="Validation performance", xlabel="Epoch")
    axes[1].legend()
    axes[2].plot(history["epoch"], history["learning_rate"], color="#6A3D9A", linewidth=2)
    axes[2].set(title="Learning-rate schedule", xlabel="Epoch", ylabel="Learning rate")
    for ax in axes:
        ax.grid(True, linestyle="--", linewidth=0.45, alpha=0.35)
    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def choose_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def read_csv_collection(path: str, label_column: Optional[str] = None) -> pd.DataFrame:
    p = Path(path)
    files = [p] if p.is_file() else sorted(p.rglob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found under {p.resolve()}")

    # Locate a canonical header. Some DAPT2020 distributions contain one
    # headerless CSV; its first traffic record must not be treated as column names.
    canonical_columns: Optional[List[str]] = None
    if label_column:
        for file in files:
            header = pd.read_csv(file, nrows=0, encoding="utf-8-sig", low_memory=False)
            cleaned = [str(c).strip().lstrip("\ufeff") for c in header.columns]
            if label_column in cleaned:
                canonical_columns = cleaned
                break

    frames = []
    for file in files:
        header = pd.read_csv(file, nrows=0, encoding="utf-8-sig", low_memory=False)
        cleaned_header = [str(c).strip().lstrip("\ufeff") for c in header.columns]
        has_expected_header = label_column is None or label_column in cleaned_header

        if has_expected_header:
            frame = pd.read_csv(file, encoding="utf-8-sig", low_memory=False)
            frame.columns = [str(c).strip().lstrip("\ufeff") for c in frame.columns]
        else:
            if canonical_columns is None:
                raise ValueError(
                    f"Could not recover the headerless CSV {file.name}: "
                    "no canonical header was found."
                )
            frame = pd.read_csv(file, header=None, encoding="utf-8-sig", low_memory=False)
            if frame.shape[1] != len(canonical_columns):
                raise ValueError(
                    f"Headerless CSV {file.name} has {frame.shape[1]} columns; "
                    f"the canonical schema has {len(canonical_columns)}."
                )
            frame.columns = canonical_columns
            print(f"Recovered canonical header for: {file.name}")

        frame["__source_file__"] = file.name
        frame["__source_row__"] = np.arange(len(frame), dtype=np.int64)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False)


def clean_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)
    for col in out.columns:
        if not pd.api.types.is_numeric_dtype(out[col]):
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def make_binary_labels(series: pd.Series, benign_values: Sequence[str]) -> np.ndarray:
    benign = {str(x).strip().lower() for x in benign_values}
    values = series.astype(str).str.strip().str.lower()
    return (~values.isin(benign)).astype(np.int64).to_numpy()


def derive_groups(df: pd.DataFrame, cfg: Config) -> np.ndarray:
    if cfg.group_column and cfg.group_column in df.columns:
        group = df[cfg.group_column].astype(str).fillna("missing")
    else:
        group = df["__source_file__"].astype(str)
    return group.to_numpy()


def group_stratified_split(
    groups: np.ndarray,
    labels: np.ndarray,
    cfg: Config,
    seed: int,
) -> Dict[str, np.ndarray]:
    unique_groups, inv = np.unique(groups, return_inverse=True)
    group_labels = np.zeros(len(unique_groups), dtype=np.int64)
    for i in range(len(unique_groups)):
        group_labels[i] = int(labels[inv == i].max())

    rng = np.random.default_rng(seed)
    train_groups, val_groups, test_groups = [], [], []
    present_classes = np.unique(group_labels)
    for cls in present_classes:
        cls_groups = unique_groups[group_labels == cls].copy()
        rng.shuffle(cls_groups)
        n = len(cls_groups)
        n_train = max(1, int(round(cfg.train_fraction * n)))
        n_val = max(1, int(round(cfg.val_fraction * n))) if n >= 3 else 0
        if n_train + n_val >= n:
            n_train = max(1, n - 2) if n >= 3 else max(1, n - 1)
            n_val = 1 if n >= 3 else 0
        train_groups.extend(cls_groups[:n_train])
        val_groups.extend(cls_groups[n_train:n_train + n_val])
        test_groups.extend(cls_groups[n_train + n_val:])

    split = {
        "train": np.flatnonzero(np.isin(groups, train_groups)),
        "val": np.flatnonzero(np.isin(groups, val_groups)),
        "test": np.flatnonzero(np.isin(groups, test_groups)),
    }
    for name, idx in split.items():
        if len(idx) == 0:
            raise ValueError(f"Empty {name} split. Use a group column with more unique groups.")
    assert not (set(groups[split["train"]]) & set(groups[split["val"]]))
    assert not (set(groups[split["train"]]) & set(groups[split["test"]]))
    assert not (set(groups[split["val"]]) & set(groups[split["test"]]))
    return split


def prepare_tabular_data(cfg: Config, seed: int = 13) -> Dict[str, object]:
    df = read_csv_collection(cfg.data_path, cfg.label_column)
    if cfg.label_column not in df.columns:
        raise KeyError(f"Label column '{cfg.label_column}' not found. Available: {list(df.columns)}")
    labels = make_binary_labels(df[cfg.label_column], cfg.benign_values)
    groups = derive_groups(df, cfg)

    if cfg.timestamp_column and cfg.timestamp_column in df.columns:
        timestamp = pd.to_datetime(
            df[cfg.timestamp_column],
            format="mixed",
            dayfirst=True,
            errors="coerce",
        )
        df = df.assign(__sort_time__=timestamp)
    else:
        df = df.assign(__sort_time__=df["__source_row__"])

    forbidden = set(cfg.drop_columns) | {
        cfg.label_column, "__source_file__", "__source_row__", "__sort_time__"
    }
    feature_cols = [c for c in df.columns if c not in forbidden]
    features = clean_numeric_frame(df[feature_cols])
    nonempty = features.notna().any(axis=0)
    features = features.loc[:, nonempty]
    feature_cols = list(features.columns)

    split = group_stratified_split(groups, labels, cfg, seed)
    train_frame = features.iloc[split["train"]]
    medians = train_frame.median(axis=0).fillna(0.0)
    features = features.fillna(medians)
    variance = features.iloc[split["train"]].var(axis=0)
    keep = variance[variance > 0].index.tolist()
    features = features[keep]
    feature_cols = keep

    scaler = StandardScaler()
    scaler.fit(features.iloc[split["train"]].to_numpy(np.float32))
    x = scaler.transform(features.to_numpy(np.float32)).astype(np.float32)

    result_dir = Path(cfg.output_dir) / "prepared"
    result_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(scaler, result_dir / "scaler.joblib")
    save_json({"feature_columns": feature_cols, "medians": medians[feature_cols].to_dict()}, result_dir / "schema.json")
    save_json({k: v.tolist() for k, v in split.items()}, result_dir / "row_splits.json")

    return {
        "frame": df,
        "x": x,
        "y": labels,
        "groups": groups,
        "split": split,
        "feature_columns": feature_cols,
    }


def make_windows(
    frame: pd.DataFrame,
    x: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    row_indices: np.ndarray,
    length: int,
    stride: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    allowed = set(row_indices.tolist())
    sequences, targets, sequence_groups = [], [], []
    for group in np.unique(groups[row_indices]):
        idx = np.flatnonzero(groups == group)
        idx = np.array([i for i in idx if i in allowed], dtype=np.int64)
        if len(idx) == 0:
            continue
        order = np.argsort(frame.iloc[idx]["__sort_time__"].to_numpy(), kind="stable")
        idx = idx[order]
        if len(idx) < length:
            pad = np.repeat(idx[:1], length - len(idx))
            win = np.concatenate([pad, idx])
            sequences.append(x[win])
            targets.append(int(y[idx].max()))
            sequence_groups.append(group)
            continue
        for start in range(0, len(idx) - length + 1, stride):
            win = idx[start:start + length]
            sequences.append(x[win])
            targets.append(int(y[win].max()))
            sequence_groups.append(group)
    if not sequences:
        raise ValueError("No sequence windows were created.")
    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(targets, dtype=np.int64),
        np.asarray(sequence_groups),
    )


class SequenceDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = torch.from_numpy(x)
        self.y = torch.from_numpy(y)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, index: int):
        return self.x[index], self.y[index]


class EDyT(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.alpha = nn.Linear(dim, dim)
        self.beta = nn.Linear(dim, dim)
        self.gamma = nn.Linear(dim, dim)

    def forward(self, h: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        descriptor = h.mean(dim=1)
        alpha = F.softplus(self.alpha(descriptor)) + 1e-4
        beta = self.beta(descriptor)
        gamma = torch.sigmoid(self.gamma(descriptor))
        transformed = torch.tanh(alpha[:, None, :] * h + beta[:, None, :])
        out = gamma[:, None, :] * transformed + (1.0 - gamma[:, None, :]) * h
        return out, descriptor


class AdaptiveController(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim // 2), nn.SiLU(), nn.Linear(dim // 2, 3))

    def forward(self, descriptor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        raw = self.net(descriptor)
        sparse_threshold = torch.sigmoid(raw[:, 0])
        delta = F.softplus(raw[:, 1]) + 1e-4
        tau = F.softplus(raw[:, 2]) + 1e-4
        return sparse_threshold, delta, tau


class SparseKAN(nn.Module):
    """KAN edge functions implemented with Gaussian basis functions and soft masks."""
    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        num_basis: int,
        basis_min: float,
        basis_max: float,
        temperature: float,
    ):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_basis = num_basis
        self.temperature = temperature
        centers = torch.linspace(basis_min, basis_max, num_basis)
        self.register_buffer("centers", centers)
        spacing = (basis_max - basis_min) / max(1, num_basis - 1)
        self.log_width = nn.Parameter(torch.tensor(math.log(max(spacing, 1e-2))))
        self.linear = nn.Parameter(torch.empty(out_dim, in_dim))
        self.coeff = nn.Parameter(torch.empty(out_dim, in_dim, num_basis))
        self.edge_score = nn.Parameter(torch.zeros(out_dim, in_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.linear)
        nn.init.normal_(self.coeff, std=0.02)

    def forward(self, x: torch.Tensor, threshold: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        width = self.log_width.exp().clamp_min(1e-3)
        basis = torch.exp(-0.5 * ((x.unsqueeze(-1) - self.centers) / width) ** 2)
        nonlinear = torch.einsum("btik,oik->btoi", basis, self.coeff)
        edge_value = nonlinear + x.unsqueeze(2) * self.linear.unsqueeze(0).unsqueeze(0)
        edge_strength = torch.sigmoid(self.edge_score)
        mask = torch.sigmoid(
            (edge_strength.unsqueeze(0) - threshold[:, None, None])
            / self.temperature
        )
        out = (edge_value * mask[:, None, :, :]).sum(dim=-1) + self.bias
        return out, mask


class SwiGLUBranch(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float):
        super().__init__()
        self.w1 = nn.Linear(in_dim, out_dim)
        self.w2 = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(F.silu(self.w1(x)) * self.w2(x))


class DSSSM(nn.Module):
    """Dual-gated selective SSM with stable diagonal continuous dynamics."""
    def __init__(self, in_dim: int, state_dim: int, dropout: float):
        super().__init__()
        self.state_dim = state_dim
        self.a_log = nn.Parameter(torch.zeros(state_dim))
        self.b_proj = nn.Linear(in_dim, state_dim)
        self.r_x = nn.Linear(in_dim, state_dim)
        self.r_s = nn.Linear(state_dim, state_dim, bias=False)
        self.u_x = nn.Linear(in_dim, state_dim)
        self.u_s = nn.Linear(state_dim, state_dim, bias=False)
        self.c_proj = nn.Linear(state_dim, state_dim)
        self.d_proj = nn.Linear(in_dim, state_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, delta: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        batch, steps, _ = x.shape
        state = x.new_zeros(batch, self.state_dim)
        outputs = []
        a_cont = -(F.softplus(self.a_log) + 1e-3)
        for t in range(steps):
            xt = x[:, t, :]
            dt = (delta / (1.0 + tau)).unsqueeze(-1)
            a_bar = torch.exp(dt * a_cont.unsqueeze(0))
            b_bar = (1.0 - a_bar) * self.b_proj(xt)
            candidate = a_bar * state + b_bar
            retention = torch.sigmoid(self.r_x(xt) + self.r_s(state))
            update = torch.sigmoid(self.u_x(xt) + self.u_s(state))
            state = retention * state + update * candidate
            outputs.append(self.c_proj(state) + self.d_proj(xt))
        return self.dropout(torch.stack(outputs, dim=1))


class AdaptiveFusion(nn.Module):
    def __init__(self, dims: Sequence[int], fusion_dim: int, dropout: float):
        super().__init__()
        self.projections = nn.ModuleList([nn.Linear(d, fusion_dim) for d in dims])
        self.weight_net = nn.Sequential(
            nn.Linear(sum(dims), fusion_dim), nn.SiLU(), nn.Dropout(dropout), nn.Linear(fusion_dim, 3)
        )

    def forward(self, branches: Sequence[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        weights = torch.softmax(self.weight_net(torch.cat(branches, dim=-1)), dim=-1)
        projected = [layer(x) for layer, x in zip(self.projections, branches)]
        fused = torch.cat([weights[:, i:i + 1] * projected[i] for i in range(3)], dim=-1)
        return fused, weights


class KAMBAPlusPlus(nn.Module):
    def __init__(self, input_dim: int, cfg: Config):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, cfg.hidden_dim)
        self.edyt = EDyT(cfg.hidden_dim)
        self.controller = AdaptiveController(cfg.hidden_dim)
        self.skan = SparseKAN(
            cfg.hidden_dim, cfg.skan_dim, cfg.num_basis,
            cfg.basis_min, cfg.basis_max, cfg.gate_temperature,
        )
        self.swiglu = SwiGLUBranch(cfg.hidden_dim, cfg.swiglu_dim, cfg.dropout)
        self.dsssm = DSSSM(cfg.hidden_dim, cfg.state_dim, cfg.dropout)
        self.fusion = AdaptiveFusion(
            [cfg.skan_dim, cfg.swiglu_dim, cfg.state_dim], cfg.fusion_dim, cfg.dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(3 * cfg.fusion_dim, cfg.fusion_dim),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fusion_dim, 2),
        )

    def forward(self, x: torch.Tensor, return_aux: bool = False):
        h0 = self.input_projection(x)
        he, descriptor = self.edyt(h0)
        sparse_threshold, delta, tau = self.controller(descriptor)
        zs_seq, mask = self.skan(he, sparse_threshold)
        zt_seq = self.swiglu(he)
        zd_seq = self.dsssm(he, delta, tau)
        branches = [zs_seq.mean(1), zt_seq.mean(1), zd_seq.mean(1)]
        fused, weights = self.fusion(branches)
        logits = self.classifier(fused)
        if not return_aux:
            return logits
        return logits, {
            "mask": mask,
            "fusion_weights": weights,
            "sparse_threshold": sparse_threshold,
            "delta": delta,
            "tau": tau,
            "embedding": fused,
        }


def sparse_loss(aux: Dict[str, torch.Tensor]) -> torch.Tensor:
    return aux["mask"].abs().mean()


@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    ys, probs, aux_rows = [], [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits, aux = model(x, return_aux=True)
        ys.append(y.numpy())
        probs.append(torch.softmax(logits, -1)[:, 1].cpu().numpy())
        aux_rows.append({
            "fusion": aux["fusion_weights"].cpu().numpy(),
            "threshold": aux["sparse_threshold"].cpu().numpy(),
            "delta": aux["delta"].cpu().numpy(),
            "tau": aux["tau"].cpu().numpy(),
            "sparsity": (aux["mask"] < 0.5).float().mean((1, 2)).cpu().numpy(),
            "embedding": aux["embedding"].cpu().numpy(),
        })
    merged = {k: np.concatenate([row[k] for row in aux_rows]) for k in aux_rows[0]}
    return np.concatenate(ys), np.concatenate(probs), merged


def select_threshold(y: np.ndarray, p: np.ndarray) -> float:
    precision, recall, threshold = precision_recall_curve(y, p)
    if len(threshold) == 0:
        return 0.5
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(threshold[int(np.nanargmax(f1))])


def compute_metrics(y: np.ndarray, p: np.ndarray, threshold: float) -> Dict[str, float]:
    pred = (p >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "specificity": tn / max(tn + fp, 1),
        "f1": f1_score(y, pred, zero_division=0),
        "mcc": matthews_corrcoef(y, pred),
        "auroc": roc_auc_score(y, p) if len(np.unique(y)) == 2 else float("nan"),
        "auprc": average_precision_score(y, p) if len(np.unique(y)) == 2 else float("nan"),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def make_loader(x, y, cfg: Config, shuffle: bool) -> DataLoader:
    return DataLoader(
        SequenceDataset(x, y), batch_size=cfg.batch_size, shuffle=shuffle,
        num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(),
        persistent_workers=cfg.num_workers > 0,
    )


def train_one_seed(
    cfg: Config,
    seed: int,
    arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    device: torch.device,
) -> Dict[str, object]:
    set_seed(seed, cfg.deterministic)
    run_dir = Path(cfg.output_dir) / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    train_x, train_y = arrays["train"]
    val_x, val_y = arrays["val"]
    test_x, test_y = arrays["test"]
    train_loader = make_loader(train_x, train_y, cfg, True)
    val_loader = make_loader(val_x, val_y, cfg, False)
    test_loader = make_loader(test_x, test_y, cfg, False)

    model = KAMBAPlusPlus(train_x.shape[-1], cfg).to(device)
    counts = np.bincount(train_y, minlength=2)
    weights = torch.tensor(len(train_y) / np.maximum(2 * counts, 1), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    amp_enabled = cfg.use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    save_json({
        "optimizer": "AdamW",
        "initial_learning_rate": cfg.learning_rate,
        "weight_decay": cfg.weight_decay,
        "scheduler": "CosineAnnealingLR",
        "scheduler_T_max": cfg.epochs,
        "gradient_clip": cfg.gradient_clip,
        "mixed_precision": amp_enabled,
    }, run_dir / "optimizer_settings.json")

    best_score, best_epoch, stale = -np.inf, 0, 0
    history = []
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running, seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                logits, aux = model(xb, return_aux=True)
                loss = criterion(logits, yb) + cfg.sparse_regularizer * sparse_loss(aux)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.detach()) * len(yb)
            seen += len(yb)
        scheduler.step()

        val_true, val_prob, _ = predict(model, val_loader, device)
        threshold = select_threshold(val_true, val_prob)
        val_metrics = compute_metrics(val_true, val_prob, threshold)
        row = {
            "epoch": epoch,
            "train_loss": running / seen,
            "learning_rate": optimizer.param_groups[0]["lr"],
            **val_metrics,
        }
        history.append(row)
        if val_metrics["auprc"] > best_score + 1e-5:
            best_score, best_epoch, stale = val_metrics["auprc"], epoch, 0
            torch.save({"model": model.state_dict(), "config": asdict(cfg)}, run_dir / "best_model.pt")
        else:
            stale += 1
        if stale >= cfg.patience:
            break

    checkpoint = torch.load(run_dir / "best_model.pt", map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    val_true, val_prob, _ = predict(model, val_loader, device)
    threshold = select_threshold(val_true, val_prob)
    test_true, test_prob, aux = predict(model, test_loader, device)
    metrics = compute_metrics(test_true, test_prob, threshold)
    metrics.update({
        "seed": seed,
        "best_epoch": best_epoch,
        "parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "mean_sparse_fraction": float(aux["sparsity"].mean()),
        "mean_w_skan": float(aux["fusion"][:, 0].mean()),
        "mean_w_swiglu": float(aux["fusion"][:, 1].mean()),
        "mean_w_dsssm": float(aux["fusion"][:, 2].mean()),
    })
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(run_dir / "history.csv", index=False)
    plot_training_curves(history_frame, run_dir / "training_validation_curves.png",
                         f"{cfg.dataset_name}: seed {seed}")
    pd.DataFrame({"label": test_true, "prob_apt": test_prob}).to_csv(run_dir / "test_predictions.csv", index=False)
    save_json(metrics, run_dir / "test_metrics.json")
    return metrics


@torch.no_grad()
def profile_inference(model: nn.Module, sample: torch.Tensor, device: torch.device, repeats: int = 200):
    model.eval()
    sample = sample.to(device)
    for _ in range(30):
        model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000.0)
    arr = np.asarray(times)
    return {
        "batch_size": len(sample),
        "batch_latency_ms_mean": float(arr.mean()),
        "batch_latency_ms_p95": float(np.percentile(arr, 95)),
        "batch_latency_ms_p99": float(np.percentile(arr, 99)),
        "latency_ms_per_sequence": float(arr.mean() / len(sample)),
        "throughput_sequences_per_second": float(1000.0 * len(sample) / arr.mean()),
        "peak_gpu_memory_mb": float(torch.cuda.max_memory_allocated() / 2**20) if device.type == "cuda" else 0.0,
    }


def summarize_runs(rows: List[Dict[str, object]], output_dir: Path) -> None:
    frame = pd.DataFrame(rows)
    frame.to_csv(output_dir / "all_seed_results.csv", index=False)
    numeric = frame.select_dtypes(include=[np.number])
    summary = pd.DataFrame({"mean": numeric.mean(), "std": numeric.std(ddof=1)})
    summary.to_csv(output_dir / "mean_std_results.csv")
    save_json({k: {"mean": float(v["mean"]), "std": float(v["std"])} for k, v in summary.to_dict("index").items()}, output_dir / "mean_std_results.json")


def save_master_results_csv(
    cfg: Config,
    seed_rows: List[Dict[str, object]],
    efficiency: Dict[str, float],
    history: pd.DataFrame,
    before_tsne: pd.DataFrame,
    after_tsne: pd.DataFrame,
    test_true: np.ndarray,
    test_prob: np.ndarray,
    threshold: float,
    best_seed: int,
    output_path: Path,
) -> None:
    """Save every final plotting value in one tidy, analysis-ready CSV."""
    records: List[Dict[str, object]] = []

    for row in seed_rows:
        seed = int(row["seed"])
        for metric, value in row.items():
            if metric == "seed" or not isinstance(value, (int, float, np.integer, np.floating)):
                continue
            records.append({
                "dataset": cfg.dataset_name, "record_type": "seed_metric",
                "seed": seed, "split": "test", "metric": metric, "value": float(value),
            })

    seed_frame = pd.DataFrame(seed_rows).select_dtypes(include=[np.number])
    for metric in seed_frame.columns:
        if metric == "seed":
            continue
        records.append({
            "dataset": cfg.dataset_name, "record_type": "summary_metric",
            "split": "test", "metric": metric,
            "value": float(seed_frame[metric].mean()),
            "std": float(seed_frame[metric].std(ddof=1)),
        })

    for metric, value in efficiency.items():
        records.append({
            "dataset": cfg.dataset_name, "record_type": "efficiency",
            "seed": best_seed, "split": "test", "metric": metric, "value": float(value),
        })

    optimizer_values = {
        "optimizer": "AdamW",
        "initial_learning_rate": cfg.learning_rate,
        "weight_decay": cfg.weight_decay,
        "scheduler": "CosineAnnealingLR",
        "scheduler_T_max": cfg.epochs,
        "gradient_clip": cfg.gradient_clip,
        "batch_size": cfg.batch_size,
        "sequence_length": cfg.sequence_length,
        "sparse_regularizer": cfg.sparse_regularizer,
    }
    for metric, value in optimizer_values.items():
        record = {
            "dataset": cfg.dataset_name, "record_type": "training_configuration",
            "seed": best_seed, "metric": metric,
        }
        if isinstance(value, str):
            record["text_value"] = value
        else:
            record["value"] = float(value)
        records.append(record)

    history_metrics = [c for c in history.columns if c != "epoch"]
    for _, row in history.iterrows():
        for metric in history_metrics:
            if pd.notna(row[metric]):
                records.append({
                    "dataset": cfg.dataset_name, "record_type": "training_history",
                    "seed": best_seed, "split": "validation" if metric != "train_loss" else "train",
                    "epoch": int(row["epoch"]), "metric": metric, "value": float(row[metric]),
                })

    for stage, frame in [("before_training", before_tsne), ("after_training", after_tsne)]:
        for row in frame.itertuples(index=False):
            records.append({
                "dataset": cfg.dataset_name, "record_type": "tsne",
                "seed": best_seed if stage == "after_training" else cfg.seeds[0],
                "split": "test", "stage": stage,
                "sample_index": int(row.sample_index), "label": int(row.label),
                "tsne_x": float(row.tsne_x), "tsne_y": float(row.tsne_y),
            })

    pred = (test_prob >= threshold).astype(np.int64)
    cm = confusion_matrix(test_true, pred, labels=[0, 1])
    for true_label in (0, 1):
        denom = max(int(cm[true_label].sum()), 1)
        for predicted_label in (0, 1):
            count = int(cm[true_label, predicted_label])
            records.append({
                "dataset": cfg.dataset_name, "record_type": "confusion_matrix",
                "seed": best_seed, "split": "test", "threshold": threshold,
                "true_label": true_label, "predicted_label": predicted_label,
                "count": count, "percentage": 100.0 * count / denom,
            })

    pd.DataFrame.from_records(records).to_csv(output_path, index=False)


def run(cfg: Config) -> None:
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    save_json(asdict(cfg), output_dir / "config.json")
    device = choose_device(cfg.device)
    print(f"Device: {device}")
    prepared = prepare_tabular_data(cfg, seed=cfg.seeds[0])
    arrays = {}
    sequence_info = {}
    for split_name, row_idx in prepared["split"].items():
        sx, sy, sg = make_windows(
            prepared["frame"], prepared["x"], prepared["y"], prepared["groups"],
            row_idx, cfg.sequence_length, cfg.stride,
        )
        arrays[split_name] = (sx, sy)
        sequence_info[split_name] = {
            "sequences": len(sy), "benign": int((sy == 0).sum()), "apt": int((sy == 1).sum()),
            "unique_groups": int(len(np.unique(sg))),
        }
    save_json(sequence_info, output_dir / "sequence_summary.json")
    print(sequence_info)

    figures_dir = output_dir / "figures"
    before_tsne = plot_tsne(
        arrays["test"][0].mean(axis=1),
        arrays["test"][1],
        figures_dir / "tsne_before_training.png",
        f"{cfg.dataset_name}: standardized input representation before training",
        cfg,
        cfg.seeds[0],
    )

    rows = []
    for seed in cfg.seeds:
        print(f"Training seed {seed}")
        rows.append(train_one_seed(cfg, seed, arrays, device))
        print(rows[-1])
    summarize_runs(rows, output_dir)

    best_seed = int(pd.DataFrame(rows).sort_values("auprc", ascending=False).iloc[0]["seed"])
    model = KAMBAPlusPlus(arrays["train"][0].shape[-1], cfg).to(device)
    ckpt = torch.load(output_dir / f"seed_{best_seed}" / "best_model.pt", map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    batch = torch.from_numpy(arrays["test"][0][:min(cfg.batch_size, len(arrays["test"][0]))])
    efficiency = profile_inference(model, batch, device)
    save_json(efficiency, output_dir / "efficiency.json")
    test_loader = make_loader(arrays["test"][0], arrays["test"][1], cfg, False)
    test_true, test_prob, test_aux = predict(model, test_loader, device)
    val_loader = make_loader(arrays["val"][0], arrays["val"][1], cfg, False)
    val_true, val_prob, _ = predict(model, val_loader, device)
    final_threshold = select_threshold(val_true, val_prob)
    after_tsne = plot_tsne(
        test_aux["embedding"],
        test_true,
        figures_dir / "tsne_after_training.png",
        f"{cfg.dataset_name}: learned KAMBA++ representation after training",
        cfg,
        cfg.seeds[0],
    )
    plot_confusion(
        test_true,
        test_prob,
        final_threshold,
        figures_dir / "confusion_matrix_test.png",
        f"{cfg.dataset_name}: test confusion matrix",
    )
    best_history = pd.read_csv(output_dir / f"seed_{best_seed}" / "history.csv")
    plot_training_curves(
        best_history,
        figures_dir / "training_validation_curves_best_seed.png",
        f"{cfg.dataset_name}: best-seed training and validation",
    )
    save_master_results_csv(
        cfg=cfg,
        seed_rows=rows,
        efficiency=efficiency,
        history=best_history,
        before_tsne=before_tsne,
        after_tsne=after_tsne,
        test_true=test_true,
        test_prob=test_prob,
        threshold=final_threshold,
        best_seed=best_seed,
        output_path=output_dir / "final_results_for_figures.csv",
    )


def synthetic_smoke_test() -> None:
    cfg = Config(hidden_dim=16, skan_dim=12, swiglu_dim=12, state_dim=12, fusion_dim=12, num_basis=4)
    model = KAMBAPlusPlus(10, cfg)
    x = torch.randn(4, 8, 10)
    logits, aux = model(x, return_aux=True)
    assert logits.shape == (4, 2)
    assert aux["fusion_weights"].shape == (4, 3)
    assert torch.allclose(aux["fusion_weights"].sum(-1), torch.ones(4), atol=1e-5)
    loss = F.cross_entropy(logits, torch.tensor([0, 1, 0, 1])) + cfg.sparse_regularizer * sparse_loss(aux)
    loss.backward()
    print("Synthetic smoke test passed.")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, help="Path to JSON configuration")
    parser.add_argument("--smoke-test", action="store_true")
    args = parser.parse_args()
    if args.smoke_test:
        synthetic_smoke_test()
        return
    if not args.config:
        parser.error("--config is required unless --smoke-test is used")
    run(load_config(args.config))



## 3. GPU and architecture smoke test


In [3]:
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
synthetic_smoke_test()


PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA version: 11.8
Synthetic smoke test passed.


## 4. Dataset locations
Expected folders: `data/DAPT2020` and `data/CICIDS2017`, each containing extracted CSV files.


In [4]:
PROJECT_ROOT = Path.cwd()
DAPT_DATA = PROJECT_ROOT / 'data' / 'DAPT2020'
CICIDS_DATA = PROJECT_ROOT / 'data' / 'CICIDS2017'
print('DAPT2020 folder:', DAPT_DATA.exists(), '| CSV files:', len(list(DAPT_DATA.rglob('*.csv'))))
print('CICIDS2017 folder:', CICIDS_DATA.exists(), '| CSV files:', len(list(CICIDS_DATA.rglob('*.csv'))))


DAPT2020 folder: True | CSV files: 10
CICIDS2017 folder: True | CSV files: 8


## 5. Verify and recover the complete DAPT2020 collection
This is a read-only check. The original CSV files are not changed.


In [5]:
dapt_check = read_csv_collection(str(DAPT_DATA), label_column='Stage')
print('Total records:', len(dapt_check))
print('Number of columns:', len(dapt_check.columns))
display(dapt_check['Stage'].value_counts(dropna=False).to_frame('Count'))
assert len(dapt_check) == 86691, f'Expected 86,691 records, found {len(dapt_check):,}'
print('DAPT2020 verification passed.')


Recovered canonical header for: enp0s3-pvt-thursday.pcap_Flow.csv
Total records: 86691
Number of columns: 87


,Count
Stage,
Benign,44258
BENIGN,19454
Reconnaissance,11909
Establish Foothold,8604
Lateral Movement,2451
Data Exfiltration,15


DAPT2020 verification passed.


## 6. Configure DAPT2020
Confirm the displayed settings before training.


In [6]:
dapt_cfg = Config(
    dataset_name='DAPT2020',
    data_path=str(DAPT_DATA),
    output_dir=str(PROJECT_ROOT / 'outputs' / 'DAPT2020'),
    label_column='Stage',
    benign_values=('Benign', 'BENIGN', 'Normal', 'normal', '0'),
    group_column='Flow ID',
    timestamp_column='Timestamp',
    drop_columns=(
        'Flow ID', 'Src IP', 'Dst IP', 'Source IP', 'Destination IP',
        'Timestamp', 'Activity', 'Stage',
    ),
    sequence_length=8,
    stride=8,
    batch_size=256,
    epochs=60,
    patience=10,
    seeds=(13, 27, 41, 55, 69),
    device='auto',
)
dapt_cfg


Config(dataset_name='DAPT2020', data_path='<PROJECT_ROOT>\\data\\DAPT2020', output_dir='<PROJECT_ROOT>\\outputs\\DAPT2020', label_column='Stage', benign_values=('Benign', 'BENIGN', 'Normal', 'normal', '0'), group_column='Flow ID', timestamp_column='Timestamp', drop_columns=('Flow ID', 'Src IP', 'Dst IP', 'Source IP', 'Destination IP', 'Timestamp', 'Activity', 'Stage'), sequence_length=8, stride=8, min_group_size=2, train_fraction=0.7, val_fraction=0.15, test_fraction=0.15, hidden_dim=128, skan_dim=96, swiglu_dim=96, state_dim=96, fusion_dim=96, num_basis=8, basis_min=-3.0, basis_max=3.0, gate_temperature=0.2, dropout=0.2, batch_size=256, epochs=60, patience=10, learning_rate=0.0003, weight_decay=0.0001, sparse_regularizer=1e-05, gradient_clip=1.0, num_workers=0, seeds=(13, 27, 41, 55, 69), threshold_metric='f1', use_amp=True, deterministic=True, device='auto', tsne_max_samples=5000, tsne_perplexity=30.0)

In [7]:
import copy

pilot_cfg = copy.deepcopy(dapt_cfg)

pilot_cfg.seeds = (13,)
pilot_cfg.epochs = 5
pilot_cfg.patience = 5
pilot_cfg.tsne_max_samples = 2000

pilot_cfg.output_dir = str(
    PROJECT_ROOT
    / "outputs"
    / "DAPT2020_Pilot_Corrected"
)

run(pilot_cfg)

Device: cuda
Recovered canonical header for: enp0s3-pvt-thursday.pcap_Flow.csv
{'train': {'sequences': 29147, 'benign': 20251, 'apt': 8896, 'unique_groups': 28171}, 'val': {'sequences': 6067, 'benign': 4162, 'apt': 1905, 'unique_groups': 6036}, 'test': {'sequences': 6694, 'benign': 4788, 'apt': 1906, 'unique_groups': 6037}}
Training seed 13
{'threshold': 0.5370265245437622, 'accuracy': 0.9542874215715567, 'balanced_accuracy': 0.9606237305400612, 'precision': 0.8777148253068933, 'recall': 0.9753410283315844, 'specificity': 0.945906432748538, 'f1': 0.9239562624254473, 'mcc': 0.8939409605899455, 'auroc': 0.9880184239893193, 'auprc': 0.9692448097948488, 'tn': 4529, 'fp': 259, 'fn': 47, 'tp': 1859, 'seed': 13, 'best_epoch': 4, 'parameters': 375721, 'mean_sparse_fraction': 0.19657129049301147, 'mean_w_skan': 0.6708629727363586, 'mean_w_swiglu': 0.028456514701247215, 'mean_w_dsssm': 0.3006805181503296}


In [8]:
import copy

full_seed_cfg = copy.deepcopy(dapt_cfg)

full_seed_cfg.seeds = (13,)
full_seed_cfg.epochs = 60
full_seed_cfg.patience = 10
full_seed_cfg.tsne_max_samples = 5000

full_seed_cfg.output_dir = str(
    PROJECT_ROOT
    / "outputs"
    / "DAPT2020_Seed13_Full"
)

run(full_seed_cfg)

Device: cuda
Recovered canonical header for: enp0s3-pvt-thursday.pcap_Flow.csv
{'train': {'sequences': 29147, 'benign': 20251, 'apt': 8896, 'unique_groups': 28171}, 'val': {'sequences': 6067, 'benign': 4162, 'apt': 1905, 'unique_groups': 6036}, 'test': {'sequences': 6694, 'benign': 4788, 'apt': 1906, 'unique_groups': 6037}}
Training seed 13
{'threshold': 0.6517281532287598, 'accuracy': 0.9734090230056768, 'balanced_accuracy': 0.9743062842485717, 'precision': 0.9332998996990973, 'recall': 0.9763903462749213, 'specificity': 0.9722222222222222, 'f1': 0.9543589743589743, 'mcc': 0.9360862963794953, 'auroc': 0.9978231802836928, 'auprc': 0.9949798606855422, 'tn': 4655, 'fp': 133, 'fn': 45, 'tp': 1861, 'seed': 13, 'best_epoch': 50, 'parameters': 375721, 'mean_sparse_fraction': 0.2728104889392853, 'mean_w_skan': 0.4805900454521179, 'mean_w_swiglu': 0.17758877575397491, 'mean_w_dsssm': 0.34182119369506836}


In [9]:
import gc

gc.collect()
torch.cuda.empty_cache()

run(dapt_cfg)

Device: cuda
Recovered canonical header for: enp0s3-pvt-thursday.pcap_Flow.csv
{'train': {'sequences': 29147, 'benign': 20251, 'apt': 8896, 'unique_groups': 28171}, 'val': {'sequences': 6067, 'benign': 4162, 'apt': 1905, 'unique_groups': 6036}, 'test': {'sequences': 6694, 'benign': 4788, 'apt': 1906, 'unique_groups': 6037}}
Training seed 13
{'threshold': 0.6517281532287598, 'accuracy': 0.9734090230056768, 'balanced_accuracy': 0.9743062842485717, 'precision': 0.9332998996990973, 'recall': 0.9763903462749213, 'specificity': 0.9722222222222222, 'f1': 0.9543589743589743, 'mcc': 0.9360862963794953, 'auroc': 0.9978231802836928, 'auprc': 0.9949798606855422, 'tn': 4655, 'fp': 133, 'fn': 45, 'tp': 1861, 'seed': 13, 'best_epoch': 50, 'parameters': 375721, 'mean_sparse_fraction': 0.2728104889392853, 'mean_w_skan': 0.4805900454521179, 'mean_w_swiglu': 0.17758877575397491, 'mean_w_dsssm': 0.34182119369506836}
Training seed 27
{'threshold': 0.6489715576171875, 'accuracy': 0.9714669853600238, 'balanc

In [10]:
dapt_results_dir = Path(dapt_cfg.output_dir)

display(
    pd.read_csv(
        dapt_results_dir / "all_seed_results.csv"
    )
)

display(
    pd.read_csv(
        dapt_results_dir / "mean_std_results.csv"
    )
)

with open(
    dapt_results_dir / "efficiency.json",
    "r"
) as file:
    efficiency_results = json.load(file)

efficiency_results

,threshold,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,auroc,auprc,...,fp,fn,tp,seed,best_epoch,parameters,mean_sparse_fraction,mean_w_skan,mean_w_swiglu,mean_w_dsssm
0,0.651728,0.973409,0.974306,0.933300,0.976390,0.972222,0.954359,0.936086,0.997823,0.994980,...,133,45,1861,13,50,375721,0.272810,0.480590,0.177589,0.341821
1,0.648972,0.971467,0.974370,0.923457,0.981112,0.967627,0.951412,0.932097,0.997145,0.992625,...,155,36,1870,27,26,375721,0.203112,0.418359,0.052135,0.529505
2,0.660361,0.983567,0.981249,0.966736,0.975866,0.986633,0.971279,0.959792,0.998421,0.996468,...,64,46,1860,41,41,375721,0.149142,0.062810,0.043368,0.893821
3,0.633407,0.985211,0.983819,0.967892,0.980588,0.987051,0.974199,0.963874,0.998245,0.995902,...,62,37,1869,55,38,375721,0.246474,0.468124,0.134361,0.397515
4,0.686427,0.980729,0.977844,0.961558,0.971144,0.984545,0.966327,0.952853,0.997923,0.995265,...,74,55,1851,69,34,375721,0.107362,0.611415,0.091886,0.296700


,Unnamed: 0,mean,std
0,threshold,0.656179,0.019511
1,accuracy,0.978877,0.006131
2,balanced_accuracy,0.978318,0.004206
3,precision,0.950589,0.020709
4,recall,0.977020,0.004054
5,specificity,0.979616,0.009044
6,f1,0.963515,0.010157
7,mcc,0.948940,0.014186
8,auroc,0.997911,0.000491
9,auprc,0.995048,0.001472


{'batch_size': 256,
 'batch_latency_ms_mean': 6.132480999995096,
 'batch_latency_ms_p95': 7.107849999920288,
 'batch_latency_ms_p99': 7.5279659999341675,
 'latency_ms_per_sequence': 0.023955003906230843,
 'throughput_sequences_per_second': 41744.93161906327,
 'peak_gpu_memory_mb': 330.57568359375}

In [11]:
import gc

gc.collect()
torch.cuda.empty_cache()

official_dir = Path(dapt_cfg.output_dir)
figures_dir = official_dir / "figures"
device = choose_device(dapt_cfg.device)

# ---------------------------------------------------------
# Reconstruct the unchanged leakage-safe partitions
# ---------------------------------------------------------

prepared_final = prepare_tabular_data(
    dapt_cfg,
    seed=dapt_cfg.seeds[0]
)

official_arrays = {}

for split_name, row_indices in prepared_final["split"].items():

    sequence_x, sequence_y, sequence_groups = make_windows(
        frame=prepared_final["frame"],
        x=prepared_final["x"],
        y=prepared_final["y"],
        groups=prepared_final["groups"],
        row_indices=row_indices,
        length=dapt_cfg.sequence_length,
        stride=dapt_cfg.stride,
    )

    official_arrays[split_name] = (
        sequence_x,
        sequence_y,
    )

# ---------------------------------------------------------
# Load predetermined representative seed 13
# ---------------------------------------------------------

representative_seed = 13

model = KAMBAPlusPlus(
    input_dim=official_arrays["train"][0].shape[-1],
    cfg=dapt_cfg,
).to(device)

checkpoint = torch.load(
    official_dir
    / f"seed_{representative_seed}"
    / "best_model.pt",
    map_location=device,
    weights_only=False,
)

model.load_state_dict(checkpoint["model"])
model.eval()

# ---------------------------------------------------------
# Select threshold using validation data only
# ---------------------------------------------------------

val_loader = make_loader(
    official_arrays["val"][0],
    official_arrays["val"][1],
    dapt_cfg,
    shuffle=False,
)

test_loader = make_loader(
    official_arrays["test"][0],
    official_arrays["test"][1],
    dapt_cfg,
    shuffle=False,
)

val_true, val_prob, _ = predict(
    model,
    val_loader,
    device,
)

official_threshold = select_threshold(
    val_true,
    val_prob,
)

test_true, test_prob, test_aux = predict(
    model,
    test_loader,
    device,
)

representative_metrics = compute_metrics(
    test_true,
    test_prob,
    official_threshold,
)

representative_metrics.update({
    "representative_seed": representative_seed,
    "selection_rule": "Predetermined first seed",
})

save_json(
    representative_metrics,
    official_dir
    / "representative_seed_13_metrics.json",
)

representative_metrics

Recovered canonical header for: enp0s3-pvt-thursday.pcap_Flow.csv


{'threshold': 0.6517281532287598,
 'accuracy': 0.9734090230056768,
 'balanced_accuracy': 0.9743062842485717,
 'precision': 0.9332998996990973,
 'recall': 0.9763903462749213,
 'specificity': 0.9722222222222222,
 'f1': 0.9543589743589743,
 'mcc': 0.9360862963794953,
 'auroc': 0.9978231802836928,
 'auprc': 0.9949798606855422,
 'tn': 4655,
 'fp': 133,
 'fn': 45,
 'tp': 1861,
 'representative_seed': 13,
 'selection_rule': 'Predetermined first seed'}

In [12]:
before_tsne = plot_tsne(
    representation=official_arrays["test"][0].mean(axis=1),
    labels=official_arrays["test"][1],
    path=figures_dir / "tsne_before_training.png",
    title=(
        "DAPT2020: standardized input representation "
        "before training"
    ),
    cfg=dapt_cfg,
    seed=representative_seed,
)

after_tsne = plot_tsne(
    representation=test_aux["embedding"],
    labels=test_true,
    path=figures_dir / "tsne_after_training.png",
    title=(
        "DAPT2020: learned KAMBA++ representation "
        "after training"
    ),
    cfg=dapt_cfg,
    seed=representative_seed,
)

plot_confusion(
    y=test_true,
    p=test_prob,
    threshold=official_threshold,
    path=figures_dir / "confusion_matrix_test.png",
    title="DAPT2020: test confusion matrix",
)

seed_13_history = pd.read_csv(
    official_dir
    / "seed_13"
    / "history.csv"
)

plot_training_curves(
    history=seed_13_history,
    path=(
        figures_dir
        / "training_validation_curves_seed_13.png"
    ),
    title="DAPT2020: training and validation",
)

In [13]:
efficiency_rows = []

test_x = official_arrays["test"][0]

for batch_size in [1, 32, 128, 256]:

    sample = torch.from_numpy(
        test_x[:batch_size]
    )

    result = profile_inference(
        model=model,
        sample=sample,
        device=device,
        repeats=200,
    )

    efficiency_rows.append(result)

efficiency_by_batch = pd.DataFrame(
    efficiency_rows
)

display(efficiency_by_batch)

efficiency_by_batch.to_csv(
    official_dir
    / "efficiency_by_batch_size.csv",
    index=False,
)

,batch_size,batch_latency_ms_mean,batch_latency_ms_p95,batch_latency_ms_p99,latency_ms_per_sequence,throughput_sequences_per_second,peak_gpu_memory_mb
0,1,3.393374,3.873300,4.675118,3.393374,294.691950,20.392578
1,32,3.437536,4.142725,4.615735,0.107423,9308.993419,58.100586
2,128,3.889426,4.789875,5.322168,0.030386,32909.740409,174.875000
3,256,6.149084,7.289870,7.835841,0.024020,41632.217091,330.575684


In [14]:
all_seed_rows = pd.read_csv(
    official_dir / "all_seed_results.csv"
).to_dict("records")

batch_256_efficiency = (
    efficiency_by_batch.loc[
        efficiency_by_batch["batch_size"] == 256
    ]
    .iloc[0]
    .to_dict()
)

save_master_results_csv(
    cfg=dapt_cfg,
    seed_rows=all_seed_rows,
    efficiency=batch_256_efficiency,
    history=seed_13_history,
    before_tsne=before_tsne,
    after_tsne=after_tsne,
    test_true=test_true,
    test_prob=test_prob,
    threshold=official_threshold,
    best_seed=representative_seed,
    output_path=(
        official_dir
        / "final_results_for_figures.csv"
    ),
)

master_results = pd.read_csv(
    official_dir
    / "final_results_for_figures.csv"
)

additional_efficiency = []

for _, efficiency_row in efficiency_by_batch.iterrows():

    batch_size = int(
        efficiency_row["batch_size"]
    )

    for metric, value in efficiency_row.items():

        if metric == "batch_size":
            continue

        additional_efficiency.append({
            "dataset": "DAPT2020",
            "record_type": "efficiency_by_batch",
            "seed": representative_seed,
            "split": "test",
            "batch_size": batch_size,
            "metric": metric,
            "value": float(value),
        })

master_results = pd.concat(
    [
        master_results,
        pd.DataFrame(additional_efficiency),
    ],
    ignore_index=True,
)

master_results.to_csv(
    official_dir
    / "final_results_for_figures.csv",
    index=False,
)

print(
    "Master CSV saved:",
    official_dir
    / "final_results_for_figures.csv"
)

print("Total master records:", len(master_results))

display(
    master_results[
        master_results["record_type"]
        == "efficiency_by_batch"
    ]
)

Master CSV saved: <PROJECT_ROOT>\outputs\DAPT2020\final_results_for_figures.csv
Total master records: 11124


,dataset,record_type,seed,split,metric,value,std,text_value,epoch,stage,sample_index,label,tsne_x,tsne_y,threshold,true_label,predicted_label,count,percentage,batch_size
11100,DAPT2020,efficiency_by_batch,13.0,test,batch_latency_ms_mean,3.393374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
11101,DAPT2020,efficiency_by_batch,13.0,test,batch_latency_ms_p95,3.873300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
11102,DAPT2020,efficiency_by_batch,13.0,test,batch_latency_ms_p99,4.675118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
11103,DAPT2020,efficiency_by_batch,13.0,test,latency_ms_per_sequence,3.393374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
11104,DAPT2020,efficiency_by_batch,13.0,test,throughput_sequences_per_second,294.691950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
11105,DAPT2020,efficiency_by_batch,13.0,test,peak_gpu_memory_mb,20.392578,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
11106,DAPT2020,efficiency_by_batch,13.0,test,batch_latency_ms_mean,3.437536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0
11107,DAPT2020,efficiency_by_batch,13.0,test,batch_latency_ms_p95,4.142725,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0
11108,DAPT2020,efficiency_by_batch,13.0,test,batch_latency_ms_p99,4.615735,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0
11109,DAPT2020,efficiency_by_batch,13.0,test,latency_ms_per_sequence,0.107423,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0


## 9. Inspect CICIDS2017 columns


In [16]:
# Point CICIDS_DATA to the newly downloaded full dataset

CICIDS_DATA = (
    PROJECT_ROOT
    / "data"
    / "CICIDS2017_Full"
)

print("CICIDS2017 path:", CICIDS_DATA)
print("Folder exists:", CICIDS_DATA.exists())
print(
    "CSV files found:",
    len(list(CICIDS_DATA.rglob("*.csv")))
)

CICIDS2017 path: <PROJECT_ROOT>\data\CICIDS2017_Full
Folder exists: True
CSV files found: 8


In [18]:
cicids_files = sorted(
    CICIDS_DATA.rglob("*.csv")
)

cicids_file_summary = []
cicids_schema_issues = []

required_cicids_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label",
]

for file in cicids_files:

    header = pd.read_csv(
        file,
        nrows=0,
        encoding="cp1252",
        low_memory=False,
    )

    clean_name = {
        original: str(original).strip().lstrip("\ufeff")
        for original in header.columns
    }

    available = set(clean_name.values())

    missing = [
        column
        for column in required_cicids_columns
        if column not in available
    ]

    if missing:
        cicids_schema_issues.append({
            "File": file.name,
            "Missing": missing,
            "Available": sorted(available),
        })
        continue

    selected_original = [
        original
        for original, cleaned in clean_name.items()
        if cleaned in [
            "Flow ID",
            "Timestamp",
            "Label",
        ]
    ]

    part = pd.read_csv(
        file,
        usecols=selected_original,
        encoding="cp1252",
        low_memory=False,
    )

    part = part.rename(columns=clean_name)

    normalized_label = (
        part["Label"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    cicids_file_summary.append({
        "File": file.name,
        "Records": len(part),
        "Unique Flow IDs": int(
            part["Flow ID"].nunique()
        ),
        "Benign": int(
            normalized_label.eq("BENIGN").sum()
        ),
        "Attack": int(
            (
                part["Label"].notna()
                & normalized_label.ne("BENIGN")
            ).sum()
        ),
        "Missing labels": int(
            part["Label"].isna().sum()
        ),
        "Missing Flow IDs": int(
            part["Flow ID"].isna().sum()
        ),
    })

print("Files found:", len(cicids_files))

print(
    "Files successfully audited:",
    len(cicids_file_summary)
)

print(
    "Files with schema issues:",
    len(cicids_schema_issues)
)

cicids_audit_df = pd.DataFrame(
    cicids_file_summary
)

display(cicids_audit_df)

if not cicids_audit_df.empty:
    print(
        "\nTotal records:",
        f"{cicids_audit_df['Records'].sum():,}"
    )

    print(
        "Total benign:",
        f"{cicids_audit_df['Benign'].sum():,}"
    )

    print(
        "Total attacks:",
        f"{cicids_audit_df['Attack'].sum():,}"
    )

    print(
        "Missing labels:",
        f"{cicids_audit_df['Missing labels'].sum():,}"
    )

    print(
        "Missing Flow IDs:",
        f"{cicids_audit_df['Missing Flow IDs'].sum():,}"
    )

if cicids_schema_issues:
    for issue in cicids_schema_issues:
        print("\nFile:", issue["File"])
        print("Missing:", issue["Missing"])
        print("Available:", issue["Available"])

Files found: 8
Files successfully audited: 8
Files with schema issues: 0


,File,Records,Unique Flow IDs,Benign,Attack,Missing labels,Missing Flow IDs
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,86421,97718,128027,0,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,229683,127537,158930,0,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,101977,189067,1966,0,0
3,Monday-WorkingHours.pcap_ISCX.csv,529918,249213,529918,0,0,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,183044,288566,36,0,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,89866,168186,2180,288602,288602
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,211628,432074,13835,0,0
7,Wednesday-workingHours.pcap_ISCX.csv,692703,226768,440031,252672,0,0



Total records: 3,119,345
Total benign: 2,273,097
Total attacks: 557,646
Missing labels: 288,602
Missing Flow IDs: 288,602


In [19]:
cicids_files = sorted(
    CICIDS_DATA.rglob("*.csv")
)

if cicids_files:

    cicids_sample = pd.read_csv(
        cicids_files[0],
        nrows=5,
        encoding="cp1252",
        low_memory=False,
    )

    cicids_sample.columns = (
        cicids_sample.columns
        .astype(str)
        .str.strip()
        .str.lstrip("\ufeff")
    )

    print("File:", cicids_files[0].name)
    print(
        "Number of columns:",
        len(cicids_sample.columns)
    )

    print(
        "Columns:",
        cicids_sample.columns.tolist()
    )

    display(
        cicids_sample.head()
    )

else:
    print(
        "No CICIDS2017 CSV files found in:",
        CICIDS_DATA
    )

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Number of columns: 85
Columns: ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443,192.168.10.5,54865,6,7/7/2017 3:30,3,2,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80,192.168.10.5,55054,6,7/7/2017 3:30,109,1,1,...,20,0,0,0,0,0,0,0,0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80,192.168.10.5,55055,6,7/7/2017 3:30,52,1,1,...,20,0,0,0,0,0,0,0,0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443,192.168.10.16,46236,6,7/7/2017 3:30,34,1,1,...,20,0,0,0,0,0,0,0,0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443,192.168.10.5,54863,6,7/7/2017 3:30,3,2,0,...,20,0,0,0,0,0,0,0,0,BENIGN


In [20]:
cicids_files = sorted(
    CICIDS_DATA.rglob("*.csv")
)

cicids_file_summary = []
cicids_schema_issues = []

required_cicids_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label",
]

for file in cicids_files:

    header = pd.read_csv(
        file,
        nrows=0,
        encoding="cp1252",
        low_memory=False,
    )

    clean_name = {
        original: str(original).strip().lstrip("\ufeff")
        for original in header.columns
    }

    available = set(clean_name.values())

    missing = [
        column
        for column in required_cicids_columns
        if column not in available
    ]

    if missing:
        cicids_schema_issues.append({
            "File": file.name,
            "Missing": missing,
            "Available": sorted(available),
        })
        continue

    selected_original = [
        original
        for original, cleaned in clean_name.items()
        if cleaned in [
            "Flow ID",
            "Timestamp",
            "Label",
        ]
    ]

    part = pd.read_csv(
        file,
        usecols=selected_original,
        encoding="cp1252",
        low_memory=False,
    )

    part = part.rename(columns=clean_name)

    normalized_label = (
        part["Label"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    cicids_file_summary.append({
        "File": file.name,
        "Records": len(part),
        "Unique Flow IDs": int(
            part["Flow ID"].nunique()
        ),
        "Benign": int(
            normalized_label.eq("BENIGN").sum()
        ),
        "Attack": int(
            (
                part["Label"].notna()
                & normalized_label.ne("BENIGN")
            ).sum()
        ),
        "Missing labels": int(
            part["Label"].isna().sum()
        ),
        "Missing Flow IDs": int(
            part["Flow ID"].isna().sum()
        ),
    })

print("Files found:", len(cicids_files))

print(
    "Files successfully audited:",
    len(cicids_file_summary)
)

print(
    "Files with schema issues:",
    len(cicids_schema_issues)
)

cicids_audit_df = pd.DataFrame(
    cicids_file_summary
)

display(cicids_audit_df)

if not cicids_audit_df.empty:
    print(
        "\nTotal records:",
        f"{cicids_audit_df['Records'].sum():,}"
    )

    print(
        "Total benign:",
        f"{cicids_audit_df['Benign'].sum():,}"
    )

    print(
        "Total attacks:",
        f"{cicids_audit_df['Attack'].sum():,}"
    )

    print(
        "Missing labels:",
        f"{cicids_audit_df['Missing labels'].sum():,}"
    )

    print(
        "Missing Flow IDs:",
        f"{cicids_audit_df['Missing Flow IDs'].sum():,}"
    )

if cicids_schema_issues:
    for issue in cicids_schema_issues:
        print("\nFile:", issue["File"])
        print("Missing:", issue["Missing"])
        print("Available:", issue["Available"])

Files found: 8
Files successfully audited: 8
Files with schema issues: 0


,File,Records,Unique Flow IDs,Benign,Attack,Missing labels,Missing Flow IDs
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,86421,97718,128027,0,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,229683,127537,158930,0,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,101977,189067,1966,0,0
3,Monday-WorkingHours.pcap_ISCX.csv,529918,249213,529918,0,0,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,183044,288566,36,0,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,89866,168186,2180,288602,288602
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,211628,432074,13835,0,0
7,Wednesday-workingHours.pcap_ISCX.csv,692703,226768,440031,252672,0,0



Total records: 3,119,345
Total benign: 2,273,097
Total attacks: 557,646
Missing labels: 288,602
Missing Flow IDs: 288,602


In [21]:
# ============================================================
# Diagnose incomplete rows in the CICIDS2017 Web Attacks file
# ============================================================

web_attack_file = next(
    file
    for file in cicids_files
    if "Morning-WebAttacks" in file.name
)

header = pd.read_csv(
    web_attack_file,
    nrows=0,
    encoding="cp1252",
    low_memory=False,
)

clean_name = {
    original: str(original).strip().lstrip("\ufeff")
    for original in header.columns
}

selected_original = [
    original
    for original, cleaned in clean_name.items()
    if cleaned in [
        "Flow ID",
        "Source IP",
        "Destination IP",
        "Timestamp",
        "Label",
    ]
]

web_check = pd.read_csv(
    web_attack_file,
    usecols=selected_original,
    encoding="cp1252",
    low_memory=False,
)

web_check = web_check.rename(
    columns=clean_name
)

invalid_mask = (
    web_check["Flow ID"].isna()
    | web_check["Label"].isna()
)

invalid_indices = np.flatnonzero(
    invalid_mask.to_numpy()
)

valid_web = web_check.loc[
    ~invalid_mask
].copy()

print("File:", web_attack_file.name)
print("Total rows:", f"{len(web_check):,}")
print("Valid rows:", f"{len(valid_web):,}")
print("Invalid rows:", f"{invalid_mask.sum():,}")

if len(invalid_indices) > 0:

    print(
        "First invalid index:",
        int(invalid_indices[0])
    )

    print(
        "Last invalid index:",
        int(invalid_indices[-1])
    )

    contiguous = bool(
        np.all(np.diff(invalid_indices) == 1)
    )

    print(
        "Invalid rows form one contiguous block:",
        contiguous
    )

print("\nValid label distribution:")

valid_labels = (
    valid_web["Label"]
    .astype("string")
    .str.strip()
)

display(
    valid_labels
    .value_counts(dropna=False)
    .to_frame("Count")
)

print("\nMissing values in selected columns:")

display(
    web_check
    .isna()
    .sum()
    .to_frame("Missing")
)

print(
    "\nExpected valid Web Attacks records:",
    "170,366"
)

assert len(valid_web) == 170366, (
    "Unexpected number of valid Web Attacks records: "
    f"{len(valid_web):,}"
)

assert int(invalid_mask.sum()) == 288602, (
    "Unexpected number of incomplete rows: "
    f"{invalid_mask.sum():,}"
)

print("\nWeb Attacks diagnostic passed.")

File: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Total rows: 458,968
Valid rows: 170,366
Invalid rows: 288,602
First invalid index: 170366
Last invalid index: 458967
Invalid rows form one contiguous block: True

Valid label distribution:


,Count
Label,
BENIGN,168186
Web Attack – Brute Force,1507
Web Attack – XSS,652
Web Attack – Sql Injection,21



Missing values in selected columns:


,Missing
Flow ID,288602
Source IP,288602
Destination IP,288602
Timestamp,288602
Label,288602



Expected valid Web Attacks records: 170,366

Web Attacks diagnostic passed.


In [22]:
# ============================================================
# CICIDS2017 valid-record and Flow-ID group-size audit
# ============================================================

from collections import Counter

flow_size_counter = Counter()
flow_label_sets = {}
valid_file_summary = []

for file in cicids_files:

    header = pd.read_csv(
        file,
        nrows=0,
        encoding="cp1252",
        low_memory=False,
    )

    clean_name = {
        original: str(original).strip().lstrip("\ufeff")
        for original in header.columns
    }

    selected_original = [
        original
        for original, cleaned in clean_name.items()
        if cleaned in ["Flow ID", "Label"]
    ]

    file_records = 0
    file_valid = 0
    file_removed = 0

    for chunk in pd.read_csv(
        file,
        usecols=selected_original,
        encoding="cp1252",
        low_memory=False,
        chunksize=200_000,
    ):

        chunk = chunk.rename(
            columns=clean_name
        )

        file_records += len(chunk)

        valid_mask = (
            chunk["Flow ID"].notna()
            & chunk["Label"].notna()
        )

        valid_chunk = chunk.loc[
            valid_mask,
            ["Flow ID", "Label"]
        ].copy()

        file_valid += len(valid_chunk)
        file_removed += int((~valid_mask).sum())

        flow_ids = (
            valid_chunk["Flow ID"]
            .astype("string")
            .str.strip()
        )

        labels = (
            valid_chunk["Label"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        flow_size_counter.update(
            flow_ids.tolist()
        )

        for flow_id, label in zip(
            flow_ids.tolist(),
            labels.tolist(),
        ):
            if flow_id not in flow_label_sets:
                flow_label_sets[flow_id] = set()

            flow_label_sets[flow_id].add(label)

    valid_file_summary.append({
        "File": file.name,
        "Raw records": file_records,
        "Valid records": file_valid,
        "Removed incomplete": file_removed,
    })


valid_file_summary_df = pd.DataFrame(
    valid_file_summary
)

display(valid_file_summary_df)

group_sizes = np.asarray(
    list(flow_size_counter.values()),
    dtype=np.int64,
)

group_distribution = pd.DataFrame({
    "Group-size category": [
        "1 record",
        "2--7 records",
        "8--15 records",
        "16--31 records",
        "32 or more records",
    ],
    "Flow IDs": [
        int((group_sizes == 1).sum()),
        int(
            (
                (group_sizes >= 2)
                & (group_sizes <= 7)
            ).sum()
        ),
        int(
            (
                (group_sizes >= 8)
                & (group_sizes <= 15)
            ).sum()
        ),
        int(
            (
                (group_sizes >= 16)
                & (group_sizes <= 31)
            ).sum()
        ),
        int((group_sizes >= 32).sum()),
    ],
})

group_distribution["Percentage"] = (
    100.0
    * group_distribution["Flow IDs"]
    / len(group_sizes)
)

mixed_label_groups = sum(
    len(labels) > 1
    for labels in flow_label_sets.values()
)

print(
    "Raw records:",
    f"{valid_file_summary_df['Raw records'].sum():,}"
)

print(
    "Valid records:",
    f"{valid_file_summary_df['Valid records'].sum():,}"
)

print(
    "Removed incomplete records:",
    f"{valid_file_summary_df['Removed incomplete'].sum():,}"
)

print(
    "Unique Flow IDs:",
    f"{len(group_sizes):,}"
)

print(
    "Minimum group size:",
    int(group_sizes.min())
)

print(
    "Median group size:",
    float(np.median(group_sizes))
)

print(
    "Mean group size:",
    float(group_sizes.mean())
)

print(
    "Maximum group size:",
    int(group_sizes.max())
)

print(
    "Flow IDs with mixed benign/attack labels:",
    f"{mixed_label_groups:,}"
)

print(
    "Groups with at least 8 records:",
    f"{int((group_sizes >= 8).sum()):,}"
)

print(
    "Records belonging to groups with at least 8 records:",
    f"{sum(size for size in group_sizes if size >= 8):,}"
)

display(group_distribution)

assert (
    valid_file_summary_df["Valid records"].sum()
    == 2_830_743
), "The valid CICIDS2017 record count is not 2,830,743."

assert (
    valid_file_summary_df["Removed incomplete"].sum()
    == 288_602
), "The number of removed incomplete records is unexpected."

print(
    "\nCICIDS2017 valid-record audit passed."
)

,File,Raw records,Valid records,Removed incomplete
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,225745,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,286467,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,191033,0
3,Monday-WorkingHours.pcap_ISCX.csv,529918,529918,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,288602,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,170366,288602
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,445909,0
7,Wednesday-workingHours.pcap_ISCX.csv,692703,692703,0


Raw records: 3,119,345
Valid records: 2,830,743
Removed incomplete records: 288,602
Unique Flow IDs: 1,085,071
Minimum group size: 1
Median group size: 2.0
Mean group size: 2.608809008811405
Maximum group size: 1184
Flow IDs with mixed benign/attack labels: 29,346
Groups with at least 8 records: 26,741
Records belonging to groups with at least 8 records: 726,849


,Group-size category,Flow IDs,Percentage
0,1 record,392683,36.189613
1,2--7 records,665647,61.345940
2,8--15 records,10412,0.959569
3,16--31 records,13409,1.235772
4,32 or more records,2920,0.269107



CICIDS2017 valid-record audit passed.


In [23]:
# ============================================================
# CICIDS2017 candidate conversation-group audit
# ============================================================

from collections import Counter, defaultdict

directional_sizes = Counter()
bidirectional_sizes = Counter()

directional_label_bits = defaultdict(int)
bidirectional_label_bits = defaultdict(int)

valid_records = 0

for file in cicids_files:

    header = pd.read_csv(
        file,
        nrows=0,
        encoding="cp1252",
        low_memory=False,
    )

    clean_name = {
        original: str(original).strip().lstrip("\ufeff")
        for original in header.columns
    }

    required_for_grouping = [
        "Source IP",
        "Destination IP",
        "Protocol",
        "Label",
    ]

    selected_original = [
        original
        for original, cleaned in clean_name.items()
        if cleaned in required_for_grouping
    ]

    for chunk in pd.read_csv(
        file,
        usecols=selected_original,
        encoding="cp1252",
        low_memory=False,
        chunksize=200_000,
    ):

        chunk = chunk.rename(
            columns=clean_name
        )

        valid_mask = (
            chunk["Source IP"].notna()
            & chunk["Destination IP"].notna()
            & chunk["Protocol"].notna()
            & chunk["Label"].notna()
        )

        chunk = chunk.loc[
            valid_mask,
            required_for_grouping
        ].copy()

        valid_records += len(chunk)

        source_ip = (
            chunk["Source IP"]
            .astype("string")
            .str.strip()
        )

        destination_ip = (
            chunk["Destination IP"]
            .astype("string")
            .str.strip()
        )

        protocol = (
            chunk["Protocol"]
            .astype("string")
            .str.strip()
        )

        labels = (
            chunk["Label"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        # Direction-sensitive host-pair group
        directional_keys = (
            source_ip
            + "|"
            + destination_ip
            + "|"
            + protocol
        )

        # Direction-independent host-pair group
        endpoint_a = np.where(
            source_ip.to_numpy() <= destination_ip.to_numpy(),
            source_ip.to_numpy(),
            destination_ip.to_numpy(),
        )

        endpoint_b = np.where(
            source_ip.to_numpy() <= destination_ip.to_numpy(),
            destination_ip.to_numpy(),
            source_ip.to_numpy(),
        )

        bidirectional_keys = (
            pd.Series(endpoint_a, index=chunk.index).astype("string")
            + "|"
            + pd.Series(endpoint_b, index=chunk.index).astype("string")
            + "|"
            + protocol
        )

        label_bits = np.where(
            labels.eq("BENIGN").to_numpy(),
            1,  # benign
            2,  # attack
        )

        directional_list = directional_keys.tolist()
        bidirectional_list = bidirectional_keys.tolist()

        directional_sizes.update(
            directional_list
        )

        bidirectional_sizes.update(
            bidirectional_list
        )

        for key, bit in zip(
            directional_list,
            label_bits,
        ):
            directional_label_bits[key] |= int(bit)

        for key, bit in zip(
            bidirectional_list,
            label_bits,
        ):
            bidirectional_label_bits[key] |= int(bit)


def summarize_grouping(
    name,
    size_counter,
    label_bit_mapping,
):

    sizes = np.asarray(
        list(size_counter.values()),
        dtype=np.int64,
    )

    summary = {
        "Grouping": name,
        "Groups": len(sizes),
        "Median size": float(np.median(sizes)),
        "Mean size": float(sizes.mean()),
        "Maximum size": int(sizes.max()),
        "Groups >= 8": int((sizes >= 8).sum()),
        "Records in groups >= 8": int(
            sizes[sizes >= 8].sum()
        ),
        "Mixed-label groups": int(
            sum(
                bits == 3
                for bits in label_bit_mapping.values()
            )
        ),
    }

    distribution = pd.DataFrame({
        "Category": [
            "1 record",
            "2--7 records",
            "8--15 records",
            "16--31 records",
            "32 or more records",
        ],
        "Groups": [
            int((sizes == 1).sum()),
            int(
                (
                    (sizes >= 2)
                    & (sizes <= 7)
                ).sum()
            ),
            int(
                (
                    (sizes >= 8)
                    & (sizes <= 15)
                ).sum()
            ),
            int(
                (
                    (sizes >= 16)
                    & (sizes <= 31)
                ).sum()
            ),
            int((sizes >= 32).sum()),
        ],
    })

    distribution["Percentage"] = (
        100.0
        * distribution["Groups"]
        / len(sizes)
    )

    return summary, distribution


directional_summary, directional_distribution = (
    summarize_grouping(
        "Directional IP pair + protocol",
        directional_sizes,
        directional_label_bits,
    )
)

bidirectional_summary, bidirectional_distribution = (
    summarize_grouping(
        "Bidirectional IP pair + protocol",
        bidirectional_sizes,
        bidirectional_label_bits,
    )
)

print(
    "Valid records:",
    f"{valid_records:,}"
)

display(
    pd.DataFrame([
        directional_summary,
        bidirectional_summary,
    ])
)

print(
    "\nDirectional group-size distribution:"
)

display(
    directional_distribution
)

print(
    "\nBidirectional group-size distribution:"
)

display(
    bidirectional_distribution
)

assert valid_records == 2_830_743

print(
    "\nCandidate conversation-group audit passed."
)

Valid records: 2,830,743


,Grouping,Groups,Median size,Mean size,Maximum size,Groups >= 8,Records in groups >= 8,Mixed-label groups
0,Directional IP pair + protocol,126064,3.0,22.454809,553548,28171,2562464,11
1,Bidirectional IP pair + protocol,69384,6.0,40.798210,594264,28572,2684546,9



Directional group-size distribution:


,Category,Groups,Percentage
0,1 record,28948,22.962939
1,2--7 records,68945,54.690475
2,8--15 records,15505,12.299308
3,16--31 records,7021,5.569393
4,32 or more records,5645,4.477884



Bidirectional group-size distribution:


,Category,Groups,Percentage
0,1 record,1381,1.990372
1,2--7 records,39431,56.830105
2,8--15 records,14797,21.326242
3,16--31 records,7584,10.930474
4,32 or more records,6191,8.922806



Candidate conversation-group audit passed.


In [24]:
# ============================================================
# Updated CSV loader:
# - supports UTF-8 DAPT2020
# - supports CP1252 CICIDS2017
# - removes incomplete CICIDS2017 rows
# - creates bidirectional conversation groups
# - excludes CICIDS2017 groups shorter than T
# ============================================================

def read_csv_collection(
    path: str,
    label_column: Optional[str] = None,
) -> pd.DataFrame:

    dataset_path = Path(path)

    files = (
        [dataset_path]
        if dataset_path.is_file()
        else sorted(dataset_path.rglob("*.csv"))
    )

    if not files:
        raise FileNotFoundError(
            f"No CSV files found under "
            f"{dataset_path.resolve()}"
        )

    is_cicids2017 = (
        label_column == "Label"
    )

    encoding = (
        "cp1252"
        if is_cicids2017
        else "utf-8-sig"
    )

    canonical_columns = None

    if label_column:

        for file in files:

            header = pd.read_csv(
                file,
                nrows=0,
                encoding=encoding,
                low_memory=False,
            )

            cleaned_columns = [
                str(column)
                .strip()
                .lstrip("\ufeff")
                for column in header.columns
            ]

            if label_column in cleaned_columns:
                canonical_columns = cleaned_columns
                break

    frames = []

    for file in files:

        header = pd.read_csv(
            file,
            nrows=0,
            encoding=encoding,
            low_memory=False,
        )

        cleaned_header = [
            str(column)
            .strip()
            .lstrip("\ufeff")
            for column in header.columns
        ]

        has_expected_header = (
            label_column is None
            or label_column in cleaned_header
        )

        if has_expected_header:

            frame = pd.read_csv(
                file,
                encoding=encoding,
                low_memory=False,
            )

            frame.columns = [
                str(column)
                .strip()
                .lstrip("\ufeff")
                for column in frame.columns
            ]

        else:

            if canonical_columns is None:
                raise ValueError(
                    "Could not recover the headerless CSV "
                    f"{file.name}: no canonical header "
                    "was found."
                )

            frame = pd.read_csv(
                file,
                header=None,
                encoding=encoding,
                low_memory=False,
            )

            if frame.shape[1] != len(
                canonical_columns
            ):
                raise ValueError(
                    f"Headerless CSV {file.name} has "
                    f"{frame.shape[1]} columns; the "
                    f"canonical schema has "
                    f"{len(canonical_columns)}."
                )

            frame.columns = canonical_columns

            print(
                "Recovered canonical header for:",
                file.name
            )

        frame["__source_file__"] = file.name

        frame["__source_row__"] = np.arange(
            len(frame),
            dtype=np.int64,
        )

        frames.append(frame)

    combined = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    if is_cicids2017:

        required_metadata = [
            "Flow ID",
            "Source IP",
            "Destination IP",
            "Protocol",
            "Timestamp",
            "Label",
        ]

        missing_columns = [
            column
            for column in required_metadata
            if column not in combined.columns
        ]

        if missing_columns:
            raise KeyError(
                "CICIDS2017 is missing required "
                f"metadata columns: {missing_columns}"
            )

        raw_count = len(combined)

        complete_mask = (
            combined["Flow ID"].notna()
            & combined["Source IP"].notna()
            & combined["Destination IP"].notna()
            & combined["Protocol"].notna()
            & combined["Timestamp"].notna()
            & combined["Label"].notna()
        )

        combined = (
            combined.loc[complete_mask]
            .copy()
            .reset_index(drop=True)
        )

        removed_incomplete = (
            raw_count - len(combined)
        )

        source_ip = (
            combined["Source IP"]
            .astype("string")
            .str.strip()
        )

        destination_ip = (
            combined["Destination IP"]
            .astype("string")
            .str.strip()
        )

        protocol = (
            combined["Protocol"]
            .astype("string")
            .str.strip()
        )

        source_array = source_ip.to_numpy()
        destination_array = (
            destination_ip.to_numpy()
        )

        endpoint_a = np.where(
            source_array <= destination_array,
            source_array,
            destination_array,
        )

        endpoint_b = np.where(
            source_array <= destination_array,
            destination_array,
            source_array,
        )

        combined["__conversation_id__"] = (
            pd.Series(
                endpoint_a,
                index=combined.index,
                dtype="string",
            )
            + "|"
            + pd.Series(
                endpoint_b,
                index=combined.index,
                dtype="string",
            )
            + "|"
            + protocol
        )

        conversation_sizes = (
            combined["__conversation_id__"]
            .value_counts()
        )

        eligible_conversations = (
            conversation_sizes[
                conversation_sizes >= 8
            ]
            .index
        )

        before_short_filter = len(combined)

        combined = (
            combined.loc[
                combined[
                    "__conversation_id__"
                ].isin(eligible_conversations)
            ]
            .copy()
            .reset_index(drop=True)
        )

        removed_short = (
            before_short_filter - len(combined)
        )

        print(
            "CICIDS2017 raw rows:",
            f"{raw_count:,}"
        )

        print(
            "Removed incomplete rows:",
            f"{removed_incomplete:,}"
        )

        print(
            "Valid official records:",
            f"{before_short_filter:,}"
        )

        print(
            "Removed records from groups < 8:",
            f"{removed_short:,}"
        )

        print(
            "Records retained for sequencing:",
            f"{len(combined):,}"
        )

        print(
            "Eligible conversation groups:",
            f"{combined['__conversation_id__'].nunique():,}"
        )

        assert before_short_filter == 2_830_743

        assert removed_incomplete == 288_602

        assert len(combined) == 2_684_546

        assert (
            combined["__conversation_id__"]
            .value_counts()
            .min()
            >= 8
        )

    return combined

In [26]:
# ============================================================
# Efficient group split and sequence construction
# for large CICIDS2017 data
# ============================================================

def group_stratified_split(
    groups: np.ndarray,
    labels: np.ndarray,
    cfg: Config,
    seed: int,
) -> Dict[str, np.ndarray]:

    unique_groups, inverse = np.unique(
        groups,
        return_inverse=True,
    )

    # A group is considered attack-associated if at least
    # one record in that group is malicious.
    group_labels = np.zeros(
        len(unique_groups),
        dtype=np.int64,
    )

    np.maximum.at(
        group_labels,
        inverse,
        labels,
    )

    rng = np.random.default_rng(seed)

    train_groups = []
    val_groups = []
    test_groups = []

    for class_value in np.unique(group_labels):

        class_groups = unique_groups[
            group_labels == class_value
        ].copy()

        rng.shuffle(class_groups)

        number_of_groups = len(class_groups)

        number_train = max(
            1,
            int(
                round(
                    cfg.train_fraction
                    * number_of_groups
                )
            ),
        )

        number_val = (
            max(
                1,
                int(
                    round(
                        cfg.val_fraction
                        * number_of_groups
                    )
                ),
            )
            if number_of_groups >= 3
            else 0
        )

        if (
            number_train + number_val
            >= number_of_groups
        ):

            number_train = (
                max(1, number_of_groups - 2)
                if number_of_groups >= 3
                else max(1, number_of_groups - 1)
            )

            number_val = (
                1
                if number_of_groups >= 3
                else 0
            )

        train_groups.extend(
            class_groups[:number_train]
        )

        val_groups.extend(
            class_groups[
                number_train:
                number_train + number_val
            ]
        )

        test_groups.extend(
            class_groups[
                number_train + number_val:
            ]
        )

    train_group_set = set(train_groups)
    val_group_set = set(val_groups)
    test_group_set = set(test_groups)

    split_names = np.empty(
        len(groups),
        dtype=np.int8,
    )

    split_names.fill(-1)

    split_names[
        np.isin(groups, list(train_group_set))
    ] = 0

    split_names[
        np.isin(groups, list(val_group_set))
    ] = 1

    split_names[
        np.isin(groups, list(test_group_set))
    ] = 2

    split = {
        "train": np.flatnonzero(
            split_names == 0
        ),
        "val": np.flatnonzero(
            split_names == 1
        ),
        "test": np.flatnonzero(
            split_names == 2
        ),
    }

    for split_name, indices in split.items():

        if len(indices) == 0:
            raise ValueError(
                f"Empty {split_name} split."
            )

    assert not (
        train_group_set & val_group_set
    )

    assert not (
        train_group_set & test_group_set
    )

    assert not (
        val_group_set & test_group_set
    )

    return split


def make_windows(
    frame: pd.DataFrame,
    x: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    row_indices: np.ndarray,
    length: int,
    stride: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:

    selected = pd.DataFrame({
        "row_index": row_indices,
        "group": groups[row_indices],
        "sort_time": (
            frame.iloc[row_indices][
                "__sort_time__"
            ].to_numpy()
        ),
    })

    selected = selected.sort_values(
        by=[
            "group",
            "sort_time",
            "row_index",
        ],
        kind="stable",
    )

    sequences = []
    targets = []
    sequence_groups = []

    for group, group_frame in selected.groupby(
        "group",
        sort=False,
    ):

        indices = (
            group_frame["row_index"]
            .to_numpy(dtype=np.int64)
        )

        # CICIDS2017 groups shorter than T have already
        # been excluded by the dataset loader.
        if len(indices) < length:
            continue

        for start in range(
            0,
            len(indices) - length + 1,
            stride,
        ):

            window_indices = indices[
                start:start + length
            ]

            sequences.append(
                x[window_indices]
            )

            # Attack if any constituent record is attack
            targets.append(
                int(y[window_indices].max())
            )

            sequence_groups.append(group)

    if not sequences:
        raise ValueError(
            "No sequence windows were created."
        )

    return (
        np.asarray(
            sequences,
            dtype=np.float32,
        ),
        np.asarray(
            targets,
            dtype=np.int64,
        ),
        np.asarray(
            sequence_groups,
        ),
    )

print(
    "Efficient splitting and window functions loaded."
)

Efficient splitting and window functions loaded.


## 10. Configure CICIDS2017
Do not run training until the label and timestamp columns are verified.


In [27]:
cicids_cfg = Config(

    dataset_name="CICIDS2017",

    data_path=str(CICIDS_DATA),

    output_dir=str(
        PROJECT_ROOT
        / "outputs"
        / "CICIDS2017"
    ),

    label_column="Label",

    benign_values=(
        "BENIGN",
        "Benign",
        "benign",
        "0",
    ),

    group_column="__conversation_id__",

    timestamp_column="Timestamp",

    drop_columns=(
        "Flow ID",
        "Source IP",
        "Destination IP",
        "Src IP",
        "Dst IP",
        "Timestamp",
        "Label",
        "__conversation_id__",
    ),

    sequence_length=8,
    stride=8,

    # Exclude conversation groups shorter than one full window
    min_group_size=8,

    hidden_dim=128,
    skan_dim=96,
    swiglu_dim=96,
    state_dim=96,
    fusion_dim=96,

    num_basis=8,
    dropout=0.2,

    batch_size=256,
    epochs=60,
    patience=10,

    learning_rate=3e-4,
    weight_decay=1e-4,
    sparse_regularizer=1e-5,
    gradient_clip=1.0,

    seeds=(13, 27, 41, 55, 69),

    use_amp=True,
    deterministic=True,
    device="auto",
)

cicids_cfg

Config(dataset_name='CICIDS2017', data_path='<PROJECT_ROOT>\\data\\CICIDS2017_Full', output_dir='<PROJECT_ROOT>\\outputs\\CICIDS2017', label_column='Label', benign_values=('BENIGN', 'Benign', 'benign', '0'), group_column='__conversation_id__', timestamp_column='Timestamp', drop_columns=('Flow ID', 'Source IP', 'Destination IP', 'Src IP', 'Dst IP', 'Timestamp', 'Label', '__conversation_id__'), sequence_length=8, stride=8, min_group_size=8, train_fraction=0.7, val_fraction=0.15, test_fraction=0.15, hidden_dim=128, skan_dim=96, swiglu_dim=96, state_dim=96, fusion_dim=96, num_basis=8, basis_min=-3.0, basis_max=3.0, gate_temperature=0.2, dropout=0.2, batch_size=256, epochs=60, patience=10, learning_rate=0.0003, weight_decay=0.0001, sparse_regularizer=1e-05, gradient_clip=1.0, num_workers=0, seeds=(13, 27, 41, 55, 69), threshold_metric='f1', use_amp=True, deterministic=True, device='auto', tsne_max_samples=5000, tsne_perplexity=30.0)

In [29]:
# ============================================================
# Updated CSV collection loader
#
# DAPT2020:
# - UTF-8 encoding
# - automatic recovery of a headerless CSV
#
# CICIDS2017:
# - Windows-1252 encoding
# - removal of incomplete appended records
# - normalized protocol values
# - bidirectional conversation grouping
# - exclusion of groups shorter than T=8
# ============================================================

def read_csv_collection(
    path: str,
    label_column: Optional[str] = None,
) -> pd.DataFrame:

    dataset_path = Path(path)

    files = (
        [dataset_path]
        if dataset_path.is_file()
        else sorted(dataset_path.rglob("*.csv"))
    )

    if not files:
        raise FileNotFoundError(
            f"No CSV files found under "
            f"{dataset_path.resolve()}"
        )

    # CICIDS2017 uses Label and Windows-1252.
    # DAPT2020 uses Stage and UTF-8.
    is_cicids2017 = (
        label_column == "Label"
    )

    encoding = (
        "cp1252"
        if is_cicids2017
        else "utf-8-sig"
    )

    # --------------------------------------------------------
    # Locate a canonical header
    # --------------------------------------------------------

    canonical_columns = None

    if label_column is not None:

        for file in files:

            header = pd.read_csv(
                file,
                nrows=0,
                encoding=encoding,
                low_memory=False,
            )

            cleaned_columns = [
                str(column)
                .strip()
                .lstrip("\ufeff")
                for column in header.columns
            ]

            if label_column in cleaned_columns:
                canonical_columns = cleaned_columns
                break

    # --------------------------------------------------------
    # Read all source files
    # --------------------------------------------------------

    frames = []

    for file in files:

        header = pd.read_csv(
            file,
            nrows=0,
            encoding=encoding,
            low_memory=False,
        )

        cleaned_header = [
            str(column)
            .strip()
            .lstrip("\ufeff")
            for column in header.columns
        ]

        has_expected_header = (
            label_column is None
            or label_column in cleaned_header
        )

        if has_expected_header:

            frame = pd.read_csv(
                file,
                encoding=encoding,
                low_memory=False,
            )

            frame.columns = [
                str(column)
                .strip()
                .lstrip("\ufeff")
                for column in frame.columns
            ]

        else:

            if canonical_columns is None:
                raise ValueError(
                    "Could not recover the headerless "
                    f"CSV {file.name}: no canonical "
                    "header was found."
                )

            frame = pd.read_csv(
                file,
                header=None,
                encoding=encoding,
                low_memory=False,
            )

            if frame.shape[1] != len(
                canonical_columns
            ):
                raise ValueError(
                    f"Headerless CSV {file.name} has "
                    f"{frame.shape[1]} columns; the "
                    f"canonical schema has "
                    f"{len(canonical_columns)} columns."
                )

            frame.columns = canonical_columns

            print(
                "Recovered canonical header for:",
                file.name,
            )

        frame["__source_file__"] = file.name

        frame["__source_row__"] = np.arange(
            len(frame),
            dtype=np.int64,
        )

        frames.append(frame)

    combined = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    # --------------------------------------------------------
    # CICIDS2017-specific validation and grouping
    # --------------------------------------------------------

    if is_cicids2017:

        required_metadata = [
            "Flow ID",
            "Source IP",
            "Destination IP",
            "Protocol",
            "Timestamp",
            "Label",
        ]

        missing_columns = [
            column
            for column in required_metadata
            if column not in combined.columns
        ]

        if missing_columns:
            raise KeyError(
                "CICIDS2017 is missing required "
                f"metadata columns: {missing_columns}"
            )

        raw_count = len(combined)

        # Remove the incomplete block appended to the
        # Thursday Web Attacks CSV.
        complete_mask = (
            combined["Flow ID"].notna()
            & combined["Source IP"].notna()
            & combined["Destination IP"].notna()
            & combined["Protocol"].notna()
            & combined["Timestamp"].notna()
            & combined["Label"].notna()
        )

        combined = (
            combined.loc[complete_mask]
            .copy()
            .reset_index(drop=True)
        )

        removed_incomplete = (
            raw_count - len(combined)
        )

        valid_official_count = len(combined)

        # ----------------------------------------------------
        # Normalize endpoint and protocol values
        # ----------------------------------------------------

        source_ip = (
            combined["Source IP"]
            .astype("string")
            .str.strip()
        )

        destination_ip = (
            combined["Destination IP"]
            .astype("string")
            .str.strip()
        )

        protocol_numeric = pd.to_numeric(
            combined["Protocol"],
            errors="coerce",
        )

        if protocol_numeric.isna().any():
            raise ValueError(
                "Protocol contains invalid values after "
                "removing incomplete CICIDS2017 rows."
            )

        # Convert values such as 6.0 and 17.0 into
        # stable identifiers 6 and 17.
        protocol = (
            protocol_numeric
            .round()
            .astype("Int64")
            .astype("string")
        )

        # ----------------------------------------------------
        # Build a direction-independent conversation ID
        # ----------------------------------------------------

        source_array = source_ip.to_numpy()
        destination_array = (
            destination_ip.to_numpy()
        )

        endpoint_a = np.where(
            source_array <= destination_array,
            source_array,
            destination_array,
        )

        endpoint_b = np.where(
            source_array <= destination_array,
            destination_array,
            source_array,
        )

        combined["__conversation_id__"] = (
            pd.Series(
                endpoint_a,
                index=combined.index,
                dtype="string",
            )
            + "|"
            + pd.Series(
                endpoint_b,
                index=combined.index,
                dtype="string",
            )
            + "|"
            + protocol
        )

        if (
            combined["__conversation_id__"]
            .isna()
            .any()
        ):
            raise ValueError(
                "Missing conversation identifiers were "
                "generated for CICIDS2017."
            )

        # ----------------------------------------------------
        # Keep only groups that form at least one complete
        # sequence of length eight
        # ----------------------------------------------------

        conversation_sizes = (
            combined["__conversation_id__"]
            .value_counts()
        )

        eligible_conversations = (
            conversation_sizes[
                conversation_sizes >= 8
            ]
            .index
        )

        before_short_filter = len(combined)

        combined = (
            combined.loc[
                combined[
                    "__conversation_id__"
                ].isin(eligible_conversations)
            ]
            .copy()
            .reset_index(drop=True)
        )

        removed_short = (
            before_short_filter - len(combined)
        )

        retained_conversations = int(
            combined[
                "__conversation_id__"
            ].nunique()
        )

        minimum_conversation_size = int(
            combined[
                "__conversation_id__"
            ]
            .value_counts()
            .min()
        )

        # ----------------------------------------------------
        # Report and verify the preprocessing outcome
        # ----------------------------------------------------

        print(
            "CICIDS2017 raw rows:",
            f"{raw_count:,}",
        )

        print(
            "Removed incomplete rows:",
            f"{removed_incomplete:,}",
        )

        print(
            "Valid official records:",
            f"{valid_official_count:,}",
        )

        print(
            "Removed records from groups < 8:",
            f"{removed_short:,}",
        )

        print(
            "Records retained for sequencing:",
            f"{len(combined):,}",
        )

        print(
            "Eligible conversation groups:",
            f"{retained_conversations:,}",
        )

        print(
            "Minimum retained group size:",
            minimum_conversation_size,
        )

        retention_percentage = (
            100.0
            * len(combined)
            / valid_official_count
        )

        print(
            "Valid-record retention:",
            f"{retention_percentage:.2f}%",
        )

        assert raw_count == 3_119_345

        assert removed_incomplete == 288_602

        assert valid_official_count == 2_830_743

        assert removed_short == 127_833

        assert len(combined) == 2_702_910

        assert retained_conversations == 27_464

        assert minimum_conversation_size >= 8

        print(
            "CICIDS2017 loader verification passed."
        )

    return combined

In [30]:
# ============================================================
# Efficient group-disjoint splitting and sequence construction
# ============================================================

def group_stratified_split(
    groups: np.ndarray,
    labels: np.ndarray,
    cfg: Config,
    seed: int,
) -> Dict[str, np.ndarray]:

    unique_groups, inverse = np.unique(
        groups,
        return_inverse=True,
    )

    # A conversation group is attack-associated if it
    # contains at least one malicious record.
    group_labels = np.zeros(
        len(unique_groups),
        dtype=np.int64,
    )

    np.maximum.at(
        group_labels,
        inverse,
        labels,
    )

    rng = np.random.default_rng(seed)

    train_groups = []
    val_groups = []
    test_groups = []

    for class_value in np.unique(
        group_labels
    ):

        class_groups = unique_groups[
            group_labels == class_value
        ].copy()

        rng.shuffle(class_groups)

        number_of_groups = len(
            class_groups
        )

        number_train = max(
            1,
            int(
                round(
                    cfg.train_fraction
                    * number_of_groups
                )
            ),
        )

        number_val = (
            max(
                1,
                int(
                    round(
                        cfg.val_fraction
                        * number_of_groups
                    )
                ),
            )
            if number_of_groups >= 3
            else 0
        )

        if (
            number_train + number_val
            >= number_of_groups
        ):

            number_train = (
                max(1, number_of_groups - 2)
                if number_of_groups >= 3
                else max(1, number_of_groups - 1)
            )

            number_val = (
                1
                if number_of_groups >= 3
                else 0
            )

        train_groups.extend(
            class_groups[:number_train]
        )

        val_groups.extend(
            class_groups[
                number_train:
                number_train + number_val
            ]
        )

        test_groups.extend(
            class_groups[
                number_train + number_val:
            ]
        )

    train_group_set = set(
        train_groups
    )

    val_group_set = set(
        val_groups
    )

    test_group_set = set(
        test_groups
    )

    # Verify group-disjoint partitions.
    assert not (
        train_group_set & val_group_set
    )

    assert not (
        train_group_set & test_group_set
    )

    assert not (
        val_group_set & test_group_set
    )

    split_assignment = np.full(
        len(groups),
        fill_value=-1,
        dtype=np.int8,
    )

    split_assignment[
        np.isin(
            groups,
            list(train_group_set),
        )
    ] = 0

    split_assignment[
        np.isin(
            groups,
            list(val_group_set),
        )
    ] = 1

    split_assignment[
        np.isin(
            groups,
            list(test_group_set),
        )
    ] = 2

    split = {
        "train": np.flatnonzero(
            split_assignment == 0
        ),
        "val": np.flatnonzero(
            split_assignment == 1
        ),
        "test": np.flatnonzero(
            split_assignment == 2
        ),
    }

    for split_name, indices in (
        split.items()
    ):

        if len(indices) == 0:
            raise ValueError(
                f"Empty {split_name} split."
            )

    print(
        "Conversation groups:",
        f"{len(unique_groups):,}",
    )

    print(
        "Training groups:",
        f"{len(train_group_set):,}",
    )

    print(
        "Validation groups:",
        f"{len(val_group_set):,}",
    )

    print(
        "Test groups:",
        f"{len(test_group_set):,}",
    )

    return split


def make_windows(
    frame: pd.DataFrame,
    x: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    row_indices: np.ndarray,
    length: int,
    stride: int,
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:

    selected = pd.DataFrame({
        "row_index": row_indices,
        "group": groups[row_indices],
        "sort_time": (
            frame.iloc[row_indices][
                "__sort_time__"
            ].to_numpy()
        ),
    })

    selected = selected.sort_values(
        by=[
            "group",
            "sort_time",
            "row_index",
        ],
        kind="stable",
    )

    sequences = []
    targets = []
    sequence_groups = []

    for group, group_frame in (
        selected.groupby(
            "group",
            sort=False,
        )
    ):

        indices = (
            group_frame["row_index"]
            .to_numpy(dtype=np.int64)
        )

        # Do not create padded pseudo-sequences.
        if len(indices) < length:
            continue

        for start in range(
            0,
            len(indices) - length + 1,
            stride,
        ):

            window_indices = indices[
                start:start + length
            ]

            sequences.append(
                x[window_indices]
            )

            # Malicious if any record in the sequence
            # is labeled as an attack.
            targets.append(
                int(
                    y[window_indices].max()
                )
            )

            sequence_groups.append(
                group
            )

    if not sequences:
        raise ValueError(
            "No sequence windows were created."
        )

    sequence_array = np.asarray(
        sequences,
        dtype=np.float32,
    )

    target_array = np.asarray(
        targets,
        dtype=np.int64,
    )

    group_array = np.asarray(
        sequence_groups,
    )

    return (
        sequence_array,
        target_array,
        group_array,
    )


print(
    "Efficient splitting and window functions loaded."
)


Efficient splitting and window functions loaded.


## 11. Run CICIDS2017
Uncomment only after its schema is verified.


In [31]:
# ============================================================
# CICIDS2017 preprocessing and sequence preview
# ============================================================

import gc

# Release large audit objects that are no longer required.
objects_to_release = [
    "web_check",
    "valid_web",
    "flow_size_counter",
    "flow_label_sets",
    "directional_sizes",
    "bidirectional_sizes",
    "directional_label_bits",
    "bidirectional_label_bits",
    "group_sizes",
]

for object_name in objects_to_release:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

print(
    "Preparing CICIDS2017 data..."
)

prepared_cicids = prepare_tabular_data(
    cicids_cfg,
    seed=cicids_cfg.seeds[0],
)

print(
    "\nRetained feature dimensions:",
    len(
        prepared_cicids[
            "feature_columns"
        ]
    ),
)

print(
    "Retained feature names:"
)

print(
    prepared_cicids[
        "feature_columns"
    ]
)

cicids_sequence_summary = {}

for split_name, row_indices in (
    prepared_cicids["split"].items()
):

    print(
        f"\nConstructing {split_name} "
        "sequences..."
    )

    (
        sequence_x,
        sequence_y,
        sequence_groups,
    ) = make_windows(
        frame=prepared_cicids["frame"],
        x=prepared_cicids["x"],
        y=prepared_cicids["y"],
        groups=prepared_cicids["groups"],
        row_indices=row_indices,
        length=cicids_cfg.sequence_length,
        stride=cicids_cfg.stride,
    )

    cicids_sequence_summary[
        split_name
    ] = {
        "Records": len(row_indices),
        "Sequences": len(sequence_y),
        "Benign": int(
            (sequence_y == 0).sum()
        ),
        "Attack": int(
            (sequence_y == 1).sum()
        ),
        "Unique groups": int(
            len(
                np.unique(
                    sequence_groups
                )
            )
        ),
        "Shape": str(
            sequence_x.shape
        ),
    }

    # Preview arrays are not required for training.
    del sequence_x
    del sequence_y
    del sequence_groups

    gc.collect()


print(
    "\nCICIDS2017 sequence summary:"
)

cicids_sequence_summary_df = (
    pd.DataFrame(
        cicids_sequence_summary
    ).T
)

display(
    cicids_sequence_summary_df
)


# ============================================================
# Verify conversation-group isolation
# ============================================================

train_groups = set(
    prepared_cicids["groups"][
        prepared_cicids[
            "split"
        ]["train"]
    ]
)

validation_groups = set(
    prepared_cicids["groups"][
        prepared_cicids[
            "split"
        ]["val"]
    ]
)

test_groups = set(
    prepared_cicids["groups"][
        prepared_cicids[
            "split"
        ]["test"]
    ]
)

train_validation_overlap = len(
    train_groups
    & validation_groups
)

train_test_overlap = len(
    train_groups
    & test_groups
)

validation_test_overlap = len(
    validation_groups
    & test_groups
)

print(
    "Train–validation overlap:",
    train_validation_overlap,
)

print(
    "Train–test overlap:",
    train_test_overlap,
)

print(
    "Validation–test overlap:",
    validation_test_overlap,
)

assert train_validation_overlap == 0
assert train_test_overlap == 0
assert validation_test_overlap == 0


# ============================================================
# Verify sequence totals and class presence
# ============================================================

for split_name in [
    "train",
    "val",
    "test",
]:

    split_summary = (
        cicids_sequence_summary[
            split_name
        ]
    )

    assert (
        split_summary["Sequences"] > 0
    )

    assert (
        split_summary["Benign"] > 0
    )

    assert (
        split_summary["Attack"] > 0
    )

print(
    "\nCICIDS2017 preprocessing "
    "verification passed."
)


Preparing CICIDS2017 data...
CICIDS2017 raw rows: 3,119,345
Removed incomplete rows: 288,602
Valid official records: 2,830,743
Removed records from groups < 8: 127,833
Records retained for sequencing: 2,702,910
Eligible conversation groups: 27,464
Minimum retained group size: 8
Valid-record retention: 95.48%
CICIDS2017 loader verification passed.
Conversation groups: 27,464
Training groups: 19,225
Validation groups: 4,120
Test groups: 4,119

Retained feature dimensions: 72
Retained feature names:
['Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd I

,Records,Sequences,Benign,Attack,Unique groups,Shape
train,1570014,189420,189239,181,19225,"(189420, 8, 72)"
val,291957,35045,34972,73,4120,"(35045, 8, 72)"
test,840939,103669,33453,70216,4119,"(103669, 8, 72)"


Train–validation overlap: 0
Train–test overlap: 0
Validation–test overlap: 0

CICIDS2017 preprocessing verification passed.


In [32]:
# ============================================================
# Diagnose CICIDS2017 sequence concentration by conversation
# ============================================================

diagnostic_frame = pd.DataFrame({
    "row_index": np.arange(
        len(prepared_cicids["frame"]),
        dtype=np.int64,
    ),
    "group": prepared_cicids["groups"],
    "label": prepared_cicids["y"],
    "sort_time": (
        prepared_cicids["frame"][
            "__sort_time__"
        ].to_numpy()
    ),
})

diagnostic_frame = (
    diagnostic_frame.sort_values(
        by=[
            "group",
            "sort_time",
            "row_index",
        ],
        kind="stable",
    )
)

group_sequence_statistics = []

window_length = (
    cicids_cfg.sequence_length
)

window_stride = (
    cicids_cfg.stride
)

for group, group_frame in (
    diagnostic_frame.groupby(
        "group",
        sort=False,
    )
):

    group_labels = (
        group_frame["label"]
        .to_numpy(dtype=np.int64)
    )

    benign_sequences = 0
    attack_sequences = 0
    total_sequences = 0

    for start in range(
        0,
        len(group_labels)
        - window_length
        + 1,
        window_stride,
    ):

        window_label = int(
            group_labels[
                start:
                start + window_length
            ].max()
        )

        total_sequences += 1

        if window_label == 1:
            attack_sequences += 1
        else:
            benign_sequences += 1

    group_sequence_statistics.append({
        "Conversation ID": group,
        "Records": len(group_labels),
        "Sequences": total_sequences,
        "Benign sequences": benign_sequences,
        "Attack sequences": attack_sequences,
        "Contains attack": int(
            attack_sequences > 0
        ),
    })


group_sequence_statistics_df = (
    pd.DataFrame(
        group_sequence_statistics
    )
)

print(
    "Conversation groups:",
    f"{len(group_sequence_statistics_df):,}"
)

print(
    "Total sequences:",
    f"{group_sequence_statistics_df['Sequences'].sum():,}"
)

print(
    "Total benign sequences:",
    f"{group_sequence_statistics_df['Benign sequences'].sum():,}"
)

print(
    "Total attack sequences:",
    f"{group_sequence_statistics_df['Attack sequences'].sum():,}"
)

print(
    "Groups containing attack sequences:",
    f"{group_sequence_statistics_df['Contains attack'].sum():,}"
)

print(
    "\nLargest attack-sequence contributors:"
)

display(
    group_sequence_statistics_df
    .sort_values(
        "Attack sequences",
        ascending=False,
    )
    .head(20)
)

attack_total = (
    group_sequence_statistics_df[
        "Attack sequences"
    ].sum()
)

attack_concentration = (
    group_sequence_statistics_df
    .sort_values(
        "Attack sequences",
        ascending=False,
    )[
        "Attack sequences"
    ]
    .cumsum()
    / max(attack_total, 1)
)

for top_k in [1, 3, 5, 10, 20, 50]:

    concentration_value = float(
        attack_concentration
        .iloc[
            min(
                top_k - 1,
                len(
                    attack_concentration
                ) - 1,
            )
        ]
    )

    print(
        f"Attack sequences in top {top_k} groups:",
        f"{100.0 * concentration_value:.2f}%",
    )

print(
    "\nConversation-group concentration "
    "diagnostic completed."
)

Conversation groups: 27,464
Total sequences: 328,134
Total benign sequences: 257,664
Total attack sequences: 70,470
Groups containing attack sequences: 7

Largest attack-sequence contributors:


,Conversation ID,Records,Sequences,Benign sequences,Attack sequences,Contains attack
6050,172.16.0.1|192.168.10.50|6,598411,74801,4585,70216,1
13496,192.168.10.15|205.174.165.73|6,858,107,34,73,1
24190,192.168.10.8|205.174.165.73|6,688,86,34,52,1
25817,192.168.10.9|205.174.165.73|6,514,64,17,47,1
11682,192.168.10.14|205.174.165.73|6,487,60,17,43,1
22099,192.168.10.5|205.174.165.73|6,427,53,17,36,1
6052,172.16.0.1|192.168.10.51|6,8534,1066,1063,3,1
18338,192.168.10.19|209.85.232.109|6,8,1,1,0,0
18312,192.168.10.19|209.197.3.19|6,57,7,7,0,0
18311,192.168.10.19|209.197.3.14|6,12,1,1,0,0


Attack sequences in top 1 groups: 99.64%
Attack sequences in top 3 groups: 99.82%
Attack sequences in top 5 groups: 99.94%
Attack sequences in top 10 groups: 100.00%
Attack sequences in top 20 groups: 100.00%
Attack sequences in top 50 groups: 100.00%

Conversation-group concentration diagnostic completed.


In [33]:
# ============================================================
# Compare temporal-block grouping for CICIDS2017
# ============================================================

import gc

# Release the previous large diagnostic table.
globals().pop(
    "diagnostic_frame",
    None,
)

gc.collect()

temporal_base = pd.DataFrame({
    "source_file": (
        prepared_cicids["frame"][
            "__source_file__"
        ].astype("string")
    ),
    "source_row": (
        prepared_cicids["frame"][
            "__source_row__"
        ].to_numpy(dtype=np.int64)
    ),
    "timestamp": pd.to_datetime(
        prepared_cicids["frame"][
            "__sort_time__"
        ],
        errors="coerce",
    ),
    "label": prepared_cicids[
        "y"
    ].astype(np.int64),
})

assert (
    temporal_base["timestamp"]
    .notna()
    .all()
)


def evaluate_temporal_blocks(
    base_frame: pd.DataFrame,
    interval: str,
    sequence_length: int = 8,
):

    block_frame = base_frame.copy()

    block_time = (
        block_frame["timestamp"]
        .dt.floor(interval)
    )

    block_frame["block_id"] = (
        block_frame["source_file"]
        + "|"
        + block_time.astype("string")
    )

    block_frame = (
        block_frame.sort_values(
            by=[
                "block_id",
                "timestamp",
                "source_row",
            ],
            kind="stable",
        )
    )

    block_statistics = []

    for block_id, current_block in (
        block_frame.groupby(
            "block_id",
            sort=False,
        )
    ):

        labels = (
            current_block["label"]
            .to_numpy(dtype=np.int64)
        )

        number_of_complete_sequences = (
            len(labels)
            // sequence_length
        )

        if number_of_complete_sequences == 0:
            continue

        usable_count = (
            number_of_complete_sequences
            * sequence_length
        )

        window_labels = (
            labels[:usable_count]
            .reshape(
                number_of_complete_sequences,
                sequence_length,
            )
            .max(axis=1)
        )

        benign_sequences = int(
            (window_labels == 0).sum()
        )

        attack_sequences = int(
            (window_labels == 1).sum()
        )

        block_statistics.append({
            "Block ID": block_id,
            "Records": len(labels),
            "Usable records": usable_count,
            "Sequences": (
                number_of_complete_sequences
            ),
            "Benign sequences": (
                benign_sequences
            ),
            "Attack sequences": (
                attack_sequences
            ),
            "Contains attack": int(
                attack_sequences > 0
            ),
        })

    statistics = pd.DataFrame(
        block_statistics
    )

    total_attack_sequences = int(
        statistics[
            "Attack sequences"
        ].sum()
    )

    sorted_attack_counts = (
        statistics[
            "Attack sequences"
        ]
        .sort_values(
            ascending=False
        )
        .reset_index(drop=True)
    )

    top_one_share = (
        float(
            sorted_attack_counts.iloc[0]
            / total_attack_sequences
        )
        if total_attack_sequences > 0
        else 0.0
    )

    top_five_share = (
        float(
            sorted_attack_counts
            .head(5)
            .sum()
            / total_attack_sequences
        )
        if total_attack_sequences > 0
        else 0.0
    )

    summary = {
        "Interval": interval,
        "Eligible blocks": len(statistics),
        "Attack blocks": int(
            statistics[
                "Contains attack"
            ].sum()
        ),
        "Total sequences": int(
            statistics[
                "Sequences"
            ].sum()
        ),
        "Benign sequences": int(
            statistics[
                "Benign sequences"
            ].sum()
        ),
        "Attack sequences": (
            total_attack_sequences
        ),
        "Usable records": int(
            statistics[
                "Usable records"
            ].sum()
        ),
        "Retention (%)": (
            100.0
            * statistics[
                "Usable records"
            ].sum()
            / len(base_frame)
        ),
        "Largest attack block (%)": (
            100.0 * top_one_share
        ),
        "Top-5 attack blocks (%)": (
            100.0 * top_five_share
        ),
    }

    return summary, statistics


temporal_results = {}
temporal_summaries = []

for interval in [
    "1min",
    "5min",
    "15min",
]:

    print(
        f"Evaluating {interval} blocks..."
    )

    summary, statistics = (
        evaluate_temporal_blocks(
            temporal_base,
            interval=interval,
            sequence_length=(
                cicids_cfg.sequence_length
            ),
        )
    )

    temporal_summaries.append(
        summary
    )

    temporal_results[
        interval
    ] = statistics


temporal_summary_df = pd.DataFrame(
    temporal_summaries
)

print(
    "\nTemporal-block comparison:"
)

display(
    temporal_summary_df
)

for interval in [
    "1min",
    "5min",
    "15min",
]:

    print(
        f"\nLargest attack contributors "
        f"for {interval}:"
    )

    display(
        temporal_results[
            interval
        ]
        .sort_values(
            "Attack sequences",
            ascending=False,
        )
        .head(10)
    )

print(
    "\nTemporal-block audit completed."
)

Evaluating 1min blocks...
Evaluating 5min blocks...
Evaluating 15min blocks...

Temporal-block comparison:


,Interval,Eligible blocks,Attack blocks,Total sequences,Benign sequences,Attack sequences,Usable records,Retention (%),Largest attack block (%),Top-5 attack blocks (%)
0,1min,2448,512,336788,263508,73280,2694304,99.681602,7.748362,27.237991
1,5min,493,140,337641,264310,73331,2701128,99.934071,16.081875,56.242244
2,15min,169,57,337788,264422,73366,2702304,99.977580,32.232914,82.963498



Largest attack contributors for 1min:


,Block ID,Records,Usable records,Sequences,Benign sequences,Attack sequences,Contains attack
208,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,46149,46144,5768,90,5678,1
205,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,44114,44112,5514,4,5510,1
207,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,35946,35944,4493,96,4397,1
2311,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,18333,18328,2291,55,2236,1
2312,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,17591,17584,2198,59,2139,1
2318,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,14847,14840,1855,80,1775,1
2325,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,15939,15936,1992,218,1774,1
2313,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,15417,15416,1927,162,1765,1
2321,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,15331,15328,1916,158,1758,1
2323,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,14619,14616,1827,72,1755,1



Largest attack contributors for 5min:


,Block ID,Records,Usable records,Sequences,Benign sequences,Attack sequences,Contains attack
41,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,97517,97512,12189,396,11793,1
468,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,69717,69712,8714,824,7890,1
466,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,68015,68008,8501,619,7882,1
467,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,67122,67120,8390,516,7874,1
42,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,50677,50672,6334,530,5804,1
465,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,39694,39688,4961,587,4374,1
6,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,43589,43584,5448,1249,4199,1
8,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,47017,47016,5877,1879,3998,1
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,43530,43528,5441,1553,3888,1
5,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,34036,34032,4254,1266,2988,1



Largest attack contributors for 15min:


,Block ID,Records,Usable records,Sequences,Benign sequences,Attack sequences,Contains attack
160,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,204854,204848,25606,1958,23648,1
14,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,152433,152432,19054,1458,17596,1
2,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,134136,134136,16767,4681,12086,1
159,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,51222,51216,6402,1852,4550,1
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,38493,38488,4811,1824,2987,1
161,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,30318,30312,3789,1760,2029,1
15,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,25540,25536,3192,1745,1447,1
16,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,24297,24296,3037,1841,1196,1
3,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.c...,19606,19600,2450,1496,954,1
162,Wednesday-workingHours.pcap_ISCX.csv|2017-07-0...,14134,14128,1766,1228,538,1



Temporal-block audit completed.


In [35]:
# ============================================================
# CSV loader with CICIDS2017 one-minute temporal grouping
# ============================================================

def read_csv_collection(
    path: str,
    label_column: Optional[str] = None,
) -> pd.DataFrame:

    dataset_path = Path(path)

    files = (
        [dataset_path]
        if dataset_path.is_file()
        else sorted(dataset_path.rglob("*.csv"))
    )

    if not files:
        raise FileNotFoundError(
            f"No CSV files found under "
            f"{dataset_path.resolve()}"
        )

    is_cicids2017 = (
        label_column == "Label"
    )

    encoding = (
        "cp1252"
        if is_cicids2017
        else "utf-8-sig"
    )

    # --------------------------------------------------------
    # Find a canonical header
    # --------------------------------------------------------

    canonical_columns = None

    if label_column is not None:

        for file in files:

            header = pd.read_csv(
                file,
                nrows=0,
                encoding=encoding,
                low_memory=False,
            )

            cleaned_columns = [
                str(column)
                .strip()
                .lstrip("\ufeff")
                for column in header.columns
            ]

            if label_column in cleaned_columns:
                canonical_columns = cleaned_columns
                break

    # --------------------------------------------------------
    # Read all CSV files
    # --------------------------------------------------------

    frames = []

    for file in files:

        header = pd.read_csv(
            file,
            nrows=0,
            encoding=encoding,
            low_memory=False,
        )

        cleaned_header = [
            str(column)
            .strip()
            .lstrip("\ufeff")
            for column in header.columns
        ]

        has_expected_header = (
            label_column is None
            or label_column in cleaned_header
        )

        if has_expected_header:

            frame = pd.read_csv(
                file,
                encoding=encoding,
                low_memory=False,
            )

            frame.columns = [
                str(column)
                .strip()
                .lstrip("\ufeff")
                for column in frame.columns
            ]

        else:

            if canonical_columns is None:
                raise ValueError(
                    "Could not recover the headerless "
                    f"CSV {file.name}: no canonical "
                    "header was found."
                )

            frame = pd.read_csv(
                file,
                header=None,
                encoding=encoding,
                low_memory=False,
            )

            if frame.shape[1] != len(
                canonical_columns
            ):
                raise ValueError(
                    f"Headerless CSV {file.name} has "
                    f"{frame.shape[1]} columns; the "
                    f"canonical schema has "
                    f"{len(canonical_columns)} columns."
                )

            frame.columns = canonical_columns

            print(
                "Recovered canonical header for:",
                file.name,
            )

        frame["__source_file__"] = file.name

        frame["__source_row__"] = np.arange(
            len(frame),
            dtype=np.int64,
        )

        frames.append(frame)

    combined = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    # --------------------------------------------------------
    # CICIDS2017-specific processing
    # --------------------------------------------------------

    if is_cicids2017:

        required_metadata = [
            "Flow ID",
            "Source IP",
            "Destination IP",
            "Protocol",
            "Timestamp",
            "Label",
        ]

        missing_columns = [
            column
            for column in required_metadata
            if column not in combined.columns
        ]

        if missing_columns:
            raise KeyError(
                "CICIDS2017 is missing required "
                f"metadata columns: {missing_columns}"
            )

        raw_count = len(combined)

        # Remove the incomplete block appended to the
        # Thursday Web Attacks file.
        complete_mask = (
            combined["Flow ID"].notna()
            & combined["Source IP"].notna()
            & combined["Destination IP"].notna()
            & combined["Protocol"].notna()
            & combined["Timestamp"].notna()
            & combined["Label"].notna()
        )

        combined = (
            combined.loc[complete_mask]
            .copy()
            .reset_index(drop=True)
        )

        removed_incomplete = (
            raw_count - len(combined)
        )

        valid_official_count = len(combined)

        # ----------------------------------------------------
        # Normalize protocol
        # ----------------------------------------------------

        protocol_numeric = pd.to_numeric(
            combined["Protocol"],
            errors="coerce",
        )

        if protocol_numeric.isna().any():
            raise ValueError(
                "Protocol contains invalid values after "
                "removing incomplete CICIDS2017 rows."
            )

        combined["Protocol"] = (
            protocol_numeric
            .round()
            .astype("Int64")
        )

        # ----------------------------------------------------
        # Parse timestamps
        # ----------------------------------------------------

        parsed_timestamp = pd.to_datetime(
            combined["Timestamp"],
            format="mixed",
            dayfirst=True,
            errors="coerce",
        )

        unparsed_timestamp_count = int(
            parsed_timestamp.isna().sum()
        )

        if unparsed_timestamp_count > 0:
            raise ValueError(
                "CICIDS2017 contains "
                f"{unparsed_timestamp_count:,} "
                "unparsed timestamps."
            )

        # ----------------------------------------------------
        # Create one-minute temporal-block identifiers
        # ----------------------------------------------------

        minute_timestamp = (
            parsed_timestamp.dt.floor("min")
        )

        combined["__time_block_id__"] = (
            combined["__source_file__"]
            .astype("string")
            + "|"
            + minute_timestamp.astype("string")
        )

        if (
            combined["__time_block_id__"]
            .isna()
            .any()
        ):
            raise ValueError(
                "Missing temporal-block identifiers "
                "were generated for CICIDS2017."
            )

        # ----------------------------------------------------
        # Retain blocks containing at least eight records
        # ----------------------------------------------------

        block_sizes = (
            combined["__time_block_id__"]
            .value_counts()
        )

        eligible_blocks = (
            block_sizes[
                block_sizes >= 8
            ]
            .index
        )

        before_short_filter = len(combined)

        combined = (
            combined.loc[
                combined[
                    "__time_block_id__"
                ].isin(eligible_blocks)
            ]
            .copy()
            .reset_index(drop=True)
        )

        removed_short = (
            before_short_filter - len(combined)
        )

        retained_blocks = int(
            combined[
                "__time_block_id__"
            ].nunique()
        )

        minimum_block_size = int(
            combined[
                "__time_block_id__"
            ]
            .value_counts()
            .min()
        )

        retention_percentage = (
            100.0
            * len(combined)
            / valid_official_count
        )

        # ----------------------------------------------------
        # Report and verify
        # ----------------------------------------------------

        print(
            "CICIDS2017 raw rows:",
            f"{raw_count:,}",
        )

        print(
            "Removed incomplete rows:",
            f"{removed_incomplete:,}",
        )

        print(
            "Valid official records:",
            f"{valid_official_count:,}",
        )

        print(
            "Removed records from blocks < 8:",
            f"{removed_short:,}",
        )

        print(
            "Records retained for sequencing:",
            f"{len(combined):,}",
        )

        print(
            "Eligible one-minute blocks:",
            f"{retained_blocks:,}",
        )

        print(
            "Minimum retained block size:",
            minimum_block_size,
        )

        print(
            "Valid-record retention:",
            f"{retention_percentage:.2f}%",
        )

        assert raw_count == 3_119_345
        assert removed_incomplete == 288_602
        assert valid_official_count == 2_830_743
        assert minimum_block_size >= 8

        print(
            "CICIDS2017 temporal-block "
            "loader verification passed."
        )

    return combined


print(
    "Updated temporal-block CSV loader loaded."
)

Updated temporal-block CSV loader loaded.


In [36]:
# ============================================================
# Weighted temporal-block splitting
#
# The splitter:
# - keeps every one-minute block in exactly one partition;
# - estimates benign/attack sequence counts per block;
# - balances both classes toward 70/15/15;
# - uses the same fixed split seed for reproducibility.
# ============================================================

def group_stratified_split(
    groups: np.ndarray,
    labels: np.ndarray,
    cfg: Config,
    seed: int,
) -> Dict[str, np.ndarray]:

    unique_groups, inverse = np.unique(
        groups,
        return_inverse=True,
    )

    number_of_groups = len(
        unique_groups
    )

    sequence_length = int(
        cfg.sequence_length
    )

    sequence_stride = int(
        cfg.stride
    )

    # --------------------------------------------------------
    # Collect each block's rows in their original stable order
    # --------------------------------------------------------

    stable_order = np.argsort(
        inverse,
        kind="stable",
    )

    sorted_inverse = inverse[
        stable_order
    ]

    group_starts = np.r_[
        0,
        np.flatnonzero(
            np.diff(sorted_inverse)
        ) + 1,
    ]

    group_ends = np.r_[
        group_starts[1:],
        len(stable_order),
    ]

    # Columns:
    # 0 = estimated benign sequences
    # 1 = estimated attack sequences
    group_sequence_counts = np.zeros(
        (number_of_groups, 2),
        dtype=np.int64,
    )

    for group_index, (
        start_position,
        end_position,
    ) in enumerate(
        zip(
            group_starts,
            group_ends,
        )
    ):

        row_indices = stable_order[
            start_position:end_position
        ]

        group_labels = labels[
            row_indices
        ]

        benign_sequences = 0
        attack_sequences = 0

        for window_start in range(
            0,
            len(group_labels)
            - sequence_length
            + 1,
            sequence_stride,
        ):

            window_label = int(
                group_labels[
                    window_start:
                    window_start
                    + sequence_length
                ].max()
            )

            if window_label == 1:
                attack_sequences += 1
            else:
                benign_sequences += 1

        group_sequence_counts[
            group_index,
            0,
        ] = benign_sequences

        group_sequence_counts[
            group_index,
            1,
        ] = attack_sequences

    total_class_sequences = (
        group_sequence_counts.sum(
            axis=0
        )
    )

    if np.any(
        total_class_sequences == 0
    ):
        raise ValueError(
            "At least one sequence class is absent "
            "before partitioning."
        )

    # --------------------------------------------------------
    # Set target sequence counts
    # --------------------------------------------------------

    split_names = [
        "train",
        "val",
        "test",
    ]

    split_fractions = np.asarray(
        [
            cfg.train_fraction,
            cfg.val_fraction,
            cfg.test_fraction,
        ],
        dtype=np.float64,
    )

    split_fractions = (
        split_fractions
        / split_fractions.sum()
    )

    target_counts = (
        split_fractions[:, None]
        * total_class_sequences[None, :]
    )

    current_counts = np.zeros(
        (3, 2),
        dtype=np.float64,
    )

    assigned_split = np.full(
        number_of_groups,
        fill_value=-1,
        dtype=np.int8,
    )

    # --------------------------------------------------------
    # Allocate the most influential blocks first
    # --------------------------------------------------------

    contribution = np.maximum(
        (
            group_sequence_counts[:, 0]
            / total_class_sequences[0]
        ),
        (
            group_sequence_counts[:, 1]
            / total_class_sequences[1]
        ),
    )

    rng = np.random.default_rng(
        seed
    )

    random_tie_breaker = rng.random(
        number_of_groups
    )

    allocation_order = np.lexsort(
        (
            random_tie_breaker,
            -contribution,
        )
    )

    normalizer = np.maximum(
        target_counts,
        1.0,
    )

    for group_index in allocation_order:

        group_counts = (
            group_sequence_counts[
                group_index
            ].astype(np.float64)
        )

        candidate_costs = []

        for split_index in range(3):

            candidate_counts = (
                current_counts.copy()
            )

            candidate_counts[
                split_index
            ] += group_counts

            normalized_error = (
                (
                    candidate_counts
                    - target_counts
                )
                / normalizer
            )

            balance_cost = float(
                np.square(
                    normalized_error
                ).sum()
            )

            # Small secondary term to discourage highly
            # unequal block counts when class costs tie.
            block_counts = np.bincount(
                assigned_split[
                    assigned_split >= 0
                ],
                minlength=3,
            ).astype(np.float64)

            block_counts[
                split_index
            ] += 1.0

            expected_block_counts = (
                split_fractions
                * (
                    int(
                        (
                            assigned_split
                            >= 0
                        ).sum()
                    )
                    + 1
                )
            )

            group_balance_cost = float(
                np.square(
                    (
                        block_counts
                        - expected_block_counts
                    )
                    / np.maximum(
                        expected_block_counts,
                        1.0,
                    )
                ).sum()
            )

            candidate_costs.append(
                balance_cost
                + 1e-4
                * group_balance_cost
            )

        minimum_cost = min(
            candidate_costs
        )

        best_candidates = [
            index
            for index, cost
            in enumerate(
                candidate_costs
            )
            if np.isclose(
                cost,
                minimum_cost,
                rtol=1e-12,
                atol=1e-12,
            )
        ]

        selected_split = int(
            rng.choice(
                best_candidates
            )
        )

        assigned_split[
            group_index
        ] = selected_split

        current_counts[
            selected_split
        ] += group_counts

    # --------------------------------------------------------
    # Convert block assignments into row-index partitions
    # --------------------------------------------------------

    train_groups = unique_groups[
        assigned_split == 0
    ]

    validation_groups = unique_groups[
        assigned_split == 1
    ]

    test_groups = unique_groups[
        assigned_split == 2
    ]

    split = {
        "train": np.flatnonzero(
            np.isin(
                groups,
                train_groups,
            )
        ),
        "val": np.flatnonzero(
            np.isin(
                groups,
                validation_groups,
            )
        ),
        "test": np.flatnonzero(
            np.isin(
                groups,
                test_groups,
            )
        ),
    }

    # --------------------------------------------------------
    # Leakage and class-balance verification
    # --------------------------------------------------------

    train_group_set = set(
        train_groups
    )

    validation_group_set = set(
        validation_groups
    )

    test_group_set = set(
        test_groups
    )

    assert not (
        train_group_set
        & validation_group_set
    )

    assert not (
        train_group_set
        & test_group_set
    )

    assert not (
        validation_group_set
        & test_group_set
    )

    for split_index, split_name in enumerate(
        split_names
    ):

        if len(split[split_name]) == 0:
            raise ValueError(
                f"Empty {split_name} split."
            )

        if np.any(
            current_counts[
                split_index
            ] == 0
        ):
            raise ValueError(
                f"The {split_name} split does not "
                "contain both sequence classes."
            )

    split_summary = pd.DataFrame({
        "Split": split_names,
        "Blocks": [
            len(train_groups),
            len(validation_groups),
            len(test_groups),
        ],
        "Estimated benign sequences": (
            current_counts[:, 0]
            .astype(np.int64)
        ),
        "Estimated attack sequences": (
            current_counts[:, 1]
            .astype(np.int64)
        ),
        "Target benign sequences": (
            np.round(
                target_counts[:, 0]
            ).astype(np.int64)
        ),
        "Target attack sequences": (
            np.round(
                target_counts[:, 1]
            ).astype(np.int64)
        ),
    })

    print(
        "Weighted temporal-block split:"
    )

    display(
        split_summary
    )

    print(
        "Train–validation block overlap:",
        len(
            train_group_set
            & validation_group_set
        ),
    )

    print(
        "Train–test block overlap:",
        len(
            train_group_set
            & test_group_set
        ),
    )

    print(
        "Validation–test block overlap:",
        len(
            validation_group_set
            & test_group_set
        ),
    )

    return split


print(
    "Weighted temporal-block splitter loaded."
)

Weighted temporal-block splitter loaded.


In [37]:
# ============================================================
# Final CICIDS2017 configuration
# ============================================================

cicids_cfg = Config(

    dataset_name="CICIDS2017",

    data_path=str(
        CICIDS_DATA
    ),

    output_dir=str(
        PROJECT_ROOT
        / "outputs"
        / "CICIDS2017"
    ),

    label_column="Label",

    benign_values=(
        "BENIGN",
        "Benign",
        "benign",
        "0",
    ),

    # One-minute source-file-specific temporal blocks
    group_column="__time_block_id__",

    timestamp_column="Timestamp",

    drop_columns=(
        "Flow ID",
        "Source IP",
        "Destination IP",
        "Src IP",
        "Dst IP",
        "Timestamp",
        "Label",
        "__conversation_id__",
        "__time_block_id__",
    ),

    sequence_length=8,
    stride=8,
    min_group_size=8,

    train_fraction=0.70,
    val_fraction=0.15,
    test_fraction=0.15,

    hidden_dim=128,
    skan_dim=96,
    swiglu_dim=96,
    state_dim=96,
    fusion_dim=96,

    num_basis=8,
    dropout=0.2,

    batch_size=256,
    epochs=60,
    patience=10,

    learning_rate=3e-4,
    weight_decay=1e-4,
    sparse_regularizer=1e-5,
    gradient_clip=1.0,

    seeds=(
        13,
        27,
        41,
        55,
        69,
    ),

    use_amp=True,
    deterministic=True,
    device="auto",
)

cicids_cfg

Config(dataset_name='CICIDS2017', data_path='<PROJECT_ROOT>\\data\\CICIDS2017_Full', output_dir='<PROJECT_ROOT>\\outputs\\CICIDS2017', label_column='Label', benign_values=('BENIGN', 'Benign', 'benign', '0'), group_column='__time_block_id__', timestamp_column='Timestamp', drop_columns=('Flow ID', 'Source IP', 'Destination IP', 'Src IP', 'Dst IP', 'Timestamp', 'Label', '__conversation_id__', '__time_block_id__'), sequence_length=8, stride=8, min_group_size=8, train_fraction=0.7, val_fraction=0.15, test_fraction=0.15, hidden_dim=128, skan_dim=96, swiglu_dim=96, state_dim=96, fusion_dim=96, num_basis=8, basis_min=-3.0, basis_max=3.0, gate_temperature=0.2, dropout=0.2, batch_size=256, epochs=60, patience=10, learning_rate=0.0003, weight_decay=0.0001, sparse_regularizer=1e-05, gradient_clip=1.0, num_workers=0, seeds=(13, 27, 41, 55, 69), threshold_metric='f1', use_amp=True, deterministic=True, device='auto', tsne_max_samples=5000, tsne_perplexity=30.0)

In [38]:
# Release the previous CICIDS2017 preprocessing result

import gc

globals().pop(
    "prepared_cicids",
    None,
)

globals().pop(
    "cicids_sequence_summary",
    None,
)

globals().pop(
    "cicids_sequence_summary_df",
    None,
)

globals().pop(
    "temporal_base",
    None,
)

globals().pop(
    "temporal_results",
    None,
)

globals().pop(
    "group_sequence_statistics_df",
    None,
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Previous CICIDS2017 preview released."
)

Previous CICIDS2017 preview released.


In [39]:
# ============================================================
# CICIDS2017 temporal-block preprocessing preview
# ============================================================

import gc

print(
    "Preparing CICIDS2017 data..."
)

prepared_cicids = prepare_tabular_data(
    cicids_cfg,
    seed=cicids_cfg.seeds[0],
)

print(
    "\nRetained feature dimensions:",
    len(
        prepared_cicids[
            "feature_columns"
        ]
    ),
)

print(
    "\nRetained feature names:"
)

print(
    prepared_cicids[
        "feature_columns"
    ]
)


# ============================================================
# Construct and summarize each partition
# ============================================================

cicids_sequence_summary = {}

for split_name, row_indices in (
    prepared_cicids["split"].items()
):

    print(
        f"\nConstructing {split_name} "
        "sequences..."
    )

    (
        sequence_x,
        sequence_y,
        sequence_groups,
    ) = make_windows(
        frame=prepared_cicids["frame"],
        x=prepared_cicids["x"],
        y=prepared_cicids["y"],
        groups=prepared_cicids["groups"],
        row_indices=row_indices,
        length=cicids_cfg.sequence_length,
        stride=cicids_cfg.stride,
    )

    number_of_sequences = len(
        sequence_y
    )

    number_of_benign = int(
        (sequence_y == 0).sum()
    )

    number_of_attacks = int(
        (sequence_y == 1).sum()
    )

    cicids_sequence_summary[
        split_name
    ] = {
        "Records": int(
            len(row_indices)
        ),
        "Sequences": int(
            number_of_sequences
        ),
        "Benign": number_of_benign,
        "Attack": number_of_attacks,
        "Attack ratio (%)": (
            100.0
            * number_of_attacks
            / number_of_sequences
        ),
        "Unique blocks": int(
            len(
                np.unique(
                    sequence_groups
                )
            )
        ),
        "Shape": str(
            sequence_x.shape
        ),
    }

    # The preview arrays are not needed afterward.
    del sequence_x
    del sequence_y
    del sequence_groups

    gc.collect()


cicids_sequence_summary_df = (
    pd.DataFrame(
        cicids_sequence_summary
    ).T
)

print(
    "\nCICIDS2017 sequence summary:"
)

display(
    cicids_sequence_summary_df
)


# ============================================================
# Verify temporal-block isolation
# ============================================================

train_blocks = set(
    prepared_cicids["groups"][
        prepared_cicids[
            "split"
        ]["train"]
    ]
)

validation_blocks = set(
    prepared_cicids["groups"][
        prepared_cicids[
            "split"
        ]["val"]
    ]
)

test_blocks = set(
    prepared_cicids["groups"][
        prepared_cicids[
            "split"
        ]["test"]
    ]
)

train_validation_overlap = len(
    train_blocks
    & validation_blocks
)

train_test_overlap = len(
    train_blocks
    & test_blocks
)

validation_test_overlap = len(
    validation_blocks
    & test_blocks
)

print(
    "Train–validation block overlap:",
    train_validation_overlap,
)

print(
    "Train–test block overlap:",
    train_test_overlap,
)

print(
    "Validation–test block overlap:",
    validation_test_overlap,
)

assert train_validation_overlap == 0
assert train_test_overlap == 0
assert validation_test_overlap == 0


# ============================================================
# Verify class presence and approximate split proportions
# ============================================================

for split_name in [
    "train",
    "val",
    "test",
]:

    split_summary = (
        cicids_sequence_summary[
            split_name
        ]
    )

    assert (
        split_summary["Sequences"] > 0
    )

    assert (
        split_summary["Benign"] > 0
    )

    assert (
        split_summary["Attack"] > 0
    )


total_sequences = sum(
    summary["Sequences"]
    for summary
    in cicids_sequence_summary.values()
)

total_benign = sum(
    summary["Benign"]
    for summary
    in cicids_sequence_summary.values()
)

total_attacks = sum(
    summary["Attack"]
    for summary
    in cicids_sequence_summary.values()
)

partition_proportions = pd.DataFrame({
    "Split": [
        "train",
        "val",
        "test",
    ],
    "Sequence proportion (%)": [
        100.0
        * cicids_sequence_summary[
            split_name
        ]["Sequences"]
        / total_sequences
        for split_name in [
            "train",
            "val",
            "test",
        ]
    ],
    "Benign proportion (%)": [
        100.0
        * cicids_sequence_summary[
            split_name
        ]["Benign"]
        / total_benign
        for split_name in [
            "train",
            "val",
            "test",
        ]
    ],
    "Attack proportion (%)": [
        100.0
        * cicids_sequence_summary[
            split_name
        ]["Attack"]
        / total_attacks
        for split_name in [
            "train",
            "val",
            "test",
        ]
    ],
})

print(
    "\nActual partition proportions:"
)

display(
    partition_proportions
)


print(
    "\nTotal sequences:",
    f"{total_sequences:,}",
)

print(
    "Total benign sequences:",
    f"{total_benign:,}",
)

print(
    "Total attack sequences:",
    f"{total_attacks:,}",
)

print(
    "\nCICIDS2017 temporal-block "
    "preprocessing verification passed."
)

Preparing CICIDS2017 data...
CICIDS2017 raw rows: 3,119,345
Removed incomplete rows: 288,602
Valid official records: 2,830,743
Removed records from blocks < 8: 15
Records retained for sequencing: 2,830,728
Eligible one-minute blocks: 2,448
Minimum retained block size: 18
Valid-record retention: 100.00%
CICIDS2017 temporal-block loader verification passed.
Weighted temporal-block split:


,Split,Blocks,Estimated benign sequences,Estimated attack sequences,Target benign sequences,Target attack sequences
0,train,2036,195449,48365,195566,51374
1,val,207,41966,12492,41907,11009
2,test,205,41965,12534,41907,11009


Train–validation block overlap: 0
Train–test block overlap: 0
Validation–test block overlap: 0

Retained feature dimensions: 72

Retained feature names:
['Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Varian

,Records,Sequences,Benign,Attack,Attack ratio (%),Unique blocks,Shape
train,1957584,243814,195449,48365,19.836843,2036,"(243814, 8, 72)"
val,436399,54458,41966,12492,22.938779,207,"(54458, 8, 72)"
test,436745,54499,41965,12534,22.998587,205,"(54499, 8, 72)"


Train–validation block overlap: 0
Train–test block overlap: 0
Validation–test block overlap: 0

Actual partition proportions:


,Split,Sequence proportion (%),Benign proportion (%),Attack proportion (%)
0,train,69.113958,69.958122,65.900451
1,val,15.437210,15.021118,17.021161
2,test,15.448832,15.020760,17.078388



Total sequences: 352,771
Total benign sequences: 279,380
Total attack sequences: 73,391

CICIDS2017 temporal-block preprocessing verification passed.


In [40]:
# ============================================================
# CICIDS2017 five-epoch pilot
# ============================================================

import gc
from dataclasses import replace

# Release preprocessing-preview objects before the pilot.
globals().pop(
    "prepared_cicids",
    None,
)

globals().pop(
    "cicids_sequence_summary",
    None,
)

globals().pop(
    "cicids_sequence_summary_df",
    None,
)

globals().pop(
    "train_blocks",
    None,
)

globals().pop(
    "validation_blocks",
    None,
)

globals().pop(
    "test_blocks",
    None,
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


cicids_pilot_cfg = replace(

    cicids_cfg,

    output_dir=str(
        PROJECT_ROOT
        / "outputs"
        / "CICIDS2017_pilot"
    ),

    epochs=5,

    patience=5,

    seeds=(13,),
)


print(
    "Pilot output:",
    cicids_pilot_cfg.output_dir
)

print(
    "Pilot epochs:",
    cicids_pilot_cfg.epochs
)

print(
    "Pilot seeds:",
    cicids_pilot_cfg.seeds
)


run(
    cicids_pilot_cfg
)

Pilot output: <PROJECT_ROOT>\outputs\CICIDS2017_pilot
Pilot epochs: 5
Pilot seeds: (13,)
Device: cuda
CICIDS2017 raw rows: 3,119,345
Removed incomplete rows: 288,602
Valid official records: 2,830,743
Removed records from blocks < 8: 15
Records retained for sequencing: 2,830,728
Eligible one-minute blocks: 2,448
Minimum retained block size: 18
Valid-record retention: 100.00%
CICIDS2017 temporal-block loader verification passed.
Weighted temporal-block split:


,Split,Blocks,Estimated benign sequences,Estimated attack sequences,Target benign sequences,Target attack sequences
0,train,2036,195449,48365,195566,51374
1,val,207,41966,12492,41907,11009
2,test,205,41965,12534,41907,11009


Train–validation block overlap: 0
Train–test block overlap: 0
Validation–test block overlap: 0
{'train': {'sequences': 243814, 'benign': 195449, 'apt': 48365, 'unique_groups': 2036}, 'val': {'sequences': 54458, 'benign': 41966, 'apt': 12492, 'unique_groups': 207}, 'test': {'sequences': 54499, 'benign': 41965, 'apt': 12534, 'unique_groups': 205}}
Training seed 13
{'threshold': 0.9838637113571167, 'accuracy': 0.8998513734196958, 'balanced_accuracy': 0.7826918716275811, 'precision': 0.9978891077962285, 'recall': 0.5657411839795755, 'specificity': 0.9996425592755868, 'f1': 0.7220977596741344, 'mcc': 0.7065811933219389, 'auroc': 0.9951406531436923, 'auprc': 0.9839391705746248, 'tn': 41950, 'fp': 15, 'fn': 5443, 'tp': 7091, 'seed': 13, 'best_epoch': 5, 'parameters': 376361, 'mean_sparse_fraction': 0.7122700810432434, 'mean_w_skan': 0.316771537065506, 'mean_w_swiglu': 0.3278745710849762, 'mean_w_dsssm': 0.3553538918495178}


In [41]:
# ============================================================
# Official CICIDS2017 experiment: five independent seeds
# ============================================================

from dataclasses import replace
import gc
import torch

# Release pilot and preview objects before full training
for variable_name in [
    "prepared_cicids",
    "cicids_arrays_preview",
    "cicids_sequence_summary",
    "cicids_pilot_results",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Official configuration
cicids_official_cfg = replace(
    cicids_cfg,
    output_dir=str(
        PROJECT_ROOT
        / "outputs"
        / "CICIDS2017"
    ),
    epochs=60,
    patience=10,
    seeds=(13, 27, 41, 55, 69),
)

print("Official output:", cicids_official_cfg.output_dir)
print("Epoch limit:", cicids_official_cfg.epochs)
print("Early-stopping patience:", cicids_official_cfg.patience)
print("Seeds:", cicids_official_cfg.seeds)
print("Device:", cicids_official_cfg.device)

cicids_official_results = run(
    cicids_official_cfg
)

Official output: <PROJECT_ROOT>\outputs\CICIDS2017
Epoch limit: 60
Early-stopping patience: 10
Seeds: (13, 27, 41, 55, 69)
Device: auto
Device: cuda
CICIDS2017 raw rows: 3,119,345
Removed incomplete rows: 288,602
Valid official records: 2,830,743
Removed records from blocks < 8: 15
Records retained for sequencing: 2,830,728
Eligible one-minute blocks: 2,448
Minimum retained block size: 18
Valid-record retention: 100.00%
CICIDS2017 temporal-block loader verification passed.
Weighted temporal-block split:


,Split,Blocks,Estimated benign sequences,Estimated attack sequences,Target benign sequences,Target attack sequences
0,train,2036,195449,48365,195566,51374
1,val,207,41966,12492,41907,11009
2,test,205,41965,12534,41907,11009


Train–validation block overlap: 0
Train–test block overlap: 0
Validation–test block overlap: 0
{'train': {'sequences': 243814, 'benign': 195449, 'apt': 48365, 'unique_groups': 2036}, 'val': {'sequences': 54458, 'benign': 41966, 'apt': 12492, 'unique_groups': 207}, 'test': {'sequences': 54499, 'benign': 41965, 'apt': 12534, 'unique_groups': 205}}
Training seed 13
{'threshold': 0.9917667508125305, 'accuracy': 0.9727334446503606, 'balanced_accuracy': 0.942176032056621, 'precision': 0.9953371592539455, 'recall': 0.8855911919578746, 'specificity': 0.9987608721553676, 'f1': 0.9372625179430888, 'mcc': 0.9224816787127328, 'auroc': 0.9987613693137604, 'auprc': 0.9966274608338279, 'tn': 41913, 'fp': 52, 'fn': 1434, 'tp': 11100, 'seed': 13, 'best_epoch': 17, 'parameters': 376361, 'mean_sparse_fraction': 0.9509515166282654, 'mean_w_skan': 0.2905367314815521, 'mean_w_swiglu': 0.34559640288352966, 'mean_w_dsssm': 0.3638668656349182}
Training seed 27
{'threshold': 0.9605967402458191, 'accuracy': 0.96

## 12. Inspect CICIDS2017 results


In [42]:
# ============================================================
# Inspect official CICIDS2017 output artifacts
# ============================================================

from pathlib import Path
import pandas as pd
import json

cicids_results_dir = Path(
    cicids_official_cfg.output_dir
)

print("Results directory:", cicids_results_dir)
print("\nSaved artifacts:")

for path in sorted(
    cicids_results_dir.rglob("*")
):
    if path.is_file():
        print(
            path.relative_to(cicids_results_dir),
            f"({path.stat().st_size / 1024:.1f} KB)"
        )

all_seed_file = (
    cicids_results_dir
    / "all_seed_results.csv"
)

mean_std_file = (
    cicids_results_dir
    / "mean_std_results.csv"
)

if all_seed_file.exists():
    print("\nFive-seed results:")
    display(pd.read_csv(all_seed_file))

if mean_std_file.exists():
    print("\nMean and standard deviation:")
    display(pd.read_csv(mean_std_file))

prediction_files = sorted(
    cicids_results_dir.rglob("*prediction*.csv")
)

probability_files = sorted(
    cicids_results_dir.rglob("*probabilit*.csv")
)

print(
    "\nPrediction files:",
    [str(p.relative_to(cicids_results_dir))
     for p in prediction_files]
)

print(
    "Probability files:",
    [str(p.relative_to(cicids_results_dir))
     for p in probability_files]
)

Results directory: <PROJECT_ROOT>\outputs\CICIDS2017

Saved artifacts:
all_seed_results.csv (1.6 KB)
config.json (1.3 KB)
efficiency.json (0.3 KB)
figures\confusion_matrix_test.png (159.3 KB)
figures\training_validation_curves_best_seed.png (353.0 KB)
figures\tsne_after_training.png (1028.7 KB)
figures\tsne_before_training.png (949.1 KB)
final_results_for_figures.csv (1040.7 KB)
mean_std_results.csv (0.9 KB)
mean_std_results.json (1.7 KB)
prepared\row_splits.json (37616.3 KB)
prepared\scaler.joblib (2.3 KB)
prepared\schema.json (4.0 KB)
seed_13\best_model.pt (1488.4 KB)
seed_13\history.csv (6.3 KB)
seed_13\optimizer_settings.json (0.2 KB)
seed_13\test_metrics.json (0.6 KB)
seed_13\test_predictions.csv (804.3 KB)
seed_13\training_validation_curves.png (340.7 KB)
seed_27\best_model.pt (1488.4 KB)
seed_27\history.csv (6.9 KB)
seed_27\optimizer_settings.json (0.2 KB)
seed_27\test_metrics.json (0.6 KB)
seed_27\test_predictions.csv (819.3 KB)
seed_27\training_validation_curves.png (372.7 KB)

,threshold,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,auroc,auprc,...,fp,fn,tp,seed,best_epoch,parameters,mean_sparse_fraction,mean_w_skan,mean_w_swiglu,mean_w_dsssm
0,0.991767,0.972733,0.942176,0.995337,0.885591,0.998761,0.937263,0.922482,0.998761,0.996627,...,52,1434,11100,13,17,376361,0.950952,0.290537,0.345596,0.363867
1,0.960597,0.968165,0.932411,0.994686,0.866204,0.998618,0.926010,0.909368,0.998474,0.995370,...,58,1677,10857,27,18,376361,0.952643,0.361133,0.317622,0.321245
2,0.982364,0.962770,0.920011,0.996784,0.840833,0.999190,0.912191,0.893963,0.998646,0.996430,...,34,1995,10539,41,24,376361,0.957448,0.111375,0.621141,0.267483
3,0.956016,0.962238,0.920058,0.992756,0.841950,0.998165,0.911155,0.892240,0.998297,0.994393,...,77,1981,10553,55,22,376361,0.961942,0.325252,0.383740,0.291008
4,0.947306,0.953595,0.900821,0.993976,0.803096,0.998546,0.888399,0.867309,0.997845,0.994516,...,61,2468,10066,69,27,376361,0.968886,0.492137,0.168358,0.339505



Mean and standard deviation:


,Unnamed: 0,mean,std
0,threshold,0.967610,0.018689
1,accuracy,0.963900,0.007184
2,balanced_accuracy,0.923095,0.015538
3,precision,0.994708,0.001504
4,recall,0.847535,0.031015
5,specificity,0.998656,0.000371
6,f1,0.915003,0.018352
7,mcc,0.897072,0.020710
8,auroc,0.998405,0.000359
9,auprc,0.995467,0.001042



Prediction files: ['seed_13\\test_predictions.csv', 'seed_27\\test_predictions.csv', 'seed_41\\test_predictions.csv', 'seed_55\\test_predictions.csv', 'seed_69\\test_predictions.csv']
Probability files: []


In [43]:
# ============================================================
# Inspect prediction schema, threshold logic, and sparsity logic
# ============================================================

import inspect
import pandas as pd
from pathlib import Path

cicids_results_dir = Path(
    cicids_official_cfg.output_dir
)

# 1. Inspect seed-13 test predictions
seed13_predictions = pd.read_csv(
    cicids_results_dir
    / "seed_13"
    / "test_predictions.csv"
)

print("Prediction columns:")
print(seed13_predictions.columns.tolist())

print("\nPrediction preview:")
display(seed13_predictions.head())

print("\nPrediction dimensions:")
print(seed13_predictions.shape)

# 2. Find relevant functions/classes currently defined
relevant_objects = []

for object_name, object_value in list(globals().items()):
    lowered_name = object_name.lower()

    if any(
        keyword in lowered_name
        for keyword in [
            "threshold",
            "sparse",
            "skan",
            "evaluate",
            "predict",
        ]
    ):
        if (
            inspect.isfunction(object_value)
            or inspect.isclass(object_value)
        ):
            relevant_objects.append(object_name)

print("\nRelevant defined objects:")
print(sorted(relevant_objects))

# 3. Print source code for short relevant objects
for object_name in sorted(relevant_objects):
    object_value = globals()[object_name]

    try:
        source = inspect.getsource(object_value)

        if len(source) <= 8000:
            print("\n" + "=" * 70)
            print(object_name)
            print("=" * 70)
            print(source)

    except (TypeError, OSError):
        pass

Prediction columns:
['label', 'prob_apt']

Prediction preview:


,label,prob_apt
0,0,0.017043
1,0,0.011040
2,0,0.002681
3,0,0.007245
4,0,0.001400



Prediction dimensions:
(54499, 2)

Relevant defined objects:
['SparseKAN', 'evaluate_temporal_blocks', 'predict', 'select_threshold', 'sparse_loss']

evaluate_temporal_blocks
def evaluate_temporal_blocks(
    base_frame: pd.DataFrame,
    interval: str,
    sequence_length: int = 8,
):

    block_frame = base_frame.copy()

    block_time = (
        block_frame["timestamp"]
        .dt.floor(interval)
    )

    block_frame["block_id"] = (
        block_frame["source_file"]
        + "|"
        + block_time.astype("string")
    )

    block_frame = (
        block_frame.sort_values(
            by=[
                "block_id",
                "timestamp",
                "source_row",
            ],
            kind="stable",
        )
    )

    block_statistics = []

    for block_id, current_block in (
        block_frame.groupby(
            "block_id",
            sort=False,
        )
    ):

        labels = (
            current_block["label"]
            .to_numpy(dtype=np.i

In [44]:
# ============================================================
# Verify representative seed-13 results and saved efficiency
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

cicids_results_dir = Path(
    cicids_official_cfg.output_dir
)

# ------------------------------------------------------------
# Load representative seed-13 predictions and threshold
# ------------------------------------------------------------

seed13_predictions = pd.read_csv(
    cicids_results_dir
    / "seed_13"
    / "test_predictions.csv"
)

with open(
    cicids_results_dir
    / "seed_13"
    / "test_metrics.json",
    "r",
    encoding="utf-8",
) as file:
    saved_seed13_metrics = json.load(file)

seed13_threshold = float(
    saved_seed13_metrics["threshold"]
)

y_true = (
    seed13_predictions["label"]
    .to_numpy(dtype=np.int64)
)

y_probability = (
    seed13_predictions["prob_apt"]
    .to_numpy(dtype=np.float64)
)

y_predicted = (
    y_probability >= seed13_threshold
).astype(np.int64)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_predicted,
    labels=[0, 1],
).ravel()

verified_seed13 = {
    "representative_seed": 13,
    "selection_rule": "Predetermined first seed",
    "threshold_source": "Validation F1 maximization",
    "threshold": seed13_threshold,
    "accuracy": accuracy_score(
        y_true,
        y_predicted,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_true,
        y_predicted,
    ),
    "precision": precision_score(
        y_true,
        y_predicted,
        zero_division=0,
    ),
    "recall": recall_score(
        y_true,
        y_predicted,
        zero_division=0,
    ),
    "specificity": tn / max(tn + fp, 1),
    "f1": f1_score(
        y_true,
        y_predicted,
        zero_division=0,
    ),
    "mcc": matthews_corrcoef(
        y_true,
        y_predicted,
    ),
    "auroc": roc_auc_score(
        y_true,
        y_probability,
    ),
    "auprc": average_precision_score(
        y_true,
        y_probability,
    ),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
}

print("Verified representative result:")
print(verified_seed13)

# ------------------------------------------------------------
# Confirm seed 13 is the predetermined representative
# ------------------------------------------------------------

all_seed_results = pd.read_csv(
    cicids_results_dir
    / "all_seed_results.csv"
)

print("\nAll seeds ranked by AUPRC:")
display(
    all_seed_results[
        [
            "seed",
            "auprc",
            "auroc",
            "f1",
            "best_epoch",
        ]
    ]
    .sort_values(
        "auprc",
        ascending=False,
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Inspect efficiency measurements
# ------------------------------------------------------------

efficiency_file = (
    cicids_results_dir
    / "efficiency.json"
)

if efficiency_file.exists():
    with open(
        efficiency_file,
        "r",
        encoding="utf-8",
    ) as file:
        efficiency_results = json.load(file)

    print("\nSaved efficiency results:")
    print(efficiency_results)
else:
    print("\nNo efficiency.json file found.")

# ------------------------------------------------------------
# Inspect master CSV record types
# ------------------------------------------------------------

master_file = (
    cicids_results_dir
    / "final_results_for_figures.csv"
)

if master_file.exists():
    master_results = pd.read_csv(
        master_file,
        low_memory=False,
    )

    print(
        "\nMaster CSV records:",
        f"{len(master_results):,}",
    )

    print("\nMaster CSV record types:")
    display(
        master_results["record_type"]
        .value_counts(dropna=False)
        .rename_axis("record_type")
        .to_frame("records")
    )

Verified representative result:
{'representative_seed': 13, 'selection_rule': 'Predetermined first seed', 'threshold_source': 'Validation F1 maximization', 'threshold': 0.9917667508125305, 'accuracy': 0.9727334446503606, 'balanced_accuracy': 0.942176032056621, 'precision': 0.9953371592539455, 'recall': 0.8855911919578746, 'specificity': 0.9987608721553676, 'f1': 0.9372625179430888, 'mcc': 0.9224816787127328, 'auroc': 0.9987613693137604, 'auprc': 0.9966274608338279, 'tn': 41913, 'fp': 52, 'fn': 1434, 'tp': 11100}

All seeds ranked by AUPRC:


,seed,auprc,auroc,f1,best_epoch
0,13,0.996627,0.998761,0.937263,17
1,41,0.996430,0.998646,0.912191,24
2,27,0.995370,0.998474,0.926010,18
3,69,0.994516,0.997845,0.888399,27
4,55,0.994393,0.998297,0.911155,22



Saved efficiency results:
{'batch_size': 256, 'batch_latency_ms_mean': 6.599553999949421, 'batch_latency_ms_p95': 7.479535002130433, 'batch_latency_ms_p99': 7.760249000930341, 'latency_ms_per_sequence': 0.025779507812302427, 'throughput_sequences_per_second': 38790.50008560609, 'peak_gpu_memory_mb': 333.49658203125}

Master CSV records: 10,546

Master CSV record types:


,records
record_type,
tsne,10000
training_history,406
seed_metric,100
summary_metric,20
training_configuration,9
efficiency,7
confusion_matrix,4


In [46]:
# ============================================================
# Identify KAMBA++ model class and checkpoint structure
# ============================================================

import inspect
import torch
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# List notebook-defined classes
# ------------------------------------------------------------

notebook_classes = []

for object_name, object_value in list(globals().items()):

    if not inspect.isclass(object_value):
        continue

    object_module = getattr(
        object_value,
        "__module__",
        "",
    )

    # Retain classes defined inside this notebook/session
    if object_module == "__main__":
        try:
            signature = str(
                inspect.signature(object_value)
            )
        except (TypeError, ValueError):
            signature = "signature unavailable"

        notebook_classes.append({
            "Class": object_name,
            "Signature": signature,
        })

print("Notebook-defined classes:")

display(
    pd.DataFrame(notebook_classes)
    .sort_values("Class")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Inspect seed-13 checkpoint
# ------------------------------------------------------------

checkpoint_path = (
    Path(cicids_official_cfg.output_dir)
    / "seed_13"
    / "best_model.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

print("\nCheckpoint type:")
print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nTop-level checkpoint keys:")
    print(list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(
                f"{key}: dictionary with "
                f"{len(value):,} entries"
            )
        elif torch.is_tensor(value):
            print(
                f"{key}: tensor {tuple(value.shape)}"
            )
        else:
            print(
                f"{key}: {type(value).__name__} = "
                f"{value}"
            )

Notebook-defined classes:


,Class,Signature
0,AdaptiveController,(dim: 'int')
1,AdaptiveFusion,"(dims: 'Sequence[int]', fusion_dim: 'int', dro..."
2,Config,"(dataset_name: 'str' = 'DAPT2020', data_path: ..."
3,DSSSM,"(in_dim: 'int', state_dim: 'int', dropout: 'fl..."
4,EDyT,(dim: 'int')
5,KAMBAPlusPlus,"(input_dim: 'int', cfg: 'Config')"
6,SequenceDataset,"(x: 'np.ndarray', y: 'np.ndarray')"
7,SparseKAN,"(in_dim: 'int', out_dim: 'int', num_basis: 'in..."
8,SwiGLUBranch,"(in_dim: 'int', out_dim: 'int', dropout: 'float')"



Checkpoint type:
<class 'dict'>

Top-level checkpoint keys:
['model', 'config']
model: dictionary with 49 entries
config: dictionary with 39 entries


In [50]:
# ============================================================
# CICIDS2017 multi-batch efficiency benchmark
# Predetermined representative model: seed 13
# ============================================================

from pathlib import Path
import time
import gc
import json

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Paths and device
# ------------------------------------------------------------

cicids_results_dir = Path(
    cicids_official_cfg.output_dir
)

checkpoint_path = (
    cicids_results_dir
    / "seed_13"
    / "best_model.pt"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Benchmark device:", device)
print("Checkpoint:", checkpoint_path)

# ------------------------------------------------------------
# Reconstruct the unchanged test set
# ------------------------------------------------------------

print("\nReconstructing the official CICIDS2017 test set...")

prepared_benchmark = prepare_tabular_data(
    cicids_official_cfg,
    seed=13,
)

test_indices = (
    prepared_benchmark["split"]["test"]
)

test_x, test_y, test_groups = make_windows(
    frame=prepared_benchmark["frame"],
    x=prepared_benchmark["x"],
    y=prepared_benchmark["y"],
    groups=prepared_benchmark["groups"],
    row_indices=test_indices,
    length=cicids_official_cfg.sequence_length,
    stride=cicids_official_cfg.stride,
)

print("Test shape:", test_x.shape)
print("Test labels:", test_y.shape)
print("Test blocks:", len(set(test_groups)))

assert test_x.shape == (54499, 8, 72)
assert len(test_y) == 54499

# ------------------------------------------------------------
# Restore seed-13 model
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

model = KAMBAPlusPlus(
    input_dim=test_x.shape[-1],
    cfg=cicids_official_cfg,
)

model.load_state_dict(
    checkpoint["model"],
    strict=True,
)

model = model.to(device)
model.eval()

print(
    "Model parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)

# ------------------------------------------------------------
# Benchmark helper
# ------------------------------------------------------------

@torch.no_grad()
def benchmark_batch_size(
    model,
    x,
    y,
    batch_size,
    device,
    warmup_batches=20,
    measured_batches=200,
):
    dataset = SequenceDataset(x, y)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=True,
    )

    if len(loader) == 0:
        raise ValueError(
            f"Batch size {batch_size} exceeds "
            f"the available test samples."
        )

    # Warm-up
    warmup_iterator = iter(loader)

    for _ in range(warmup_batches):
        try:
            batch_x, _ = next(warmup_iterator)
        except StopIteration:
            warmup_iterator = iter(loader)
            batch_x, _ = next(warmup_iterator)

        batch_x = batch_x.to(
            device,
            non_blocking=True,
        )

        model(batch_x)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

    # Timed inference
    measured_iterator = iter(loader)
    batch_latencies_ms = []
    total_sequences = 0
    total_elapsed_seconds = 0.0

    for _ in range(measured_batches):
        try:
            batch_x, _ = next(measured_iterator)
        except StopIteration:
            measured_iterator = iter(loader)
            batch_x, _ = next(measured_iterator)

        batch_x = batch_x.to(
            device,
            non_blocking=True,
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        model(
            batch_x,
            return_aux=True,
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_seconds = (
            time.perf_counter() - start_time
        )

        batch_latencies_ms.append(
            elapsed_seconds * 1000.0
        )

        total_sequences += batch_x.shape[0]
        total_elapsed_seconds += elapsed_seconds

    latency_array = np.asarray(
        batch_latencies_ms,
        dtype=np.float64,
    )

    peak_memory_mb = (
        torch.cuda.max_memory_allocated(device)
        / (1024 ** 2)
        if device.type == "cuda"
        else np.nan
    )

    return {
        "batch_size": int(batch_size),
        "batch_latency_ms_mean": float(
            latency_array.mean()
        ),
        "batch_latency_ms_p95": float(
            np.percentile(latency_array, 95)
        ),
        "batch_latency_ms_p99": float(
            np.percentile(latency_array, 99)
        ),
        "latency_ms_per_sequence": float(
            1000.0
            * total_elapsed_seconds
            / total_sequences
        ),
        "throughput_sequences_per_second": float(
            total_sequences
            / total_elapsed_seconds
        ),
        "peak_gpu_memory_mb": float(
            peak_memory_mb
        ),
    }

# ------------------------------------------------------------
# Run benchmark
# ------------------------------------------------------------

batch_sizes = [1, 32, 128, 256]
efficiency_by_batch = []

for current_batch_size in batch_sizes:

    print(
        f"\nBenchmarking batch size "
        f"{current_batch_size}..."
    )

    current_result = benchmark_batch_size(
        model=model,
        x=test_x,
        y=test_y,
        batch_size=current_batch_size,
        device=device,
        warmup_batches=20,
        measured_batches=200,
    )

    efficiency_by_batch.append(
        current_result
    )

    print(current_result)

efficiency_by_batch_df = pd.DataFrame(
    efficiency_by_batch
)

print("\nCICIDS2017 multi-batch efficiency:")
display(efficiency_by_batch_df)

# ------------------------------------------------------------
# Save standalone CSV and JSON
# ------------------------------------------------------------

efficiency_csv_path = (
    cicids_results_dir
    / "efficiency_by_batch.csv"
)

efficiency_json_path = (
    cicids_results_dir
    / "efficiency_by_batch.json"
)

efficiency_by_batch_df.to_csv(
    efficiency_csv_path,
    index=False,
)

with open(
    efficiency_json_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        efficiency_by_batch,
        file,
        indent=2,
    )

# ------------------------------------------------------------
# Append safely to the master CSV
# ------------------------------------------------------------

master_csv_path = (
    cicids_results_dir
    / "final_results_for_figures.csv"
)

master_results = pd.read_csv(
    master_csv_path,
    low_memory=False,
)

# Prevent duplicates if this cell is rerun
master_results = master_results[
    master_results["record_type"]
    != "efficiency_by_batch"
].copy()

efficiency_master_rows = []

for row in efficiency_by_batch:
    current_batch_size = row["batch_size"]

    for metric_name, metric_value in row.items():
        if metric_name == "batch_size":
            continue

        efficiency_master_rows.append({
            "dataset": "CICIDS2017",
            "record_type": "efficiency_by_batch",
            "seed": 13,
            "split": "test",
            "metric": metric_name,
            "value": metric_value,
            "batch_size": current_batch_size,
        })

efficiency_master_df = pd.DataFrame(
    efficiency_master_rows
)

updated_master_results = pd.concat(
    [
        master_results,
        efficiency_master_df,
    ],
    ignore_index=True,
    sort=False,
)

updated_master_results.to_csv(
    master_csv_path,
    index=False,
)

print("\nSaved:", efficiency_csv_path)
print("Saved:", efficiency_json_path)
print("Updated:", master_csv_path)
print(
    "Updated master records:",
    f"{len(updated_master_results):,}",
)

# ------------------------------------------------------------
# Release benchmark objects
# ------------------------------------------------------------

del prepared_benchmark
del test_x
del test_y
del test_groups
del model
del checkpoint

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nMulti-batch efficiency benchmark completed.")

Benchmark device: cuda
Checkpoint: <PROJECT_ROOT>\outputs\CICIDS2017\seed_13\best_model.pt

Reconstructing the official CICIDS2017 test set...
CICIDS2017 raw rows: 3,119,345
Removed incomplete rows: 288,602
Valid official records: 2,830,743
Removed records from blocks < 8: 15
Records retained for sequencing: 2,830,728
Eligible one-minute blocks: 2,448
Minimum retained block size: 18
Valid-record retention: 100.00%
CICIDS2017 temporal-block loader verification passed.
Weighted temporal-block split:


,Split,Blocks,Estimated benign sequences,Estimated attack sequences,Target benign sequences,Target attack sequences
0,train,2036,195449,48365,195566,51374
1,val,207,41966,12492,41907,11009
2,test,205,41965,12534,41907,11009


Train–validation block overlap: 0
Train–test block overlap: 0
Validation–test block overlap: 0
Test shape: (54499, 8, 72)
Test labels: (54499,)
Test blocks: 205
Model parameters: 376,361

Benchmarking batch size 1...
{'batch_size': 1, 'batch_latency_ms_mean': 3.6580775005131727, 'batch_latency_ms_p95': 6.342179996863705, 'batch_latency_ms_p99': 7.256425011000827, 'latency_ms_per_sequence': 3.6580775005131727, 'throughput_sequences_per_second': 273.3676363772269, 'peak_gpu_memory_mb': 18.95654296875}

Benchmarking batch size 32...
{'batch_size': 32, 'batch_latency_ms_mean': 3.7957724995794706, 'batch_latency_ms_p95': 4.885585003648885, 'batch_latency_ms_p99': 6.533837002061765, 'latency_ms_per_sequence': 0.11861789061185846, 'throughput_sequences_per_second': 8430.431487541797, 'peak_gpu_memory_mb': 56.66943359375}

Benchmarking batch size 128...
{'batch_size': 128, 'batch_latency_ms_mean': 4.092497000383446, 'batch_latency_ms_p95': 5.121059999510179, 'batch_latency_ms_p99': 5.584852000

,batch_size,batch_latency_ms_mean,batch_latency_ms_p95,batch_latency_ms_p99,latency_ms_per_sequence,throughput_sequences_per_second,peak_gpu_memory_mb
0,1,3.658078,6.342180,7.256425,3.658078,273.367636,18.956543
1,32,3.795772,4.885585,6.533837,0.118618,8430.431488,56.669434
2,128,4.092497,5.121060,5.584852,0.031973,31276.748642,173.458496
3,256,6.282412,7.128410,7.610988,0.024541,40748.680607,329.178711



Saved: <PROJECT_ROOT>\outputs\CICIDS2017\efficiency_by_batch.csv
Saved: <PROJECT_ROOT>\outputs\CICIDS2017\efficiency_by_batch.json
Updated: <PROJECT_ROOT>\outputs\CICIDS2017\final_results_for_figures.csv
Updated master records: 10,570

Multi-batch efficiency benchmark completed.


In [51]:
# ============================================================
# Publication-quality CICIDS2017 figures and source data
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

RESULTS_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "CICIDS2017"
)

FIGURES_DIR = (
    PROJECT_ROOT
    / "Figures"
    / "CICIDS2017"
)

FIGURE_DATA_DIR = (
    FIGURES_DIR
    / "source_data"
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

history_path = (
    RESULTS_DIR
    / "seed_13"
    / "history.csv"
)

seed_results_path = (
    RESULTS_DIR
    / "all_seed_results.csv"
)

# ------------------------------------------------------------
# Publication style
# ------------------------------------------------------------

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": [
        "Times New Roman",
        "Times",
        "DejaVu Serif",
    ],
    "mathtext.fontset": "stix",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.linewidth": 1.0,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "lines.linewidth": 2.0,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "axes.facecolor": "#FCFCFC",
})

# Professional color palette
NAVY = "#173F5F"
BLUE = "#20639B"
TEAL = "#008C95"
GREEN = "#3CAEA3"
ORANGE = "#F28E2B"
RED = "#D1495B"
PURPLE = "#7656A3"
GOLD = "#C99700"
GRAY = "#6B7280"
LIGHT_GRID = "#D9DEE7"

# ============================================================
# Figure 1: Training and validation convergence
# ============================================================

history = pd.read_csv(history_path)

print("History columns:")
print(history.columns.tolist())

# Normalize column names
history.columns = [
    str(column).strip().lower()
    for column in history.columns
]

# Detect epoch column
epoch_column = (
    "epoch"
    if "epoch" in history.columns
    else history.columns[0]
)

# Retain source data
history.to_csv(
    FIGURE_DATA_DIR
    / "cicids2017_seed13_training_history.csv",
    index=False,
)

# Candidate columns
train_loss_column = next(
    (
        column
        for column in [
            "train_loss",
            "training_loss",
            "loss",
        ]
        if column in history.columns
    ),
    None,
)

validation_loss_column = next(
    (
        column
        for column in [
            "val_loss",
            "validation_loss",
        ]
        if column in history.columns
    ),
    None,
)

validation_metric_candidates = {
    "Validation F1": [
        "val_f1",
        "validation_f1",
        "f1",
    ],
    "Validation AUROC": [
        "val_auroc",
        "validation_auroc",
        "val_auc",
        "auroc",
    ],
    "Validation AUPRC": [
        "val_auprc",
        "validation_auprc",
        "auprc",
    ],
}

validation_columns = {}

for display_name, candidates in (
    validation_metric_candidates.items()
):
    selected_column = next(
        (
            column
            for column in candidates
            if column in history.columns
        ),
        None,
    )

    if selected_column is not None:
        validation_columns[
            display_name
        ] = selected_column

if train_loss_column is None:
    raise KeyError(
        "Training-loss column was not found. "
        f"Available columns: {history.columns.tolist()}"
    )

epochs = history[
    epoch_column
].to_numpy()

best_epoch = 17

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7.2, 3.15),
)

# ------------------------------------------------------------
# Panel (a): Loss convergence
# ------------------------------------------------------------

axes[0].plot(
    epochs,
    history[train_loss_column],
    color=NAVY,
    marker="o",
    markersize=3.2,
    markevery=max(len(epochs) // 10, 1),
    label="Training loss",
)

if validation_loss_column is not None:
    axes[0].plot(
        epochs,
        history[validation_loss_column],
        color=ORANGE,
        marker="s",
        markersize=3.0,
        markevery=max(len(epochs) // 10, 1),
        label="Validation loss",
    )

axes[0].axvline(
    best_epoch,
    color=RED,
    linestyle="--",
    linewidth=1.4,
    alpha=0.9,
    label=f"Selected epoch ({best_epoch})",
)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("(a) Optimization convergence")
axes[0].grid(
    True,
    linestyle="--",
    linewidth=0.55,
    color=LIGHT_GRID,
    alpha=0.85,
)
axes[0].legend(
    frameon=True,
    fancybox=False,
    edgecolor="#BBBBBB",
    loc="best",
)

# ------------------------------------------------------------
# Panel (b): Validation metrics
# ------------------------------------------------------------

metric_styles = {
    "Validation F1": {
        "color": RED,
        "marker": "o",
    },
    "Validation AUROC": {
        "color": BLUE,
        "marker": "s",
    },
    "Validation AUPRC": {
        "color": GREEN,
        "marker": "^",
    },
}

for display_name, column in (
    validation_columns.items()
):
    style = metric_styles[display_name]

    axes[1].plot(
        epochs,
        history[column],
        color=style["color"],
        marker=style["marker"],
        markersize=3.2,
        markevery=max(len(epochs) // 10, 1),
        label=display_name.replace(
            "Validation ",
            "",
        ),
    )

axes[1].axvline(
    best_epoch,
    color=RED,
    linestyle="--",
    linewidth=1.4,
    alpha=0.9,
)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation score")
axes[1].set_title("(b) Validation performance")
axes[1].set_ylim(0.0, 1.02)
axes[1].grid(
    True,
    linestyle="--",
    linewidth=0.55,
    color=LIGHT_GRID,
    alpha=0.85,
)
axes[1].legend(
    frameon=True,
    fancybox=False,
    edgecolor="#BBBBBB",
    loc="lower right",
)

for axis in axes:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(
        direction="out",
        length=3.5,
        width=0.8,
    )

fig.tight_layout(w_pad=2.0)

convergence_png = (
    FIGURES_DIR
    / "training_validation_curves_seed13.png"
)

convergence_pdf = (
    FIGURES_DIR
    / "training_validation_curves_seed13.pdf"
)

fig.savefig(convergence_png)
fig.savefig(convergence_pdf)
plt.show()
plt.close(fig)

# ============================================================
# Figure 2: All metrics across five seeds
# ============================================================

seed_results = pd.read_csv(
    seed_results_path
).sort_values("seed")

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "mcc",
    "auroc",
    "auprc",
]

metric_labels = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "specificity": "Specificity",
    "f1": "F1-score",
    "mcc": "MCC",
    "auroc": "AUROC",
    "auprc": "AUPRC",
}

missing_metrics = [
    metric
    for metric in metric_columns
    if metric not in seed_results.columns
]

if missing_metrics:
    raise KeyError(
        f"Missing metrics: {missing_metrics}"
    )

metric_source_data = seed_results[
    ["seed"] + metric_columns
].copy()

metric_source_data.to_csv(
    FIGURE_DATA_DIR
    / "cicids2017_five_seed_metrics.csv",
    index=False,
)

seeds = seed_results[
    "seed"
].to_numpy()

colors = [
    NAVY,
    BLUE,
    TEAL,
    RED,
    PURPLE,
    ORANGE,
    GREEN,
    GRAY,
    GOLD,
]

markers = [
    "o",
    "s",
    "^",
    "D",
    "v",
    "P",
    "X",
    "<",
    ">",
]

line_styles = [
    "-",
    "-",
    "--",
    "-",
    "--",
    "-.",
    "-.",
    ":",
    ":",
]

fig, axis = plt.subplots(
    figsize=(7.2, 4.15)
)

for (
    metric,
    color,
    marker,
    line_style,
) in zip(
    metric_columns,
    colors,
    markers,
    line_styles,
):
    axis.plot(
        seeds,
        seed_results[metric],
        label=metric_labels[metric],
        color=color,
        marker=marker,
        linestyle=line_style,
        linewidth=1.8,
        markersize=5.2,
        markeredgecolor="white",
        markeredgewidth=0.55,
    )

axis.set_xlabel("Random seed")
axis.set_ylabel("Test score")
axis.set_xticks(seeds)
axis.set_ylim(0.78, 1.01)

axis.grid(
    True,
    which="major",
    axis="both",
    linestyle="--",
    linewidth=0.55,
    color=LIGHT_GRID,
    alpha=0.9,
)

axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)

axis.tick_params(
    direction="out",
    length=3.5,
    width=0.8,
)

axis.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=False,
    columnspacing=1.4,
    handlelength=2.5,
)

fig.tight_layout()

multimetric_png = (
    FIGURES_DIR
    / "multimetric_seed_curves.png"
)

multimetric_pdf = (
    FIGURES_DIR
    / "multimetric_seed_curves.pdf"
)

fig.savefig(multimetric_png)
fig.savefig(multimetric_pdf)
plt.show()
plt.close(fig)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\nSaved figure files:")
print(convergence_png)
print(convergence_pdf)
print(multimetric_png)
print(multimetric_pdf)

print("\nSaved source-data files:")
print(
    FIGURE_DATA_DIR
    / "cicids2017_seed13_training_history.csv"
)
print(
    FIGURE_DATA_DIR
    / "cicids2017_five_seed_metrics.csv"
)

History columns:
['epoch', 'train_loss', 'learning_rate', 'threshold', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'mcc', 'auroc', 'auprc', 'tn', 'fp', 'fn', 'tp']


<TEMP_DIR>\ipykernel_12076\1636999287.py:341: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Saved figure files:
<PROJECT_ROOT>\Figures\CICIDS2017\training_validation_curves_seed13.png
<PROJECT_ROOT>\Figures\CICIDS2017\training_validation_curves_seed13.pdf
<PROJECT_ROOT>\Figures\CICIDS2017\multimetric_seed_curves.png
<PROJECT_ROOT>\Figures\CICIDS2017\multimetric_seed_curves.pdf

Saved source-data files:
<PROJECT_ROOT>\Figures\CICIDS2017\source_data\cicids2017_seed13_training_history.csv
<PROJECT_ROOT>\Figures\CICIDS2017\source_data\cicids2017_five_seed_metrics.csv


<TEMP_DIR>\ipykernel_12076\1636999287.py:512: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [52]:
# ============================================================
# Inspect KAMBA++ implementation before adding ablation switches
# ============================================================

import inspect
import pandas as pd

objects_to_inspect = [
    "KAMBAPlusPlus",
    "EDyT",
    "SparseKAN",
    "SwiGLUBranch",
    "DSSSM",
    "AdaptiveController",
    "AdaptiveFusion",
    "train_one_seed",
    "train_model",
    "run",
]

for object_name in objects_to_inspect:

    if object_name not in globals():
        continue

    object_value = globals()[object_name]

    print("\n" + "=" * 78)
    print(object_name)
    print("=" * 78)

    try:
        print(inspect.getsource(object_value))
    except (TypeError, OSError) as error:
        print(
            "Source unavailable:",
            error,
        )


KAMBAPlusPlus
Source unavailable: <class '__main__.KAMBAPlusPlus'> is a built-in class

EDyT
Source unavailable: <class '__main__.EDyT'> is a built-in class

SparseKAN
Source unavailable: <class '__main__.SparseKAN'> is a built-in class

SwiGLUBranch
Source unavailable: <class '__main__.SwiGLUBranch'> is a built-in class

DSSSM
Source unavailable: <class '__main__.DSSSM'> is a built-in class

AdaptiveController
Source unavailable: <class '__main__.AdaptiveController'> is a built-in class

AdaptiveFusion
Source unavailable: <class '__main__.AdaptiveFusion'> is a built-in class

train_one_seed
def train_one_seed(
    cfg: Config,
    seed: int,
    arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    device: torch.device,
) -> Dict[str, object]:
    set_seed(seed, cfg.deterministic)
    run_dir = Path(cfg.output_dir) / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    train_x, train_y = arrays["train"]
    val_x, val_y = arrays["val"]
    test_x, test_y = arrays["test"]

In [53]:
# ============================================================
# Inspect instantiated KAMBA++ architecture for ablation design
# ============================================================

import inspect
import torch
import pandas as pd

probe_model = KAMBAPlusPlus(
    input_dim=72,
    cfg=cicids_official_cfg,
)

print("=" * 78)
print("Complete model")
print("=" * 78)
print(probe_model)

print("\n" + "=" * 78)
print("Top-level modules")
print("=" * 78)

module_rows = []

for module_name, module in (
    probe_model.named_children()
):
    module_rows.append({
        "Attribute": module_name,
        "Class": module.__class__.__name__,
        "Trainable parameters": sum(
            parameter.numel()
            for parameter in module.parameters()
            if parameter.requires_grad
        ),
    })

display(pd.DataFrame(module_rows))

print("\n" + "=" * 78)
print("Model attributes")
print("=" * 78)

print(sorted(probe_model.__dict__.keys()))

print("\n" + "=" * 78)
print("Forward signature")
print("=" * 78)

print(
    inspect.signature(
        probe_model.forward
    )
)

if hasattr(
    probe_model.forward,
    "__code__",
):
    print("\nForward-referenced names:")
    print(
        probe_model.forward
        .__code__
        .co_names
    )

    print("\nForward local variables:")
    print(
        probe_model.forward
        .__code__
        .co_varnames
    )

print("\n" + "=" * 78)
print("Adaptive controller structure")
print("=" * 78)

print(probe_model.controller)

print("\n" + "=" * 78)
print("Adaptive fusion structure")
print("=" * 78)

print(probe_model.fusion)

if hasattr(
    probe_model.fusion.forward,
    "__code__",
):
    print("\nFusion forward names:")
    print(
        probe_model.fusion.forward
        .__code__
        .co_names
    )

    print("\nFusion forward variables:")
    print(
        probe_model.fusion.forward
        .__code__
        .co_varnames
    )

del probe_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Complete model
KAMBAPlusPlus(
  (input_projection): Linear(in_features=72, out_features=128, bias=True)
  (edyt): EDyT(
    (alpha): Linear(in_features=128, out_features=128, bias=True)
    (beta): Linear(in_features=128, out_features=128, bias=True)
    (gamma): Linear(in_features=128, out_features=128, bias=True)
  )
  (controller): AdaptiveController(
    (net): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): SiLU()
      (2): Linear(in_features=64, out_features=3, bias=True)
    )
  )
  (skan): SparseKAN()
  (swiglu): SwiGLUBranch(
    (w1): Linear(in_features=128, out_features=96, bias=True)
    (w2): Linear(in_features=128, out_features=96, bias=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (dsssm): DSSSM(
    (b_proj): Linear(in_features=128, out_features=96, bias=True)
    (r_x): Linear(in_features=128, out_features=96, bias=True)
    (r_s): Linear(in_features=96, out_features=96, bias=False)
    (u_x): Linear(in_features=128, ou

,Attribute,Class,Trainable parameters
0,input_projection,Linear,9344
1,edyt,EDyT,49536
2,controller,AdaptiveController,8451
3,skan,SparseKAN,122977
4,swiglu,SwiGLUBranch,24768
5,dsssm,DSSSM,77376
6,fusion,AdaptiveFusion,55971
7,classifier,Sequential,27938



Model attributes
['_backward_hooks', '_backward_pre_hooks', '_buffers', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_is_full_backward_hook', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_modules', '_non_persistent_buffers_set', '_parameters', '_state_dict_hooks', '_state_dict_pre_hooks', 'training']

Forward signature
(x: 'torch.Tensor', return_aux: 'bool' = False)

Forward-referenced names:
('input_projection', 'edyt', 'controller', 'skan', 'swiglu', 'dsssm', 'mean', 'fusion', 'classifier')

Forward local variables:
('self', 'x', 'return_aux', 'h0', 'he', 'descriptor', 'sparse_threshold', 'delta', 'tau', 'zs_seq', 'mask', 'zt_seq', 'zd_seq', 'branches', 'fused', 'weights', 'logits')

Adaptive controller structure
AdaptiveController(
  (net): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): SiLU()
    (2): Linear(in_features=64, out_features=3,

In [55]:
# ============================================================
# Verify branch and controller tensor interfaces — corrected
# ============================================================

import torch

probe_model = KAMBAPlusPlus(
    input_dim=72,
    cfg=cicids_official_cfg,
).eval()

probe_x = torch.randn(
    2,
    8,
    72,
)

with torch.no_grad():

    h0 = probe_model.input_projection(
        probe_x
    )

    # EDyT returns transformed features and descriptor
    he, descriptor = probe_model.edyt(
        h0
    )

    sparse_threshold, delta, tau = (
        probe_model.controller(
            descriptor
        )
    )

    print("Input projection:", h0.shape)
    print("EDyT representation:", he.shape)
    print("Global descriptor:", descriptor.shape)

    print("\nController outputs:")
    print(
        "Sparse threshold:",
        sparse_threshold.shape,
        float(sparse_threshold.min()),
        float(sparse_threshold.max()),
    )
    print(
        "Delta:",
        delta.shape,
        float(delta.min()),
        float(delta.max()),
    )
    print(
        "Tau:",
        tau.shape,
        float(tau.min()),
        float(tau.max()),
    )

    zs_seq, mask = probe_model.skan(
        he,
        sparse_threshold,
    )

    zt_seq = probe_model.swiglu(
        he
    )

    zd_seq = probe_model.dsssm(
        he,
        delta,
        tau,
    )

    print("\nBranch outputs:")
    print("SKAN sequence:", zs_seq.shape)
    print("SKAN mask:", mask.shape)
    print("SwiGLU sequence:", zt_seq.shape)
    print("DSSSM sequence:", zd_seq.shape)

    branches = [
        zs_seq.mean(dim=1),
        zt_seq.mean(dim=1),
        zd_seq.mean(dim=1),
    ]

    fused, weights = probe_model.fusion(
        branches
    )

    print("\nFusion:")
    print("SKAN descriptor:", branches[0].shape)
    print("SwiGLU descriptor:", branches[1].shape)
    print("DSSSM descriptor:", branches[2].shape)
    print("Fused representation:", fused.shape)
    print("Fusion weights:", weights.shape)

    logits, auxiliary = probe_model(
        probe_x,
        return_aux=True,
    )

    print("\nComplete forward pass:")
    print("Logits:", logits.shape)
    print("Auxiliary keys:", list(auxiliary.keys()))

    for key, value in auxiliary.items():
        if torch.is_tensor(value):
            print(
                f"{key}:",
                value.shape,
            )

del probe_model
del probe_x

Input projection: torch.Size([2, 8, 128])
EDyT representation: torch.Size([2, 8, 128])
Global descriptor: torch.Size([2, 128])

Controller outputs:
Sparse threshold: torch.Size([2]) 0.48657333850860596 0.49539050459861755
Delta: torch.Size([2]) 0.6935545206069946 0.7194525599479675
Tau: torch.Size([2]) 0.732893168926239 0.7368506789207458

Branch outputs:
SKAN sequence: torch.Size([2, 8, 96])
SKAN mask: torch.Size([2, 96, 128])
SwiGLU sequence: torch.Size([2, 8, 96])
DSSSM sequence: torch.Size([2, 8, 96])

Fusion:
SKAN descriptor: torch.Size([2, 96])
SwiGLU descriptor: torch.Size([2, 96])
DSSSM descriptor: torch.Size([2, 96])
Fused representation: torch.Size([2, 288])
Fusion weights: torch.Size([2, 3])

Complete forward pass:
Logits: torch.Size([2, 2])
Auxiliary keys: ['mask', 'fusion_weights', 'sparse_threshold', 'delta', 'tau', 'embedding']
mask: torch.Size([2, 96, 128])
fusion_weights: torch.Size([2, 3])
sparse_threshold: torch.Size([2])
delta: torch.Size([2])
tau: torch.Size([2])
e

In [56]:
# ============================================================
# KAMBA++ ablation modules and model builder
# ============================================================

import math
import torch
import torch.nn as nn
import pandas as pd


# ------------------------------------------------------------
# 1. EDyT removal: identity transformation
# ------------------------------------------------------------

class IdentityEDyT(nn.Module):

    def forward(self, x):
        descriptor = x.mean(dim=1)
        return x, descriptor


# ------------------------------------------------------------
# 2. SKAN removal: zero sparse branch
# ------------------------------------------------------------

class RemovedSKAN(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim

    def forward(
        self,
        x,
        sparse_threshold,
    ):
        batch_size, sequence_length, _ = x.shape

        output = x.new_zeros(
            batch_size,
            sequence_length,
            self.output_dim,
        )

        mask = x.new_zeros(
            batch_size,
            self.output_dim,
            self.input_dim,
        )

        return output, mask


# ------------------------------------------------------------
# 3. SwiGLU removal: zero temporal-gating branch
# ------------------------------------------------------------

class RemovedSwiGLU(nn.Module):

    def __init__(self, output_dim):
        super().__init__()
        self.output_dim = output_dim

    def forward(self, x):
        return x.new_zeros(
            x.shape[0],
            x.shape[1],
            self.output_dim,
        )


# ------------------------------------------------------------
# 4. DSSSM removal: zero state-space branch
# ------------------------------------------------------------

class RemovedDSSSM(nn.Module):

    def __init__(self, output_dim):
        super().__init__()
        self.output_dim = output_dim

    def forward(
        self,
        x,
        delta,
        tau,
    ):
        return x.new_zeros(
            x.shape[0],
            x.shape[1],
            self.output_dim,
        )


# ------------------------------------------------------------
# 5. Adaptive-controller removal
# Natural neutral values:
# threshold = 0.5
# delta = softplus(0) = ln(2)
# tau = softplus(0) = ln(2)
# ------------------------------------------------------------

class FixedController(nn.Module):

    def __init__(
        self,
        sparse_threshold=0.5,
        delta=math.log(2.0),
        tau=math.log(2.0),
    ):
        super().__init__()

        self.register_buffer(
            "fixed_sparse_threshold",
            torch.tensor(
                float(sparse_threshold)
            ),
        )

        self.register_buffer(
            "fixed_delta",
            torch.tensor(
                float(delta)
            ),
        )

        self.register_buffer(
            "fixed_tau",
            torch.tensor(
                float(tau)
            ),
        )

    def forward(self, descriptor):

        batch_size = descriptor.shape[0]

        sparse_threshold = (
            self.fixed_sparse_threshold
            .expand(batch_size)
        )

        delta = (
            self.fixed_delta
            .expand(batch_size)
        )

        tau = (
            self.fixed_tau
            .expand(batch_size)
        )

        return sparse_threshold, delta, tau


# ------------------------------------------------------------
# 6. Ablation-aware fusion
#
# The original fused representation has dimension 3 x 96 = 288.
# Removed branches retain an empty 96-dimensional slot so that
# the original classifier interface remains unchanged.
# ------------------------------------------------------------

class AblationFusion(nn.Module):

    def __init__(
        self,
        branch_dim,
        fusion_dim,
        active_indices,
        dropout,
        adaptive=True,
    ):
        super().__init__()

        self.branch_dim = branch_dim
        self.fusion_dim = fusion_dim
        self.active_indices = tuple(
            active_indices
        )
        self.adaptive = adaptive

        self.projections = nn.ModuleList([
            (
                nn.Linear(
                    branch_dim,
                    fusion_dim,
                )
                if index in self.active_indices
                else nn.Identity()
            )
            for index in range(3)
        ])

        if adaptive:
            self.weight_net = nn.Sequential(
                nn.Linear(
                    3 * branch_dim,
                    fusion_dim,
                ),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(
                    fusion_dim,
                    len(self.active_indices),
                ),
            )
        else:
            self.weight_net = None

    def forward(self, branches):

        if len(branches) != 3:
            raise ValueError(
                "AblationFusion expects three "
                "branch slots."
            )

        batch_size = branches[0].shape[0]

        if self.adaptive:
            fusion_input = torch.cat(
                branches,
                dim=-1,
            )

            active_weights = torch.softmax(
                self.weight_net(
                    fusion_input
                ),
                dim=-1,
            )
        else:
            active_weights = (
                branches[0].new_full(
                    (
                        batch_size,
                        len(self.active_indices),
                    ),
                    1.0
                    / len(self.active_indices),
                )
            )

        fused_slots = []
        complete_weights = []

        active_position = 0

        for branch_index in range(3):

            if branch_index in self.active_indices:

                projected = self.projections[
                    branch_index
                ](
                    branches[branch_index]
                )

                current_weight = (
                    active_weights[
                        :,
                        active_position:
                        active_position + 1,
                    ]
                )

                fused_slots.append(
                    current_weight
                    * projected
                )

                complete_weights.append(
                    current_weight
                )

                active_position += 1

            else:
                fused_slots.append(
                    branches[branch_index]
                    .new_zeros(
                        batch_size,
                        self.fusion_dim,
                    )
                )

                complete_weights.append(
                    branches[branch_index]
                    .new_zeros(
                        batch_size,
                        1,
                    )
                )

        fused = torch.cat(
            fused_slots,
            dim=-1,
        )

        weights = torch.cat(
            complete_weights,
            dim=-1,
        )

        return fused, weights


# ------------------------------------------------------------
# Ablation model builder
# ------------------------------------------------------------

ABLATION_VARIANTS = (
    "full",
    "without_edyt",
    "without_skan",
    "without_swiglu",
    "without_dsssm",
    "without_controller",
    "equal_fusion",
)


def build_ablation_model(
    input_dim,
    cfg,
    variant,
):

    if variant not in ABLATION_VARIANTS:
        raise ValueError(
            f"Unknown ablation variant: {variant}"
        )

    model = KAMBAPlusPlus(
        input_dim=input_dim,
        cfg=cfg,
    )

    # Full model: no replacement
    if variant == "full":
        return model

    # Without EDyT
    if variant == "without_edyt":
        model.edyt = IdentityEDyT()

    # Without SKAN
    elif variant == "without_skan":
        model.skan = RemovedSKAN(
            input_dim=cfg.hidden_dim,
            output_dim=cfg.skan_dim,
        )

        model.fusion = AblationFusion(
            branch_dim=cfg.fusion_dim,
            fusion_dim=cfg.fusion_dim,
            active_indices=(1, 2),
            dropout=cfg.dropout,
            adaptive=True,
        )

    # Without SwiGLU
    elif variant == "without_swiglu":
        model.swiglu = RemovedSwiGLU(
            output_dim=cfg.swiglu_dim,
        )

        model.fusion = AblationFusion(
            branch_dim=cfg.fusion_dim,
            fusion_dim=cfg.fusion_dim,
            active_indices=(0, 2),
            dropout=cfg.dropout,
            adaptive=True,
        )

    # Without DSSSM
    elif variant == "without_dsssm":
        model.dsssm = RemovedDSSSM(
            output_dim=cfg.state_dim,
        )

        model.fusion = AblationFusion(
            branch_dim=cfg.fusion_dim,
            fusion_dim=cfg.fusion_dim,
            active_indices=(0, 1),
            dropout=cfg.dropout,
            adaptive=True,
        )

    # Without adaptive controller
    elif variant == "without_controller":
        model.controller = FixedController()

    # Equal-weight fusion
    elif variant == "equal_fusion":
        model.fusion = AblationFusion(
            branch_dim=cfg.fusion_dim,
            fusion_dim=cfg.fusion_dim,
            active_indices=(0, 1, 2),
            dropout=cfg.dropout,
            adaptive=False,
        )

    return model


# ============================================================
# Forward smoke test for every ablation variant
# ============================================================

smoke_test_rows = []

probe_x = torch.randn(
    4,
    cicids_official_cfg.sequence_length,
    72,
)

for variant in ABLATION_VARIANTS:

    probe_model = build_ablation_model(
        input_dim=72,
        cfg=cicids_official_cfg,
        variant=variant,
    ).eval()

    with torch.no_grad():
        logits, auxiliary = probe_model(
            probe_x,
            return_aux=True,
        )

    assert logits.shape == (4, 2)
    assert auxiliary["embedding"].shape == (
        4,
        288,
    )
    assert auxiliary[
        "fusion_weights"
    ].shape == (4, 3)

    smoke_test_rows.append({
        "Variant": variant,
        "Parameters": sum(
            parameter.numel()
            for parameter in (
                probe_model.parameters()
            )
            if parameter.requires_grad
        ),
        "Logits": tuple(
            logits.shape
        ),
        "Embedding": tuple(
            auxiliary[
                "embedding"
            ].shape
        ),
        "Fusion weights": tuple(
            auxiliary[
                "fusion_weights"
            ].shape
        ),
        "Mean weight sum": float(
            auxiliary[
                "fusion_weights"
            ]
            .sum(dim=1)
            .mean()
        ),
    })

    del probe_model

display(
    pd.DataFrame(
        smoke_test_rows
    )
)

del probe_x

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "All KAMBA++ ablation "
    "forward tests passed."
)

,Variant,Parameters,Logits,Embedding,Fusion weights,Mean weight sum
0,full,376361,"(4, 2)","(4, 288)","(4, 3)",1.0
1,without_edyt,326825,"(4, 2)","(4, 288)","(4, 3)",1.0
2,without_skan,243975,"(4, 2)","(4, 288)","(4, 3)",1.0
3,without_swiglu,342184,"(4, 2)","(4, 288)","(4, 3)",1.0
4,without_dsssm,289576,"(4, 2)","(4, 288)","(4, 3)",1.0
5,without_controller,367910,"(4, 2)","(4, 288)","(4, 3)",1.0
6,equal_fusion,348326,"(4, 2)","(4, 288)","(4, 3)",1.0


All KAMBA++ ablation forward tests passed.


In [57]:
# ============================================================
# Ablation training functions
# ============================================================

from dataclasses import replace, asdict
from pathlib import Path
from typing import Dict, Tuple
import gc
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def train_ablation_seed(
    cfg: Config,
    seed: int,
    arrays: Dict[
        str,
        Tuple[np.ndarray, np.ndarray],
    ],
    device: torch.device,
    variant: str,
    experiment_root: Path,
):

    set_seed(
        seed,
        cfg.deterministic,
    )

    run_dir = (
        experiment_root
        / variant
        / f"seed_{seed}"
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    train_x, train_y = arrays["train"]
    val_x, val_y = arrays["val"]
    test_x, test_y = arrays["test"]

    train_loader = make_loader(
        train_x,
        train_y,
        cfg,
        True,
    )

    val_loader = make_loader(
        val_x,
        val_y,
        cfg,
        False,
    )

    test_loader = make_loader(
        test_x,
        test_y,
        cfg,
        False,
    )

    model = build_ablation_model(
        input_dim=train_x.shape[-1],
        cfg=cfg,
        variant=variant,
    ).to(device)

    class_counts = np.bincount(
        train_y,
        minlength=2,
    )

    class_weights = torch.tensor(
        len(train_y)
        / np.maximum(
            2 * class_counts,
            1,
        ),
        dtype=torch.float32,
        device=device,
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=cfg.epochs,
        )
    )

    amp_enabled = (
        cfg.use_amp
        and device.type == "cuda"
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled,
    )

    save_json(
        {
            "variant": variant,
            "seed": seed,
            "optimizer": "AdamW",
            "initial_learning_rate": (
                cfg.learning_rate
            ),
            "weight_decay": (
                cfg.weight_decay
            ),
            "sparse_regularizer": (
                0.0
                if variant == "without_skan"
                else cfg.sparse_regularizer
            ),
            "scheduler": (
                "CosineAnnealingLR"
            ),
            "scheduler_T_max": cfg.epochs,
            "gradient_clip": (
                cfg.gradient_clip
            ),
            "mixed_precision": (
                amp_enabled
            ),
        },
        run_dir
        / "optimizer_settings.json",
    )

    best_score = -np.inf
    best_epoch = 0
    stale_epochs = 0
    history = []

    for epoch in range(
        1,
        cfg.epochs + 1,
    ):

        model.train()

        running_loss = 0.0
        observed_samples = 0

        for batch_x, batch_y in train_loader:

            batch_x = batch_x.to(
                device,
                non_blocking=True,
            )

            batch_y = batch_y.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True,
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):

                logits, auxiliary = model(
                    batch_x,
                    return_aux=True,
                )

                classification_loss = (
                    criterion(
                        logits,
                        batch_y,
                    )
                )

                if variant == "without_skan":
                    sparsity_penalty = (
                        classification_loss
                        .new_zeros(())
                    )
                else:
                    sparsity_penalty = (
                        cfg.sparse_regularizer
                        * sparse_loss(auxiliary)
                    )

                loss = (
                    classification_loss
                    + sparsity_penalty
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.gradient_clip,
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            running_loss += (
                float(loss.detach())
                * len(batch_y)
            )

            observed_samples += len(batch_y)

        scheduler.step()

        validation_true, validation_probability, _ = (
            predict(
                model,
                val_loader,
                device,
            )
        )

        threshold = select_threshold(
            validation_true,
            validation_probability,
        )

        validation_metrics = compute_metrics(
            validation_true,
            validation_probability,
            threshold,
        )

        epoch_row = {
            "epoch": epoch,
            "train_loss": (
                running_loss
                / observed_samples
            ),
            "learning_rate": (
                optimizer
                .param_groups[0]["lr"]
            ),
            **validation_metrics,
        }

        history.append(
            epoch_row
        )

        if (
            validation_metrics["auprc"]
            > best_score + 1e-5
        ):
            best_score = (
                validation_metrics["auprc"]
            )

            best_epoch = epoch
            stale_epochs = 0

            torch.save(
                {
                    "model": (
                        model.state_dict()
                    ),
                    "config": asdict(cfg),
                    "variant": variant,
                },
                run_dir
                / "best_model.pt",
            )

        else:
            stale_epochs += 1

        if stale_epochs >= cfg.patience:
            break

    # --------------------------------------------------------
    # Restore validation-selected checkpoint
    # --------------------------------------------------------

    checkpoint = torch.load(
        run_dir / "best_model.pt",
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model"],
        strict=True,
    )

    validation_true, validation_probability, _ = (
        predict(
            model,
            val_loader,
            device,
        )
    )

    threshold = select_threshold(
        validation_true,
        validation_probability,
    )

    test_true, test_probability, test_auxiliary = (
        predict(
            model,
            test_loader,
            device,
        )
    )

    metrics = compute_metrics(
        test_true,
        test_probability,
        threshold,
    )

    if variant == "without_skan":
        mean_sparse_fraction = np.nan
    else:
        mean_sparse_fraction = float(
            test_auxiliary[
                "sparsity"
            ].mean()
        )

    metrics.update({
        "dataset": cfg.dataset_name,
        "variant": variant,
        "seed": seed,
        "best_epoch": best_epoch,
        "parameters": sum(
            parameter.numel()
            for parameter in (
                model.parameters()
            )
            if parameter.requires_grad
        ),
        "mean_sparse_fraction": (
            mean_sparse_fraction
        ),
        "mean_w_skan": float(
            test_auxiliary[
                "fusion"
            ][:, 0].mean()
        ),
        "mean_w_swiglu": float(
            test_auxiliary[
                "fusion"
            ][:, 1].mean()
        ),
        "mean_w_dsssm": float(
            test_auxiliary[
                "fusion"
            ][:, 2].mean()
        ),
    })

    pd.DataFrame(
        history
    ).to_csv(
        run_dir / "history.csv",
        index=False,
    )

    pd.DataFrame({
        "label": test_true,
        "prob_apt": test_probability,
    }).to_csv(
        run_dir
        / "test_predictions.csv",
        index=False,
    )

    save_json(
        metrics,
        run_dir
        / "test_metrics.json",
    )

    del model
    del optimizer
    del scheduler
    del scaler

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


def run_ablation_suite(
    cfg: Config,
    variants,
):

    experiment_root = Path(
        cfg.output_dir
    )

    experiment_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    save_json(
        {
            "configuration": asdict(cfg),
            "variants": list(variants),
            "checkpoint_metric": (
                "validation AUPRC"
            ),
            "threshold_selection": (
                "validation F1 maximization"
            ),
        },
        experiment_root
        / "ablation_configuration.json",
    )

    device = choose_device(
        cfg.device
    )

    print("Device:", device)
    print("Preparing:", cfg.dataset_name)

    prepared = prepare_tabular_data(
        cfg,
        seed=cfg.seeds[0],
    )

    arrays = {}

    for split_name, row_indices in (
        prepared["split"].items()
    ):

        sequence_x, sequence_y, _ = (
            make_windows(
                frame=prepared["frame"],
                x=prepared["x"],
                y=prepared["y"],
                groups=prepared["groups"],
                row_indices=row_indices,
                length=cfg.sequence_length,
                stride=cfg.stride,
            )
        )

        arrays[split_name] = (
            sequence_x,
            sequence_y,
        )

    rows = []

    for variant in variants:

        for seed in cfg.seeds:

            print(
                f"\nTraining {variant}, "
                f"seed {seed}"
            )

            result = train_ablation_seed(
                cfg=cfg,
                seed=seed,
                arrays=arrays,
                device=device,
                variant=variant,
                experiment_root=(
                    experiment_root
                ),
            )

            rows.append(
                result
            )

            print(result)

    seed_results = pd.DataFrame(
        rows
    )

    seed_results.to_csv(
        experiment_root
        / "ablation_seed_results.csv",
        index=False,
    )

    numeric_metrics = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "mcc",
        "auroc",
        "auprc",
        "parameters",
        "mean_sparse_fraction",
        "mean_w_skan",
        "mean_w_swiglu",
        "mean_w_dsssm",
    ]

    summary_rows = []

    for variant, variant_frame in (
        seed_results.groupby(
            "variant",
            sort=False,
        )
    ):

        for metric in numeric_metrics:

            values = (
                variant_frame[metric]
                .dropna()
            )

            if len(values) == 0:
                continue

            summary_rows.append({
                "dataset": (
                    cfg.dataset_name
                ),
                "variant": variant,
                "metric": metric,
                "mean": float(
                    values.mean()
                ),
                "std": float(
                    values.std(ddof=1)
                    if len(values) > 1
                    else 0.0
                ),
            })

    summary_frame = pd.DataFrame(
        summary_rows
    )

    summary_frame.to_csv(
        experiment_root
        / "ablation_mean_std.csv",
        index=False,
    )

    del prepared
    del arrays

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return seed_results, summary_frame


# ============================================================
# DAPT2020 three-epoch ablation pilot
# ============================================================

REDUCED_VARIANTS = (
    "without_edyt",
    "without_skan",
    "without_swiglu",
    "without_dsssm",
    "without_controller",
    "equal_fusion",
)

dapt_ablation_pilot_cfg = replace(
    dapt_cfg,
    output_dir=str(
        PROJECT_ROOT
        / "outputs"
        / "DAPT2020_ablation_pilot"
    ),
    epochs=3,
    patience=3,
    seeds=(13,),
)

print(
    "Pilot output:",
    dapt_ablation_pilot_cfg.output_dir,
)

dapt_ablation_pilot_results, (
    dapt_ablation_pilot_summary
) = run_ablation_suite(
    cfg=dapt_ablation_pilot_cfg,
    variants=REDUCED_VARIANTS,
)

display(
    dapt_ablation_pilot_results[
        [
            "variant",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auroc",
            "auprc",
            "parameters",
        ]
    ]
)

Pilot output: <PROJECT_ROOT>\outputs\DAPT2020_ablation_pilot
Device: cuda
Preparing: DAPT2020
Recovered canonical header for: enp0s3-pvt-thursday.pcap_Flow.csv
Weighted temporal-block split:


,Split,Blocks,Estimated benign sequences,Estimated attack sequences,Target benign sequences,Target attack sequences
0,train,28171,1032,7,1192,6
1,val,6036,335,1,255,1
2,test,6037,336,1,255,1


Train–validation block overlap: 0
Train–test block overlap: 0
Validation–test block overlap: 0

Training without_edyt, seed 13
{'threshold': 0.49322301149368286, 'accuracy': 0.9970326409495549, 'balanced_accuracy': 0.5, 'precision': 0.0, 'recall': 0.0, 'specificity': 1.0, 'f1': 0.0, 'mcc': 0.0, 'auroc': 0.0, 'auprc': 0.002967359050445104, 'tn': 336, 'fp': 0, 'fn': 1, 'tp': 0, 'dataset': 'DAPT2020', 'variant': 'without_edyt', 'seed': 13, 'best_epoch': 1, 'parameters': 326185, 'mean_sparse_fraction': 1.0, 'mean_w_skan': 0.32891619205474854, 'mean_w_swiglu': 0.35704177618026733, 'mean_w_dsssm': 0.31404203176498413}

Training without_skan, seed 13
{'threshold': 0.493582546710968, 'accuracy': 0.0, 'balanced_accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'specificity': 0.0, 'f1': 0.0, 'mcc': -1.0, 'auroc': 0.0, 'auprc': 0.002967359050445104, 'tn': 0, 'fp': 336, 'fn': 1, 'tp': 0, 'dataset': 'DAPT2020', 'variant': 'without_skan', 'seed': 13, 'best_epoch': 1, 'parameters': 243335, 'mean_spars

,variant,accuracy,precision,recall,f1,auroc,auprc,parameters
0,without_edyt,0.997033,0.0,0.0,0.0,0.000000,0.002967,326185
1,without_skan,0.000000,0.0,0.0,0.0,0.000000,0.002967,243335
2,without_swiglu,0.026706,0.0,0.0,0.0,0.000000,0.002967,341544
3,without_dsssm,0.700297,0.0,0.0,0.0,0.000000,0.002967,288936
4,without_controller,0.997033,0.0,0.0,0.0,0.000000,0.002967,367270
5,equal_fusion,0.997033,0.0,0.0,0.0,0.011905,0.003003,347686


In [58]:
# ============================================================
# Inspect saved official DAPT2020 preprocessing artifacts
# ============================================================

from pathlib import Path
import json
import joblib

dapt_official_dir = (
    PROJECT_ROOT
    / "outputs"
    / "DAPT2020"
)

dapt_prepared_dir = (
    dapt_official_dir
    / "prepared"
)

print(
    "Prepared directory exists:",
    dapt_prepared_dir.exists(),
)

if dapt_prepared_dir.exists():

    print("\nSaved preprocessing files:")

    for path in sorted(
        dapt_prepared_dir.rglob("*")
    ):
        if path.is_file():
            print(
                path.relative_to(
                    dapt_official_dir
                ),
                f"({path.stat().st_size / 1024:.1f} KB)",
            )

row_split_path = (
    dapt_prepared_dir
    / "row_splits.json"
)

if row_split_path.exists():

    with open(
        row_split_path,
        "r",
        encoding="utf-8",
    ) as file:
        saved_dapt_splits = json.load(file)

    print("\nRow-split object type:")
    print(type(saved_dapt_splits))

    if isinstance(
        saved_dapt_splits,
        dict,
    ):
        print(
            "Top-level keys:",
            saved_dapt_splits.keys(),
        )

        for key, value in (
            saved_dapt_splits.items()
        ):
            print(
                key,
                type(value),
                (
                    len(value)
                    if hasattr(
                        value,
                        "__len__",
                    )
                    else "no length"
                ),
            )

            if isinstance(
                value,
                list,
            ):
                print(
                    "First values:",
                    value[:5],
                )

schema_path = (
    dapt_prepared_dir
    / "schema.json"
)

if schema_path.exists():

    with open(
        schema_path,
        "r",
        encoding="utf-8",
    ) as file:
        saved_dapt_schema = json.load(file)

    print("\nSaved schema:")
    print(saved_dapt_schema)

scaler_path = (
    dapt_prepared_dir
    / "scaler.joblib"
)

if scaler_path.exists():

    saved_dapt_scaler = joblib.load(
        scaler_path
    )

    print("\nScaler class:")
    print(
        saved_dapt_scaler
        .__class__.__name__
    )

    print(
        "Scaler feature count:",
        getattr(
            saved_dapt_scaler,
            "n_features_in_",
            None,
        ),
    )

    print(
        "Scaler mean shape:",
        getattr(
            saved_dapt_scaler,
            "mean_",
            None,
        ).shape,
    )

Prepared directory exists: True

Saved preprocessing files:
prepared\row_splits.json (1005.1 KB)
prepared\scaler.joblib (2.2 KB)
prepared\schema.json (3.7 KB)

Row-split object type:
<class 'dict'>
Top-level keys: dict_keys(['train', 'val', 'test'])
train <class 'list'> 59263
First values: [0, 1, 5, 6, 11]
val <class 'list'> 11179
First values: [22, 41, 44, 53, 88]
test <class 'list'> 16249
First values: [2, 3, 4, 7, 8]

Saved schema:
{'feature_columns': ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IA

In [62]:
# ============================================================
# Final corrected DAPT2020 window reconstruction
# Short groups: pad one window
# Long groups: complete windows only; discard trailing remainder
# ============================================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch


def make_dapt_official_windows(
    x,
    y,
    groups,
    timestamps,
    row_indices,
    length=8,
):

    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    split_metadata = pd.DataFrame({
        "row_index": row_indices,
        "group": groups[row_indices],
        "timestamp": (
            timestamps.iloc[
                row_indices
            ].to_numpy()
        ),
    })

    split_metadata = (
        split_metadata.sort_values(
            [
                "group",
                "timestamp",
                "row_index",
            ],
            kind="stable",
        )
    )

    sequence_features = []
    sequence_labels = []
    sequence_groups = []

    for group_name, group_frame in (
        split_metadata.groupby(
            "group",
            sort=False,
        )
    ):

        ordered_indices = (
            group_frame[
                "row_index"
            ]
            .to_numpy(dtype=np.int64)
        )

        group_size = len(
            ordered_indices
        )

        if group_size == 0:
            continue

        # ----------------------------------------------------
        # Short Flow-ID group:
        # retain one window and pad it to length 8
        # ----------------------------------------------------

        if group_size < length:

            window_x = x[
                ordered_indices
            ]

            window_y = y[
                ordered_indices
            ]

            padding_x = np.zeros(
                (
                    length - group_size,
                    x.shape[1],
                ),
                dtype=np.float32,
            )

            padded_window_x = np.concatenate(
                [
                    window_x,
                    padding_x,
                ],
                axis=0,
            )

            sequence_features.append(
                padded_window_x
            )

            sequence_labels.append(
                int(window_y.max())
            )

            sequence_groups.append(
                str(group_name)
            )

            continue

        # ----------------------------------------------------
        # Flow-ID group with at least 8 records:
        # retain complete windows only
        # ----------------------------------------------------

        number_of_complete_windows = (
            group_size // length
        )

        usable_length = (
            number_of_complete_windows
            * length
        )

        usable_indices = (
            ordered_indices[
                :usable_length
            ]
        )

        for start in range(
            0,
            usable_length,
            length,
        ):

            window_indices = (
                usable_indices[
                    start:
                    start + length
                ]
            )

            window_x = x[
                window_indices
            ]

            window_y = y[
                window_indices
            ]

            sequence_features.append(
                window_x
            )

            sequence_labels.append(
                int(window_y.max())
            )

            sequence_groups.append(
                str(group_name)
            )

    return (
        np.stack(
            sequence_features
        ).astype(np.float32),
        np.asarray(
            sequence_labels,
            dtype=np.int64,
        ),
        np.asarray(
            sequence_groups,
            dtype=object,
        ),
    )


# ------------------------------------------------------------
# Reconstruct official partitions
# ------------------------------------------------------------

dapt_ablation_arrays = {}
dapt_ablation_groups = {}
dapt_restored_summary = {}

for split_name in [
    "train",
    "val",
    "test",
]:

    split_x, split_y, split_groups = (
        make_dapt_official_windows(
            x=dapt_x,
            y=dapt_y,
            groups=dapt_groups,
            timestamps=dapt_timestamp,
            row_indices=(
                official_dapt_splits[
                    split_name
                ]
            ),
            length=8,
        )
    )

    dapt_ablation_arrays[
        split_name
    ] = (
        split_x,
        split_y,
    )

    dapt_ablation_groups[
        split_name
    ] = split_groups

    dapt_restored_summary[
        split_name
    ] = {
        "Sequences": len(split_y),
        "Benign": int(
            (split_y == 0).sum()
        ),
        "Attack": int(
            (split_y == 1).sum()
        ),
        "Unique Flow IDs": len(
            set(split_groups)
        ),
        "Shape": split_x.shape,
    }

print("Restored official DAPT2020 partitions:")

display(
    pd.DataFrame(
        dapt_restored_summary
    ).T
)


# ------------------------------------------------------------
# Exact official verification
# ------------------------------------------------------------

expected_dapt_summary = {
    "train": {
        "Sequences": 29_147,
        "Benign": 20_251,
        "Attack": 8_896,
        "Unique Flow IDs": 28_171,
    },
    "val": {
        "Sequences": 6_067,
        "Benign": 4_162,
        "Attack": 1_905,
        "Unique Flow IDs": 6_036,
    },
    "test": {
        "Sequences": 6_694,
        "Benign": 4_788,
        "Attack": 1_906,
        "Unique Flow IDs": 6_037,
    },
}

for split_name in [
    "train",
    "val",
    "test",
]:

    for key in [
        "Sequences",
        "Benign",
        "Attack",
        "Unique Flow IDs",
    ]:

        observed = (
            dapt_restored_summary[
                split_name
            ][key]
        )

        expected = (
            expected_dapt_summary[
                split_name
            ][key]
        )

        assert observed == expected, (
            f"{split_name} {key}: "
            f"expected {expected:,}, "
            f"found {observed:,}"
        )


# ------------------------------------------------------------
# Verify group disjointness
# ------------------------------------------------------------

train_flow_ids = set(
    dapt_ablation_groups["train"]
)

validation_flow_ids = set(
    dapt_ablation_groups["val"]
)

test_flow_ids = set(
    dapt_ablation_groups["test"]
)

print(
    "Train-validation overlap:",
    len(
        train_flow_ids
        & validation_flow_ids
    ),
)

print(
    "Train-test overlap:",
    len(
        train_flow_ids
        & test_flow_ids
    ),
)

print(
    "Validation-test overlap:",
    len(
        validation_flow_ids
        & test_flow_ids
    ),
)

assert not (
    train_flow_ids
    & validation_flow_ids
)

assert not (
    train_flow_ids
    & test_flow_ids
)

assert not (
    validation_flow_ids
    & test_flow_ids
)

print(
    "\nOfficial DAPT2020 data "
    "reconstruction passed."
)


# ============================================================
# Rerun valid three-epoch ablation pilot
# ============================================================

valid_pilot_root = (
    PROJECT_ROOT
    / "outputs"
    / "DAPT2020_ablation_pilot_valid"
)

valid_pilot_root.mkdir(
    parents=True,
    exist_ok=True,
)

dapt_ablation_pilot_cfg = replace(
    dapt_cfg,
    output_dir=str(
        valid_pilot_root
    ),
    epochs=3,
    patience=3,
    seeds=(13,),
)

pilot_device = choose_device(
    dapt_ablation_pilot_cfg.device
)

valid_pilot_rows = []

for variant in REDUCED_VARIANTS:

    print(
        f"\nTraining valid pilot: "
        f"{variant}, seed 13"
    )

    result = train_ablation_seed(
        cfg=dapt_ablation_pilot_cfg,
        seed=13,
        arrays=dapt_ablation_arrays,
        device=pilot_device,
        variant=variant,
        experiment_root=(
            valid_pilot_root
        ),
    )

    valid_pilot_rows.append(
        result
    )

    print(result)

valid_dapt_pilot_results = pd.DataFrame(
    valid_pilot_rows
)

valid_dapt_pilot_results.to_csv(
    valid_pilot_root
    / "valid_pilot_results.csv",
    index=False,
)

print("\nValid DAPT2020 pilot results:")

display(
    valid_dapt_pilot_results[
        [
            "variant",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "parameters",
        ]
    ]
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "\nValid DAPT2020 ablation "
    "pilot completed."
)

Restored official DAPT2020 partitions:


,Sequences,Benign,Attack,Unique Flow IDs,Shape
train,29147,20251,8896,28171,"(29147, 8, 67)"
val,6067,4162,1905,6036,"(6067, 8, 67)"
test,6694,4788,1906,6037,"(6694, 8, 67)"


Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0

Official DAPT2020 data reconstruction passed.

Training valid pilot: without_edyt, seed 13
{'threshold': 0.6376358866691589, 'accuracy': 0.9490588586794144, 'balanced_accuracy': 0.9527054125344842, 'precision': 0.8727965697951405, 'recall': 0.9611752360965372, 'specificity': 0.9442355889724311, 'f1': 0.914856429463171, 'mcc': 0.8807146579445155, 'auroc': 0.9834138511721766, 'auprc': 0.948048204576105, 'tn': 4521, 'fp': 267, 'fn': 74, 'tp': 1832, 'dataset': 'DAPT2020', 'variant': 'without_edyt', 'seed': 13, 'best_epoch': 3, 'parameters': 326185, 'mean_sparse_fraction': 0.1371895968914032, 'mean_w_skan': 0.6512681245803833, 'mean_w_swiglu': 0.03636862337589264, 'mean_w_dsssm': 0.31236323714256287}

Training valid pilot: without_skan, seed 13
{'threshold': 0.7317681312561035, 'accuracy': 0.9448760083657006, 'balanced_accuracy': 0.9417284466850933, 'precision': 0.8795061728395062, 'recall': 0.934417628541448, 'sp

,variant,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,auroc,auprc,parameters
0,without_edyt,0.949059,0.952705,0.872797,0.961175,0.944236,0.914856,0.880715,0.983414,0.948048,326185
1,without_skan,0.944876,0.941728,0.879506,0.934418,0.949039,0.906131,0.867959,0.983863,0.945934,243335
2,without_swiglu,0.950254,0.947856,0.889549,0.942288,0.953425,0.915159,0.880741,0.985435,0.951700,341544
3,without_dsssm,0.911413,0.927337,0.777825,0.964323,0.890351,0.861092,0.807074,0.931314,0.692353,288936
4,without_controller,0.948611,0.953655,0.868744,0.965373,0.941938,0.914513,0.880417,0.982200,0.942371,367270
5,equal_fusion,0.910666,0.915920,0.793274,0.928122,0.903718,0.855416,0.796459,0.936128,0.737511,347686



Valid DAPT2020 ablation pilot completed.


In [63]:
# ============================================================
# Official DAPT2020 ablation study: six variants x five seeds
# ============================================================

from dataclasses import replace
from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# Official configuration and output directory
# ------------------------------------------------------------

dapt_ablation_root = (
    PROJECT_ROOT
    / "outputs"
    / "DAPT2020_ablation"
)

dapt_ablation_root.mkdir(
    parents=True,
    exist_ok=True,
)

dapt_ablation_cfg = replace(
    dapt_cfg,
    output_dir=str(
        dapt_ablation_root
    ),
    epochs=60,
    patience=10,
    seeds=(13, 27, 41, 55, 69),
)

save_json(
    {
        "dataset": "DAPT2020",
        "variants": list(
            REDUCED_VARIANTS
        ),
        "seeds": list(
            dapt_ablation_cfg.seeds
        ),
        "maximum_epochs": 60,
        "early_stopping_patience": 10,
        "checkpoint_metric": (
            "validation AUPRC"
        ),
        "threshold_selection": (
            "validation F1 maximization"
        ),
        "data_protocol": (
            "Official saved Flow-ID-disjoint "
            "partitions and padded short groups"
        ),
    },
    dapt_ablation_root
    / "ablation_configuration.json",
)

device = choose_device(
    dapt_ablation_cfg.device
)

print("Device:", device)
print("Output:", dapt_ablation_root)
print("Variants:", REDUCED_VARIANTS)
print("Seeds:", dapt_ablation_cfg.seeds)


# ------------------------------------------------------------
# Reconfirm official arrays before training
# ------------------------------------------------------------

assert (
    dapt_ablation_arrays["train"][0].shape
    == (29_147, 8, 67)
)

assert (
    dapt_ablation_arrays["val"][0].shape
    == (6_067, 8, 67)
)

assert (
    dapt_ablation_arrays["test"][0].shape
    == (6_694, 8, 67)
)

assert int(
    (
        dapt_ablation_arrays[
            "train"
        ][1] == 1
    ).sum()
) == 8_896

assert int(
    (
        dapt_ablation_arrays[
            "val"
        ][1] == 1
    ).sum()
) == 1_905

assert int(
    (
        dapt_ablation_arrays[
            "test"
        ][1] == 1
    ).sum()
) == 1_906

print(
    "Official DAPT2020 arrays verified."
)


# ------------------------------------------------------------
# Train or resume every reduced variant
# ------------------------------------------------------------

official_ablation_rows = []

for variant in REDUCED_VARIANTS:

    for seed in (
        dapt_ablation_cfg.seeds
    ):

        metrics_path = (
            dapt_ablation_root
            / variant
            / f"seed_{seed}"
            / "test_metrics.json"
        )

        if metrics_path.exists():

            with open(
                metrics_path,
                "r",
                encoding="utf-8",
            ) as file:
                result = json.load(file)

            print(
                f"Loaded completed run: "
                f"{variant}, seed {seed}"
            )

        else:

            print(
                f"\nTraining official ablation: "
                f"{variant}, seed {seed}"
            )

            result = train_ablation_seed(
                cfg=dapt_ablation_cfg,
                seed=seed,
                arrays=dapt_ablation_arrays,
                device=device,
                variant=variant,
                experiment_root=(
                    dapt_ablation_root
                ),
            )

            print(result)

        official_ablation_rows.append(
            result
        )

        # Save progress after every seed
        pd.DataFrame(
            official_ablation_rows
        ).to_csv(
            dapt_ablation_root
            / "ablation_seed_results_partial.csv",
            index=False,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ------------------------------------------------------------
# Reduced-variant seed results
# ------------------------------------------------------------

reduced_seed_results = pd.DataFrame(
    official_ablation_rows
)

reduced_seed_results.to_csv(
    dapt_ablation_root
    / "reduced_variant_seed_results.csv",
    index=False,
)


# ------------------------------------------------------------
# Add completed full KAMBA++ results without retraining
# ------------------------------------------------------------

full_seed_results = pd.read_csv(
    PROJECT_ROOT
    / "outputs"
    / "DAPT2020"
    / "all_seed_results.csv"
)

full_seed_results[
    "dataset"
] = "DAPT2020"

full_seed_results[
    "variant"
] = "full"

combined_seed_results = pd.concat(
    [
        full_seed_results,
        reduced_seed_results,
    ],
    ignore_index=True,
    sort=False,
)

variant_order = [
    "without_edyt",
    "without_skan",
    "without_swiglu",
    "without_dsssm",
    "without_controller",
    "equal_fusion",
    "full",
]

combined_seed_results[
    "variant"
] = pd.Categorical(
    combined_seed_results["variant"],
    categories=variant_order,
    ordered=True,
)

combined_seed_results = (
    combined_seed_results.sort_values(
        ["variant", "seed"]
    )
    .reset_index(drop=True)
)

combined_seed_results.to_csv(
    dapt_ablation_root
    / "all_ablation_seed_results.csv",
    index=False,
)


# ------------------------------------------------------------
# Mean and sample standard deviation
# ------------------------------------------------------------

reported_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "mcc",
    "auroc",
    "auprc",
    "parameters",
]

summary_rows = []

for variant in variant_order:

    variant_frame = (
        combined_seed_results[
            combined_seed_results[
                "variant"
            ].astype(str) == variant
        ]
    )

    for metric in reported_metrics:

        values = (
            variant_frame[metric]
            .dropna()
            .astype(float)
        )

        summary_rows.append({
            "dataset": "DAPT2020",
            "variant": variant,
            "metric": metric,
            "mean": float(
                values.mean()
            ),
            "std": float(
                values.std(ddof=1)
            ),
            "minimum": float(
                values.min()
            ),
            "maximum": float(
                values.max()
            ),
        })

ablation_mean_std = pd.DataFrame(
    summary_rows
)

ablation_mean_std.to_csv(
    dapt_ablation_root
    / "ablation_mean_std.csv",
    index=False,
)


# ------------------------------------------------------------
# LaTeX-table-ready wide summary
# ------------------------------------------------------------

mean_table = (
    ablation_mean_std.pivot(
        index="variant",
        columns="metric",
        values="mean",
    )
    .reindex(variant_order)
)

std_table = (
    ablation_mean_std.pivot(
        index="variant",
        columns="metric",
        values="std",
    )
    .reindex(variant_order)
)

latex_ready_rows = []

for variant in variant_order:

    row = {
        "variant": variant,
    }

    for metric in reported_metrics:

        mean_value = mean_table.loc[
            variant,
            metric,
        ]

        std_value = std_table.loc[
            variant,
            metric,
        ]

        if metric == "parameters":
            row[metric] = int(
                round(mean_value)
            )
        else:
            row[metric] = (
                f"{mean_value:.4f}"
                f" ± "
                f"{std_value:.4f}"
            )

    latex_ready_rows.append(row)

latex_ready_table = pd.DataFrame(
    latex_ready_rows
)

latex_ready_table.to_csv(
    dapt_ablation_root
    / "ablation_latex_ready.csv",
    index=False,
)


# ------------------------------------------------------------
# Final display
# ------------------------------------------------------------

print(
    "\nOfficial DAPT2020 ablation "
    "mean results:"
)

display(
    mean_table[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "parameters",
        ]
    ]
)

print(
    "\nLaTeX-ready mean ± std:"
)

display(
    latex_ready_table[
        [
            "variant",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "parameters",
        ]
    ]
)

print("\nSaved files:")

for filename in [
    "all_ablation_seed_results.csv",
    "reduced_variant_seed_results.csv",
    "ablation_mean_std.csv",
    "ablation_latex_ready.csv",
]:

    print(
        dapt_ablation_root
        / filename
    )

print(
    "\nOfficial DAPT2020 "
    "ablation study completed."
)

Device: cuda
Output: <PROJECT_ROOT>\outputs\DAPT2020_ablation
Variants: ('without_edyt', 'without_skan', 'without_swiglu', 'without_dsssm', 'without_controller', 'equal_fusion')
Seeds: (13, 27, 41, 55, 69)
Official DAPT2020 arrays verified.

Training official ablation: without_edyt, seed 13
{'threshold': 0.6748600006103516, 'accuracy': 0.9654914849118613, 'balanced_accuracy': 0.9701927299886652, 'precision': 0.9055690072639225, 'recall': 0.9811122770199371, 'specificity': 0.9592731829573935, 'f1': 0.9418282548476454, 'mcc': 0.9188419317127227, 'auroc': 0.9942732399379001, 'auprc': 0.9852364834020959, 'tn': 4593, 'fp': 195, 'fn': 36, 'tp': 1870, 'dataset': 'DAPT2020', 'variant': 'without_edyt', 'seed': 13, 'best_epoch': 18, 'parameters': 326185, 'mean_sparse_fraction': 0.5019879341125488, 'mean_w_skan': 0.6312557458877563, 'mean_w_swiglu': 0.09324683994054794, 'mean_w_dsssm': 0.2754974365234375}

Training official ablation: without_edyt, seed 27
{'threshold': 0.5829812288284302, 'accura

metric,accuracy,precision,recall,f1,mcc,auroc,auprc,parameters
variant,,,,,,,,
without_edyt,0.961518,0.895015,0.981637,0.935954,0.910921,0.990994,0.971252,326185.0
without_skan,0.962324,0.900907,0.975761,0.936641,0.911572,0.992785,0.980898,243335.0
without_swiglu,0.940633,0.837393,0.983526,0.904402,0.867839,0.987914,0.968243,341544.0
without_dsssm,0.965043,0.904614,0.980693,0.941108,0.917832,0.993540,0.982528,288936.0
without_controller,0.941291,0.845763,0.972613,0.904404,0.867136,0.978381,0.916300,367270.0
equal_fusion,0.967195,0.911454,0.980063,0.944505,0.922495,0.994171,0.984372,347686.0
full,0.978877,0.950589,0.977020,0.963515,0.948940,0.997911,0.995048,375721.0



LaTeX-ready mean ± std:


,variant,accuracy,precision,recall,f1,mcc,auroc,auprc,parameters
0,without_edyt,0.9615 ± 0.0127,0.8950 ± 0.0369,0.9816 ± 0.0055,0.9360 ± 0.0193,0.9109 ± 0.0264,0.9910 ± 0.0096,0.9713 ± 0.0345,326185
1,without_skan,0.9623 ± 0.0087,0.9009 ± 0.0260,0.9758 ± 0.0110,0.9366 ± 0.0138,0.9116 ± 0.0194,0.9928 ± 0.0031,0.9809 ± 0.0091,243335
2,without_swiglu,0.9406 ± 0.0120,0.8374 ± 0.0289,0.9835 ± 0.0062,0.9044 ± 0.0179,0.8678 ± 0.0245,0.9879 ± 0.0047,0.9682 ± 0.0165,341544
3,without_dsssm,0.9650 ± 0.0065,0.9046 ± 0.0121,0.9807 ± 0.0103,0.9411 ± 0.0108,0.9178 ± 0.0153,0.9935 ± 0.0028,0.9825 ± 0.0085,288936
4,without_controller,0.9413 ± 0.0139,0.8458 ± 0.0337,0.9726 ± 0.0207,0.9044 ± 0.0214,0.8671 ± 0.0299,0.9784 ± 0.0102,0.9163 ± 0.0742,367270
5,equal_fusion,0.9672 ± 0.0058,0.9115 ± 0.0118,0.9801 ± 0.0077,0.9445 ± 0.0097,0.9225 ± 0.0137,0.9942 ± 0.0025,0.9844 ± 0.0080,347686
6,full,0.9789 ± 0.0061,0.9506 ± 0.0207,0.9770 ± 0.0041,0.9635 ± 0.0102,0.9489 ± 0.0142,0.9979 ± 0.0005,0.9950 ± 0.0015,375721



Saved files:
<PROJECT_ROOT>\outputs\DAPT2020_ablation\all_ablation_seed_results.csv
<PROJECT_ROOT>\outputs\DAPT2020_ablation\reduced_variant_seed_results.csv
<PROJECT_ROOT>\outputs\DAPT2020_ablation\ablation_mean_std.csv
<PROJECT_ROOT>\outputs\DAPT2020_ablation\ablation_latex_ready.csv

Official DAPT2020 ablation study completed.


In [67]:
# ============================================================
# Numerical-stability recovery:
# CICIDS2017 without EDyT, seed 41
# ============================================================

from dataclasses import replace
import pandas as pd
import torch
import gc

# Disable mixed precision only for this numerically unstable run
cicids_fp32_cfg = replace(
    cicids_ablation_cfg,
    use_amp=False,
)

print("Recovery variant: without_edyt")
print("Recovery seed: 41")
print("Mixed precision:", cicids_fp32_cfg.use_amp)
print("Device:", device)

recovered_metrics = train_ablation_seed(
    cfg=cicids_fp32_cfg,
    seed=41,
    arrays=cicids_ablation_arrays,
    device=device,
    variant="without_edyt",
    experiment_root=cicids_ablation_output,
)

print("\nRecovered seed-41 result:")
print(recovered_metrics)

assert np.isfinite(
    recovered_metrics["accuracy"]
)

assert np.isfinite(
    recovered_metrics["f1"]
)

assert np.isfinite(
    recovered_metrics["auprc"]
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "\nSeed 41 recovery completed. "
    "Rerun the continuation cell to resume at seed 55."
)

Recovery variant: without_edyt
Recovery seed: 41
Mixed precision: False
Device: cuda

Recovered seed-41 result:
{'threshold': 0.9240581393241882, 'accuracy': 0.9674489440173214, 'balanced_accuracy': 0.9321140984785413, 'precision': 0.9906073317526901, 'recall': 0.86668262326472, 'specificity': 0.9975455736923626, 'f1': 0.9245106382978724, 'mcc': 0.9071591210334449, 'auroc': 0.9980382348454954, 'auprc': 0.9942043101329359, 'tn': 41862, 'fp': 103, 'fn': 1671, 'tp': 10863, 'dataset': 'CICIDS2017', 'variant': 'without_edyt', 'seed': 41, 'best_epoch': 28, 'parameters': 326825, 'mean_sparse_fraction': 0.9322795867919922, 'mean_w_skan': 0.18836960196495056, 'mean_w_swiglu': 0.3652347922325134, 'mean_w_dsssm': 0.4463955760002136}

Seed 41 recovery completed. Rerun the continuation cell to resume at seed 55.


In [68]:
seed41_metrics_path = (
    cicids_ablation_output
    / "without_edyt"
    / "seed_41"
    / "test_metrics.json"
)

with open(
    seed41_metrics_path,
    "r",
    encoding="utf-8",
) as file:
    seed41_metrics = json.load(file)

seed41_metrics[
    "numerical_precision"
] = "FP32 fallback"

with open(
    seed41_metrics_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        seed41_metrics,
        file,
        indent=2,
    )

print("Seed-41 FP32 fallback recorded.")

Seed-41 FP32 fallback recorded.


In [69]:
# ============================================================
# Continue official CICIDS2017 ablation training
# Resume support + FP32 numerical-stability fallback
# ============================================================

from dataclasses import replace
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import json
import gc
import inspect

print(
    "train_ablation_seed signature:",
    inspect.signature(train_ablation_seed),
)

# ------------------------------------------------------------
# Verify that official arrays remain available
# ------------------------------------------------------------

assert "cicids_ablation_arrays" in globals(), (
    "CICIDS2017 arrays are unavailable. "
    "Rerun the preprocessing section first."
)

expected_shapes = {
    "train": (243_814, 8, 72),
    "val": (54_458, 8, 72),
    "test": (54_499, 8, 72),
}

for split_name, expected_shape in (
    expected_shapes.items()
):
    observed_shape = (
        cicids_ablation_arrays[
            split_name
        ][0].shape
    )

    assert observed_shape == expected_shape, (
        f"{split_name}: expected "
        f"{expected_shape}, found "
        f"{observed_shape}"
    )

print(
    "Official CICIDS2017 arrays "
    "remain verified."
)

# ------------------------------------------------------------
# Helper: determine whether an exception was caused by
# non-finite numerical values
# ------------------------------------------------------------

def is_nonfinite_training_error(
    exception: Exception,
) -> bool:

    message = str(exception).lower()

    numerical_terms = (
        "nan",
        "infinity",
        "infinite",
        "non-finite",
        "not finite",
    )

    return any(
        term in message
        for term in numerical_terms
    )

# ------------------------------------------------------------
# Train six reduced variants with resume and FP32 fallback
# ------------------------------------------------------------

cicids_reduced_rows = []

for variant_name in ablation_variants:

    for seed in official_seeds:

        run_directory = (
            cicids_ablation_output
            / variant_name
            / f"seed_{seed}"
        )

        metrics_path = (
            run_directory
            / "test_metrics.json"
        )

        # ----------------------------------------------------
        # Load completed run
        # ----------------------------------------------------

        if metrics_path.exists():

            with open(
                metrics_path,
                "r",
                encoding="utf-8",
            ) as file:
                metrics = json.load(file)

            cicids_reduced_rows.append(
                metrics
            )

            print(
                "Loaded completed run:",
                variant_name,
                seed,
            )

            continue

        print(
            "\nTraining official CICIDS2017 "
            f"ablation: {variant_name}, "
            f"seed {seed}"
        )

        # ----------------------------------------------------
        # First attempt: official AMP configuration
        # ----------------------------------------------------

        try:

            metrics = train_ablation_seed(
                cfg=cicids_ablation_cfg,
                seed=seed,
                arrays=cicids_ablation_arrays,
                device=device,
                variant=variant_name,
                experiment_root=(
                    cicids_ablation_output
                ),
            )

            metrics[
                "numerical_precision"
            ] = (
                "AMP"
                if cicids_ablation_cfg.use_amp
                else "FP32"
            )

        except (ValueError, RuntimeError) as error:

            # Do not conceal unrelated errors
            if not is_nonfinite_training_error(
                error
            ):
                raise

            print(
                "\nNon-finite values detected for "
                f"{variant_name}, seed {seed}."
            )

            print(
                "Retrying the same run in FP32..."
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()

            fp32_cfg = replace(
                cicids_ablation_cfg,
                use_amp=False,
            )

            # ------------------------------------------------
            # Second attempt: identical protocol in FP32
            # ------------------------------------------------

            metrics = train_ablation_seed(
                cfg=fp32_cfg,
                seed=seed,
                arrays=cicids_ablation_arrays,
                device=device,
                variant=variant_name,
                experiment_root=(
                    cicids_ablation_output
                ),
            )

            metrics[
                "numerical_precision"
            ] = "FP32 fallback"

        # ----------------------------------------------------
        # Verify returned metrics
        # ----------------------------------------------------

        required_finite_metrics = [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "mcc",
            "auroc",
            "auprc",
        ]

        for metric_name in (
            required_finite_metrics
        ):
            assert np.isfinite(
                metrics[metric_name]
            ), (
                f"Non-finite {metric_name} "
                f"for {variant_name}, "
                f"seed {seed}"
            )

        # Save precision information in test_metrics.json
        run_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        with open(
            metrics_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                metrics,
                file,
                indent=2,
            )

        cicids_reduced_rows.append(
            metrics
        )

        print(metrics)

        # Save progress after every completed run
        pd.DataFrame(
            cicids_reduced_rows
        ).to_csv(
            cicids_ablation_output
            / (
                "reduced_variant_"
                "seed_results_partial.csv"
            ),
            index=False,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# Completed reduced-model results
# ------------------------------------------------------------

cicids_reduced_results_df = (
    pd.DataFrame(
        cicids_reduced_rows
    )
)

print(
    "\nAll reduced CICIDS2017 "
    "ablation runs completed."
)

display(
    cicids_reduced_results_df
)

if (
    "numerical_precision"
    in cicids_reduced_results_df.columns
):
    print(
        "\nNumerical-precision summary:"
    )

    display(
        cicids_reduced_results_df[
            [
                "variant",
                "seed",
                "numerical_precision",
            ]
        ]
    )

train_ablation_seed signature: (cfg: 'Config', seed: 'int', arrays: 'Dict[str, Tuple[np.ndarray, np.ndarray]]', device: 'torch.device', variant: 'str', experiment_root: 'Path')
Official CICIDS2017 arrays remain verified.
Loaded completed run: without_edyt 13
Loaded completed run: without_edyt 27
Loaded completed run: without_edyt 41

Training official CICIDS2017 ablation: without_edyt, seed 55
{'threshold': 0.9912298917770386, 'accuracy': 0.8999614671828841, 'balanced_accuracy': 0.782651452555186, 'precision': 0.9992949802594473, 'recall': 0.5654220520185097, 'specificity': 0.9998808530918623, 'f1': 0.7222052379496586, 'mcc': 0.7070727107932494, 'auroc': 0.9949296412506939, 'auprc': 0.9836442567750119, 'tn': 41960, 'fp': 5, 'fn': 5447, 'tp': 7087, 'dataset': 'CICIDS2017', 'variant': 'without_edyt', 'seed': 55, 'best_epoch': 3, 'parameters': 326825, 'mean_sparse_fraction': 0.6425122022628784, 'mean_w_skan': 0.4116361141204834, 'mean_w_swiglu': 0.16672787070274353, 'mean_w_dsssm': 0.4216

,threshold,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,auroc,auprc,...,dataset,variant,seed,best_epoch,parameters,mean_sparse_fraction,mean_w_skan,mean_w_swiglu,mean_w_dsssm,numerical_precision
0,0.943907,0.923723,0.837360,0.986752,0.677437,0.997283,0.803349,0.778680,0.994619,0.980979,...,CICIDS2017,without_edyt,13,5,326825,0.724063,0.315325,0.428735,0.255940,NaN
1,0.979574,0.909760,0.805352,0.992881,0.612015,0.998689,0.757256,0.736792,0.994242,0.980616,...,CICIDS2017,without_edyt,27,8,326825,0.862045,0.298084,0.242818,0.459098,NaN
2,0.924058,0.967449,0.932114,0.990607,0.866683,0.997546,0.924511,0.907159,0.998038,0.994204,...,CICIDS2017,without_edyt,41,28,326825,0.932280,0.188370,0.365235,0.446396,FP32 fallback
3,0.991230,0.899961,0.782651,0.999295,0.565422,0.999881,0.722205,0.707073,0.994930,0.983644,...,CICIDS2017,without_edyt,55,3,326825,0.642512,0.411636,0.166728,0.421636,AMP
4,0.955077,0.929136,0.848793,0.988508,0.700016,0.997569,0.819617,0.795033,0.996000,0.986832,...,CICIDS2017,without_edyt,69,17,326825,0.932195,0.406676,0.352298,0.241026,AMP
5,0.962354,0.953797,0.901036,0.994764,0.803335,0.998737,0.888859,0.867940,0.998330,0.995052,...,CICIDS2017,without_skan,13,28,243975,NaN,0.000000,0.744064,0.255936,AMP
6,0.967471,0.972165,0.940939,0.995324,0.883118,0.998761,0.935870,0.920854,0.999003,0.997333,...,CICIDS2017,without_skan,27,22,243975,NaN,0.000000,0.709403,0.290597,AMP
7,0.967119,0.974036,0.945624,0.993432,0.893011,0.998237,0.940549,0.926147,0.998684,0.996475,...,CICIDS2017,without_skan,41,29,243975,NaN,0.000000,0.461618,0.538382,AMP
8,0.977937,0.969632,0.935434,0.995265,0.872108,0.998761,0.929625,0.913600,0.998711,0.996351,...,CICIDS2017,without_skan,55,26,243975,NaN,0.000000,0.624927,0.375073,AMP
9,0.991591,0.969963,0.935845,0.996266,0.872666,0.999023,0.930379,0.914586,0.998853,0.997048,...,CICIDS2017,without_skan,69,23,243975,NaN,0.000000,0.795653,0.204347,AMP



Numerical-precision summary:


,variant,seed,numerical_precision
0,without_edyt,13,NaN
1,without_edyt,27,NaN
2,without_edyt,41,FP32 fallback
3,without_edyt,55,AMP
4,without_edyt,69,AMP
5,without_skan,13,AMP
6,without_skan,27,AMP
7,without_skan,41,AMP
8,without_skan,55,AMP
9,without_skan,69,AMP


In [70]:
# Correct missing precision metadata for previously completed runs
for seed in (13, 27):

    metrics_path = (
        cicids_ablation_output
        / "without_edyt"
        / f"seed_{seed}"
        / "test_metrics.json"
    )

    with open(
        metrics_path,
        "r",
        encoding="utf-8",
    ) as file:
        metrics = json.load(file)

    metrics["numerical_precision"] = "AMP"

    with open(
        metrics_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            indent=2,
        )

print("Precision metadata corrected.")

Precision metadata corrected.


In [71]:
# ============================================================
# Finalize official CICIDS2017 ablation results
# No model training is performed in this cell
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Experiment definitions
# ------------------------------------------------------------

reduced_variants = [
    "without_edyt",
    "without_skan",
    "without_swiglu",
    "without_dsssm",
    "without_controller",
    "equal_fusion",
]

variant_order = [
    *reduced_variants,
    "full",
]

official_seeds = (
    13,
    27,
    41,
    55,
    69,
)

reported_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "mcc",
    "auroc",
    "auprc",
    "parameters",
]

# ------------------------------------------------------------
# 2. Correct missing precision metadata
# ------------------------------------------------------------

for seed in (13, 27):

    metrics_path = (
        cicids_ablation_output
        / "without_edyt"
        / f"seed_{seed}"
        / "test_metrics.json"
    )

    assert metrics_path.exists(), (
        f"Missing metrics file: {metrics_path}"
    )

    with open(
        metrics_path,
        "r",
        encoding="utf-8",
    ) as file:
        metrics = json.load(file)

    metrics["numerical_precision"] = "AMP"

    with open(
        metrics_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            indent=2,
        )

print("Precision metadata corrected.")

# ------------------------------------------------------------
# 3. Reload all 30 reduced-model runs from disk
# ------------------------------------------------------------

cicids_reduced_rows = []

for variant_name in reduced_variants:

    for seed in official_seeds:

        metrics_path = (
            cicids_ablation_output
            / variant_name
            / f"seed_{seed}"
            / "test_metrics.json"
        )

        assert metrics_path.exists(), (
            f"Missing completed run: "
            f"{variant_name}, seed {seed}"
        )

        with open(
            metrics_path,
            "r",
            encoding="utf-8",
        ) as file:
            metrics = json.load(file)

        metrics["dataset"] = "CICIDS2017"
        metrics["variant"] = variant_name
        metrics["seed"] = int(seed)

        # Older completed runs may not contain this metadata
        if "numerical_precision" not in metrics:

            optimizer_path = (
                metrics_path.parent
                / "optimizer_settings.json"
            )

            if optimizer_path.exists():

                with open(
                    optimizer_path,
                    "r",
                    encoding="utf-8",
                ) as file:
                    optimizer_settings = json.load(file)

                mixed_precision = (
                    optimizer_settings.get(
                        "mixed_precision",
                        True,
                    )
                )

                metrics[
                    "numerical_precision"
                ] = (
                    "AMP"
                    if mixed_precision
                    else "FP32"
                )

            else:
                metrics[
                    "numerical_precision"
                ] = "AMP"

        cicids_reduced_rows.append(
            metrics
        )

cicids_reduced_frame = pd.DataFrame(
    cicids_reduced_rows
)

assert len(cicids_reduced_frame) == 30, (
    "Expected 30 reduced-model runs, "
    f"found {len(cicids_reduced_frame)}."
)

assert (
    cicids_reduced_frame[
        ["variant", "seed"]
    ]
    .duplicated()
    .sum()
    == 0
), "Duplicate reduced-model runs detected."

for metric_name in reported_metrics:

    assert np.isfinite(
        pd.to_numeric(
            cicids_reduced_frame[
                metric_name
            ],
            errors="coerce",
        )
    ).all(), (
        f"Non-finite values detected in "
        f"{metric_name}."
    )

print(
    "Reduced-model runs loaded:",
    len(cicids_reduced_frame),
)

# ------------------------------------------------------------
# 4. Load five official full KAMBA++ runs
# ------------------------------------------------------------

cicids_full_results_path = (
    PROJECT_ROOT
    / "outputs"
    / "CICIDS2017"
    / "all_seed_results.csv"
)

assert cicids_full_results_path.exists(), (
    "Official CICIDS2017 full-model "
    "results were not found."
)

cicids_full_rows = pd.read_csv(
    cicids_full_results_path
)

assert len(cicids_full_rows) == 5, (
    "Expected five full-model runs, "
    f"found {len(cicids_full_rows)}."
)

assert set(
    cicids_full_rows[
        "seed"
    ].astype(int)
) == set(official_seeds), (
    "The full-model seed set is incorrect."
)

cicids_full_rows["dataset"] = (
    "CICIDS2017"
)

cicids_full_rows["variant"] = "full"

cicids_full_rows[
    "numerical_precision"
] = "AMP"

# ------------------------------------------------------------
# 5. Combine reduced and full results
# ------------------------------------------------------------

cicids_all_ablation_results = pd.concat(
    [
        cicids_reduced_frame,
        cicids_full_rows,
    ],
    ignore_index=True,
    sort=False,
)

assert len(
    cicids_all_ablation_results
) == 35, (
    "Expected 35 total results, "
    f"found "
    f"{len(cicids_all_ablation_results)}."
)

cicids_all_ablation_results[
    "variant"
] = pd.Categorical(
    cicids_all_ablation_results[
        "variant"
    ],
    categories=variant_order,
    ordered=True,
)

cicids_all_ablation_results = (
    cicids_all_ablation_results
    .sort_values(
        by=[
            "variant",
            "seed",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Calculate mean and standard deviation
# ------------------------------------------------------------

cicids_ablation_mean_std = (
    cicids_all_ablation_results
    .groupby(
        "variant",
        observed=False,
    )[reported_metrics]
    .agg([
        "mean",
        "std",
    ])
)

cicids_ablation_mean = (
    cicids_all_ablation_results
    .groupby(
        "variant",
        observed=False,
    )[reported_metrics]
    .mean()
)

# ------------------------------------------------------------
# 7. Create LaTeX-ready mean ± standard deviation
# ------------------------------------------------------------

latex_rows = []

for variant_name in variant_order:

    current_rows = (
        cicids_all_ablation_results[
            cicids_all_ablation_results[
                "variant"
            ]
            == variant_name
        ]
    )

    assert len(current_rows) == 5, (
        f"{variant_name}: expected five "
        f"runs, found {len(current_rows)}."
    )

    latex_row = {
        "variant": variant_name
    }

    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "mcc",
        "auroc",
        "auprc",
    ]:

        mean_value = current_rows[
            metric_name
        ].mean()

        std_value = current_rows[
            metric_name
        ].std(ddof=1)

        latex_row[metric_name] = (
            f"{mean_value:.4f} "
            f"± {std_value:.4f}"
        )

    latex_row["parameters"] = int(
        round(
            current_rows[
                "parameters"
            ].mean()
        )
    )

    latex_rows.append(
        latex_row
    )

cicids_latex_ready = pd.DataFrame(
    latex_rows
)

# ------------------------------------------------------------
# 8. Calculate changes relative to full KAMBA++
# ------------------------------------------------------------

full_mean = (
    cicids_ablation_mean.loc["full"]
)

comparison_rows = []

for variant_name in reduced_variants:

    variant_mean = (
        cicids_ablation_mean.loc[
            variant_name
        ]
    )

    comparison_rows.append({
        "variant": variant_name,
        "accuracy_change_pp": (
            100.0
            * (
                variant_mean["accuracy"]
                - full_mean["accuracy"]
            )
        ),
        "f1_change_pp": (
            100.0
            * (
                variant_mean["f1"]
                - full_mean["f1"]
            )
        ),
        "mcc_change_pp": (
            100.0
            * (
                variant_mean["mcc"]
                - full_mean["mcc"]
            )
        ),
        "auprc_change_pp": (
            100.0
            * (
                variant_mean["auprc"]
                - full_mean["auprc"]
            )
        ),
    })

cicids_ablation_changes = pd.DataFrame(
    comparison_rows
)

# ------------------------------------------------------------
# 9. Save final official files
# ------------------------------------------------------------

cicids_reduced_frame.to_csv(
    cicids_ablation_output
    / "reduced_variant_seed_results.csv",
    index=False,
)

cicids_all_ablation_results.to_csv(
    cicids_ablation_output
    / "all_ablation_seed_results.csv",
    index=False,
)

cicids_ablation_mean_std.to_csv(
    cicids_ablation_output
    / "ablation_mean_std.csv"
)

cicids_latex_ready.to_csv(
    cicids_ablation_output
    / "ablation_latex_ready.csv",
    index=False,
)

cicids_ablation_changes.to_csv(
    cicids_ablation_output
    / "ablation_changes_from_full.csv",
    index=False,
)

# ------------------------------------------------------------
# 10. Display final results
# ------------------------------------------------------------

print(
    "\nOfficial CICIDS2017 "
    "ablation mean results:"
)

display(
    cicids_ablation_mean[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "parameters",
        ]
    ]
)

print(
    "\nLaTeX-ready mean ± std:"
)

display(
    cicids_latex_ready[
        [
            "variant",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "parameters",
        ]
    ]
)

print(
    "\nChanges relative to full "
    "KAMBA++ (percentage points):"
)

display(
    cicids_ablation_changes
)

print(
    "\nNumerical-precision record:"
)

display(
    cicids_reduced_frame[
        [
            "variant",
            "seed",
            "numerical_precision",
        ]
    ]
)

print("\nSaved files:")

for file_name in [
    "all_ablation_seed_results.csv",
    "reduced_variant_seed_results.csv",
    "ablation_mean_std.csv",
    "ablation_latex_ready.csv",
    "ablation_changes_from_full.csv",
]:
    print(
        cicids_ablation_output
        / file_name
    )

print(
    "\nOfficial CICIDS2017 "
    "ablation study finalized."
)

Precision metadata corrected.
Reduced-model runs loaded: 30

Official CICIDS2017 ablation mean results:


,accuracy,precision,recall,f1,mcc,auroc,auprc,parameters
variant,,,,,,,,
without_edyt,0.926006,0.991609,0.684315,0.805388,0.784947,0.995566,0.985255,326825.0
without_skan,0.967919,0.995010,0.864848,0.925056,0.908625,0.998716,0.996452,243975.0
without_swiglu,0.964220,0.994769,0.848891,0.915624,0.897971,0.998418,0.995602,342184.0
without_dsssm,0.961086,0.995221,0.834801,0.907730,0.888991,0.998202,0.995608,289576.0
without_controller,0.967999,0.995056,0.865167,0.925487,0.908900,0.998488,0.996092,367910.0
equal_fusion,0.959357,0.995886,0.826695,0.903333,0.884065,0.998580,0.995934,348326.0
full,0.963900,0.994708,0.847535,0.915003,0.897072,0.998405,0.995467,376361.0



LaTeX-ready mean ± std:


,variant,accuracy,precision,recall,f1,mcc,auroc,auprc,parameters
0,without_edyt,0.9260 ± 0.0259,0.9916 ± 0.0049,0.6843 ± 0.1150,0.8054 ± 0.0769,0.7849 ± 0.0766,0.9956 ± 0.0015,0.9853 ± 0.0056,326825
1,without_skan,0.9679 ± 0.0081,0.9950 ± 0.0010,0.8648 ± 0.0354,0.9251 ± 0.0207,0.9086 ± 0.0233,0.9987 ± 0.0003,0.9965 ± 0.0009,243975
2,without_swiglu,0.9642 ± 0.0095,0.9948 ± 0.0012,0.8489 ± 0.0417,0.9156 ± 0.0239,0.8980 ± 0.0273,0.9984 ± 0.0002,0.9956 ± 0.0006,342184
3,without_dsssm,0.9611 ± 0.0072,0.9952 ± 0.0011,0.8348 ± 0.0314,0.9077 ± 0.0185,0.8890 ± 0.0208,0.9982 ± 0.0006,0.9956 ± 0.0007,289576
4,without_controller,0.9680 ± 0.0041,0.9951 ± 0.0007,0.8652 ± 0.0185,0.9255 ± 0.0103,0.9089 ± 0.0118,0.9985 ± 0.0003,0.9961 ± 0.0002,367910
5,equal_fusion,0.9594 ± 0.0046,0.9959 ± 0.0014,0.8267 ± 0.0200,0.9033 ± 0.0121,0.8841 ± 0.0133,0.9986 ± 0.0002,0.9959 ± 0.0006,348326
6,full,0.9639 ± 0.0072,0.9947 ± 0.0015,0.8475 ± 0.0310,0.9150 ± 0.0184,0.8971 ± 0.0207,0.9984 ± 0.0004,0.9955 ± 0.0010,376361



Changes relative to full KAMBA++ (percentage points):


,variant,accuracy_change_pp,f1_change_pp,mcc_change_pp,auprc_change_pp
0,without_edyt,-3.789427,-10.961585,-11.212501,-1.021226
1,without_skan,0.401842,1.005304,1.155293,0.098454
2,without_swiglu,0.031927,0.062102,0.089818,0.013487
3,without_dsssm,-0.281473,-0.727353,-0.808173,0.014095
4,without_controller,0.409916,1.048380,1.182734,0.062428
5,equal_fusion,-0.454320,-1.167065,-1.300760,0.046668



Numerical-precision record:


,variant,seed,numerical_precision
0,without_edyt,13,AMP
1,without_edyt,27,AMP
2,without_edyt,41,FP32 fallback
3,without_edyt,55,AMP
4,without_edyt,69,AMP
5,without_skan,13,AMP
6,without_skan,27,AMP
7,without_skan,41,AMP
8,without_skan,55,AMP
9,without_skan,69,AMP



Saved files:
<PROJECT_ROOT>\outputs\CICIDS2017_ablation\all_ablation_seed_results.csv
<PROJECT_ROOT>\outputs\CICIDS2017_ablation\reduced_variant_seed_results.csv
<PROJECT_ROOT>\outputs\CICIDS2017_ablation\ablation_mean_std.csv
<PROJECT_ROOT>\outputs\CICIDS2017_ablation\ablation_latex_ready.csv
<PROJECT_ROOT>\outputs\CICIDS2017_ablation\ablation_changes_from_full.csv

Official CICIDS2017 ablation study finalized.


In [72]:
# ============================================================
# Reconstruct training runtime for full and ablation runs
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

runtime_rows = []

def collect_runtime(
    dataset,
    variant,
    seed,
    run_directory,
):
    run_directory = Path(run_directory)

    optimizer_path = (
        run_directory
        / "optimizer_settings.json"
    )

    history_path = (
        run_directory
        / "history.csv"
    )

    metrics_path = (
        run_directory
        / "test_metrics.json"
    )

    if not (
        optimizer_path.exists()
        and history_path.exists()
    ):
        print(
            "Missing runtime files:",
            run_directory,
        )
        return

    start_time = (
        optimizer_path.stat().st_mtime
    )

    end_time = (
        history_path.stat().st_mtime
    )

    elapsed_seconds = (
        end_time - start_time
    )

    if elapsed_seconds <= 0:
        print(
            "Invalid timestamp order:",
            run_directory,
        )
        return

    history = pd.read_csv(
        history_path
    )

    epochs_completed = len(history)

    best_epoch = np.nan

    if metrics_path.exists():
        with open(
            metrics_path,
            "r",
            encoding="utf-8",
        ) as file:
            metrics = json.load(file)

        best_epoch = metrics.get(
            "best_epoch",
            np.nan,
        )

    runtime_rows.append({
        "dataset": dataset,
        "variant": variant,
        "seed": int(seed),
        "epochs_completed": (
            epochs_completed
        ),
        "best_epoch": best_epoch,
        "runtime_seconds": (
            elapsed_seconds
        ),
        "runtime_minutes": (
            elapsed_seconds / 60.0
        ),
        "runtime_hours": (
            elapsed_seconds / 3600.0
        ),
        "seconds_per_epoch": (
            elapsed_seconds
            / max(epochs_completed, 1)
        ),
    })

# ------------------------------------------------------------
# DAPT2020 full model
# ------------------------------------------------------------

for seed in (13, 27, 41, 55, 69):

    collect_runtime(
        dataset="DAPT2020",
        variant="full",
        seed=seed,
        run_directory=(
            PROJECT_ROOT
            / "outputs"
            / "DAPT2020"
            / f"seed_{seed}"
        ),
    )

# ------------------------------------------------------------
# DAPT2020 ablation variants
# ------------------------------------------------------------

for variant in [
    "without_edyt",
    "without_skan",
    "without_swiglu",
    "without_dsssm",
    "without_controller",
    "equal_fusion",
]:

    for seed in (
        13,
        27,
        41,
        55,
        69,
    ):

        collect_runtime(
            dataset="DAPT2020",
            variant=variant,
            seed=seed,
            run_directory=(
                PROJECT_ROOT
                / "outputs"
                / "DAPT2020_ablation"
                / variant
                / f"seed_{seed}"
            ),
        )

# ------------------------------------------------------------
# CICIDS2017 full model
# ------------------------------------------------------------

for seed in (13, 27, 41, 55, 69):

    collect_runtime(
        dataset="CICIDS2017",
        variant="full",
        seed=seed,
        run_directory=(
            PROJECT_ROOT
            / "outputs"
            / "CICIDS2017"
            / f"seed_{seed}"
        ),
    )

# ------------------------------------------------------------
# CICIDS2017 ablation variants
# ------------------------------------------------------------

for variant in [
    "without_edyt",
    "without_skan",
    "without_swiglu",
    "without_dsssm",
    "without_controller",
    "equal_fusion",
]:

    for seed in (
        13,
        27,
        41,
        55,
        69,
    ):

        collect_runtime(
            dataset="CICIDS2017",
            variant=variant,
            seed=seed,
            run_directory=(
                PROJECT_ROOT
                / "outputs"
                / "CICIDS2017_ablation"
                / variant
                / f"seed_{seed}"
            ),
        )

# ------------------------------------------------------------
# Create runtime tables
# ------------------------------------------------------------

runtime_results = pd.DataFrame(
    runtime_rows
)

runtime_summary = (
    runtime_results
    .groupby(
        [
            "dataset",
            "variant",
        ]
    )
    .agg(
        runs=("seed", "count"),
        mean_epochs=(
            "epochs_completed",
            "mean",
        ),
        mean_minutes=(
            "runtime_minutes",
            "mean",
        ),
        std_minutes=(
            "runtime_minutes",
            "std",
        ),
        total_hours=(
            "runtime_hours",
            "sum",
        ),
        mean_seconds_per_epoch=(
            "seconds_per_epoch",
            "mean",
        ),
    )
    .reset_index()
)

suite_summary = (
    runtime_results
    .groupby("dataset")
    .agg(
        total_runs=("seed", "count"),
        total_hours=(
            "runtime_hours",
            "sum",
        ),
        mean_minutes_per_run=(
            "runtime_minutes",
            "mean",
        ),
    )
    .reset_index()
)

print("Per-run reconstructed runtime:")

display(
    runtime_results
)

print(
    "\nMean runtime by dataset "
    "and variant:"
)

display(
    runtime_summary
)

print(
    "\nTotal experimental runtime:"
)

display(
    suite_summary
)

# ------------------------------------------------------------
# Save runtime results
# ------------------------------------------------------------

runtime_output = (
    PROJECT_ROOT
    / "outputs"
    / "runtime_summary"
)

runtime_output.mkdir(
    parents=True,
    exist_ok=True,
)

runtime_results.to_csv(
    runtime_output
    / "all_training_runtimes.csv",
    index=False,
)

runtime_summary.to_csv(
    runtime_output
    / "runtime_by_variant.csv",
    index=False,
)

suite_summary.to_csv(
    runtime_output
    / "total_runtime_by_dataset.csv",
    index=False,
)

print(
    "\nRuntime results saved to:",
    runtime_output,
)

Per-run reconstructed runtime:


,dataset,variant,seed,epochs_completed,best_epoch,runtime_seconds,runtime_minutes,runtime_hours,seconds_per_epoch
0,DAPT2020,full,13,60,50,238.103465,3.968391,0.066140,3.968391
1,DAPT2020,full,27,36,26,127.597442,2.126624,0.035444,3.544373
2,DAPT2020,full,41,51,41,176.793650,2.946561,0.049109,3.466542
3,DAPT2020,full,55,48,38,167.148432,2.785807,0.046430,3.482259
4,DAPT2020,full,69,44,34,155.803851,2.596731,0.043279,3.540997
...,...,...,...,...,...,...,...,...,...
65,CICIDS2017,equal_fusion,13,33,23,979.559559,16.325993,0.272100,29.683623
66,CICIDS2017,equal_fusion,27,34,24,1012.353485,16.872558,0.281209,29.775103
67,CICIDS2017,equal_fusion,41,35,25,1044.047791,17.400797,0.290013,29.829937
68,CICIDS2017,equal_fusion,55,35,25,1039.048066,17.317468,0.288624,29.687088



Mean runtime by dataset and variant:


,dataset,variant,runs,mean_epochs,mean_minutes,std_minutes,total_hours,mean_seconds_per_epoch
0,CICIDS2017,equal_fusion,5,34.6,17.153107,0.577726,1.429426,29.744724
1,CICIDS2017,full,5,31.6,16.544736,2.181322,1.378728,31.413071
2,CICIDS2017,without_controller,5,32.8,16.049246,1.593637,1.337437,29.358997
3,CICIDS2017,without_dsssm,5,33.6,10.740542,2.119564,0.895045,19.175296
4,CICIDS2017,without_edyt,5,22.2,10.527313,4.938198,0.877276,28.431468
5,CICIDS2017,without_skan,5,35.6,11.142988,0.936346,0.928582,18.783081
6,CICIDS2017,without_swiglu,5,36.8,18.488854,0.667589,1.540738,30.161663
7,DAPT2020,equal_fusion,5,43.4,3.020759,1.678285,0.251730,4.066685
8,DAPT2020,full,5,47.8,2.884823,0.679264,0.240402,3.600512
9,DAPT2020,without_controller,5,52.0,5.171435,1.738032,0.430953,5.928312



Total experimental runtime:


,dataset,total_runs,total_hours,mean_minutes_per_run
0,CICIDS2017,35,8.387232,14.378113
1,DAPT2020,35,1.646085,2.821860



Runtime results saved to: <PROJECT_ROOT>\outputs\runtime_summary


In [74]:
# ============================================================
# Recover the exact official DAPT2020 arrays
# from saved splits, schema, medians, and scaler
# ============================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

print("Recovering official DAPT2020 arrays...")

# ------------------------------------------------------------
# 1. Locate official saved preprocessing artifacts
# ------------------------------------------------------------

dapt_official_output = Path(
    dapt_cfg.output_dir
)

dapt_prepared_directory = (
    dapt_official_output
    / "prepared"
)

dapt_split_path = (
    dapt_prepared_directory
    / "row_splits.json"
)

dapt_schema_path = (
    dapt_prepared_directory
    / "schema.json"
)

dapt_scaler_path = (
    dapt_prepared_directory
    / "scaler.joblib"
)

for required_path in [
    dapt_split_path,
    dapt_schema_path,
    dapt_scaler_path,
]:
    assert required_path.exists(), (
        f"Required artifact not found: "
        f"{required_path}"
    )

# ------------------------------------------------------------
# 2. Load saved official preprocessing information
# ------------------------------------------------------------

with open(
    dapt_split_path,
    "r",
    encoding="utf-8",
) as file:
    dapt_saved_splits = json.load(file)

with open(
    dapt_schema_path,
    "r",
    encoding="utf-8",
) as file:
    dapt_saved_schema = json.load(file)

dapt_saved_scaler = joblib.load(
    dapt_scaler_path
)

dapt_feature_columns = (
    dapt_saved_schema[
        "feature_columns"
    ]
)

dapt_saved_medians = (
    dapt_saved_schema[
        "medians"
    ]
)

assert len(dapt_feature_columns) == 67

print(
    "Saved feature dimensions:",
    len(dapt_feature_columns)
)

print(
    "Saved row counts:",
    {
        split_name: len(indices)
        for split_name, indices
        in dapt_saved_splits.items()
    },
)

# ------------------------------------------------------------
# 3. Reload DAPT2020 in the original stable file order
# ------------------------------------------------------------

dapt_restored_frame = (
    read_csv_collection(
        dapt_cfg.data_path,
        dapt_cfg.label_column,
    )
    .reset_index(drop=True)
)

dapt_restored_frame.columns = (
    dapt_restored_frame.columns
    .astype(str)
    .str.strip()
    .str.lstrip("\ufeff")
)

assert len(dapt_restored_frame) == 86_691, (
    "Expected 86,691 DAPT2020 records, "
    f"found {len(dapt_restored_frame):,}"
)

required_dapt_columns = [
    "Flow ID",
    "Timestamp",
    "Stage",
    *dapt_feature_columns,
]

missing_dapt_columns = [
    column
    for column in required_dapt_columns
    if column not in dapt_restored_frame.columns
]

assert not missing_dapt_columns, (
    "Missing DAPT2020 columns: "
    f"{missing_dapt_columns}"
)

# ------------------------------------------------------------
# 4. Reconstruct official binary labels
# ------------------------------------------------------------

normalized_stage = (
    dapt_restored_frame["Stage"]
    .astype("string")
    .str.strip()
)

dapt_restored_y = (
    ~normalized_stage.isin(
        dapt_cfg.benign_values
    )
).astype(np.int64).to_numpy()

print(
    "Record labels:",
    {
        "benign": int(
            (dapt_restored_y == 0).sum()
        ),
        "attack": int(
            (dapt_restored_y == 1).sum()
        ),
    },
)

# ------------------------------------------------------------
# 5. Restore numerical feature matrix
# ------------------------------------------------------------

dapt_numeric_frame = pd.DataFrame(
    index=dapt_restored_frame.index
)

for feature_name in dapt_feature_columns:

    numeric_values = pd.to_numeric(
        dapt_restored_frame[
            feature_name
        ],
        errors="coerce",
    )

    numeric_values = numeric_values.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    numeric_values = numeric_values.fillna(
        float(
            dapt_saved_medians[
                feature_name
            ]
        )
    )

    dapt_numeric_frame[
        feature_name
    ] = numeric_values

assert np.isfinite(
    dapt_numeric_frame.to_numpy()
).all()

dapt_restored_x = (
    dapt_saved_scaler
    .transform(
        dapt_numeric_frame[
            dapt_feature_columns
        ]
    )
    .astype(np.float32)
)

assert dapt_restored_x.shape == (
    86_691,
    67,
)

assert np.isfinite(
    dapt_restored_x
).all()

# ------------------------------------------------------------
# 6. Restore grouping and chronological order
# ------------------------------------------------------------

dapt_restored_groups = (
    dapt_restored_frame[
        "Flow ID"
    ]
    .astype("string")
    .fillna("__missing_flow_id__")
    .to_numpy()
)

dapt_restored_timestamp = pd.to_datetime(
    dapt_restored_frame[
        "Timestamp"
    ],
    errors="coerce",
    dayfirst=True,
)

assert (
    dapt_restored_timestamp
    .isna()
    .sum()
    == 0
)

# ------------------------------------------------------------
# 7. Exact official DAPT2020 window function
#
# Groups smaller than 8:
#     pad and retain one sequence
#
# Groups of size >= 8:
#     retain complete windows only
#     discard the trailing incomplete remainder
# ------------------------------------------------------------

def restore_official_dapt_windows(
    row_indices,
    sequence_length=8,
):

    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    split_metadata = pd.DataFrame({
        "row_index": row_indices,
        "group": (
            dapt_restored_groups[
                row_indices
            ]
        ),
        "timestamp": (
            dapt_restored_timestamp
            .iloc[row_indices]
            .to_numpy()
        ),
    })

    split_metadata = (
        split_metadata
        .sort_values(
            by=[
                "group",
                "timestamp",
                "row_index",
            ],
            kind="stable",
        )
    )

    sequence_rows = []
    sequence_labels = []
    sequence_groups = []

    for group_name, group_frame in (
        split_metadata.groupby(
            "group",
            sort=False,
        )
    ):

        group_rows = (
            group_frame[
                "row_index"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        group_size = len(group_rows)

        # ----------------------------------------
        # Short group: one padded sequence
        # ----------------------------------------

        if group_size < sequence_length:

            current_x = (
                dapt_restored_x[
                    group_rows
                ]
            )

            padding_count = (
                sequence_length
                - group_size
            )

            # Repeat the final observation
            # to preserve dimensional validity
            padding = np.repeat(
                current_x[-1:],
                padding_count,
                axis=0,
            )

            current_x = np.concatenate(
                [
                    current_x,
                    padding,
                ],
                axis=0,
            )

            current_y = int(
                dapt_restored_y[
                    group_rows
                ].max()
            )

            sequence_rows.append(
                current_x
            )

            sequence_labels.append(
                current_y
            )

            sequence_groups.append(
                str(group_name)
            )

            continue

        # ----------------------------------------
        # Larger group: complete windows only
        # ----------------------------------------

        usable_count = (
            group_size
            // sequence_length
        ) * sequence_length

        usable_rows = group_rows[
            :usable_count
        ]

        for start_index in range(
            0,
            usable_count,
            sequence_length,
        ):

            current_rows = (
                usable_rows[
                    start_index:
                    start_index
                    + sequence_length
                ]
            )

            sequence_rows.append(
                dapt_restored_x[
                    current_rows
                ]
            )

            sequence_labels.append(
                int(
                    dapt_restored_y[
                        current_rows
                    ].max()
                )
            )

            sequence_groups.append(
                str(group_name)
            )

    return (
        np.stack(
            sequence_rows
        ).astype(np.float32),
        np.asarray(
            sequence_labels,
            dtype=np.int64,
        ),
        np.asarray(
            sequence_groups,
            dtype=object,
        ),
    )

# ------------------------------------------------------------
# 8. Reconstruct all three official partitions
# ------------------------------------------------------------

baseline_dapt_arrays = {}
baseline_dapt_groups = {}
dapt_recovery_summary = {}

for split_name in [
    "train",
    "val",
    "test",
]:

    sequence_x, sequence_y, sequence_groups = (
        restore_official_dapt_windows(
            dapt_saved_splits[
                split_name
            ],
            sequence_length=8,
        )
    )

    baseline_dapt_arrays[
        split_name
    ] = (
        sequence_x,
        sequence_y,
    )

    baseline_dapt_groups[
        split_name
    ] = sequence_groups

    dapt_recovery_summary[
        split_name
    ] = {
        "Sequences": len(sequence_y),
        "Benign": int(
            (sequence_y == 0).sum()
        ),
        "Attack": int(
            (sequence_y == 1).sum()
        ),
        "Unique Flow IDs": len(
            set(sequence_groups)
        ),
        "Shape": sequence_x.shape,
    }

print(
    "\nRestored official "
    "DAPT2020 partitions:"
)

display(
    pd.DataFrame(
        dapt_recovery_summary
    ).T
)

# ------------------------------------------------------------
# 9. Exact verification
# ------------------------------------------------------------

expected_dapt_summary = {
    "train": {
        "Sequences": 29_147,
        "Benign": 20_251,
        "Attack": 8_896,
        "Unique Flow IDs": 28_171,
        "Shape": (29_147, 8, 67),
    },
    "val": {
        "Sequences": 6_067,
        "Benign": 4_162,
        "Attack": 1_905,
        "Unique Flow IDs": 6_036,
        "Shape": (6_067, 8, 67),
    },
    "test": {
        "Sequences": 6_694,
        "Benign": 4_788,
        "Attack": 1_906,
        "Unique Flow IDs": 6_037,
        "Shape": (6_694, 8, 67),
    },
}

for split_name, expected_values in (
    expected_dapt_summary.items()
):

    for key, expected_value in (
        expected_values.items()
    ):

        observed_value = (
            dapt_recovery_summary[
                split_name
            ][key]
        )

        assert observed_value == expected_value, (
            f"{split_name} {key}: "
            f"expected {expected_value}, "
            f"found {observed_value}"
        )

# Verify group separation
train_flow_ids = set(
    baseline_dapt_groups["train"]
)

val_flow_ids = set(
    baseline_dapt_groups["val"]
)

test_flow_ids = set(
    baseline_dapt_groups["test"]
)

print(
    "\nTrain-validation overlap:",
    len(train_flow_ids & val_flow_ids)
)

print(
    "Train-test overlap:",
    len(train_flow_ids & test_flow_ids)
)

print(
    "Validation-test overlap:",
    len(val_flow_ids & test_flow_ids)
)

assert not (
    train_flow_ids
    & val_flow_ids
)

assert not (
    train_flow_ids
    & test_flow_ids
)

assert not (
    val_flow_ids
    & test_flow_ids
)

print(
    "\nOfficial DAPT2020 arrays "
    "recovered successfully."
)

Recovering official DAPT2020 arrays...
Saved feature dimensions: 67
Saved row counts: {'train': 59263, 'val': 11179, 'test': 16249}
Recovered canonical header for: enp0s3-pvt-thursday.pcap_Flow.csv
Record labels: {'benign': 63712, 'attack': 22979}


<CONDA_ROOT>\envs\hlcda\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
<TEMP_DIR>\ipykernel_12076\2853931829.py:242: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dapt_restored_timestamp = pd.to_datetime(



Restored official DAPT2020 partitions:


,Sequences,Benign,Attack,Unique Flow IDs,Shape
train,29147,20251,8896,28171,"(29147, 8, 67)"
val,6067,4162,1905,6036,"(6067, 8, 67)"
test,6694,4788,1906,6037,"(6694, 8, 67)"



Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0

Official DAPT2020 arrays recovered successfully.


In [75]:
# ============================================================
# Baseline Experiment — Cell 1B
# Final array verification and component-interface audit
# ============================================================

import inspect
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Locate official DAPT2020 arrays
# ------------------------------------------------------------

assert "baseline_dapt_arrays" in globals(), (
    "Recovered DAPT2020 arrays are unavailable."
)

# ------------------------------------------------------------
# 2. Locate official CICIDS2017 arrays
# ------------------------------------------------------------

if "baseline_cicids_arrays" in globals():

    pass

elif "cicids_ablation_arrays" in globals():

    baseline_cicids_arrays = (
        cicids_ablation_arrays
    )

elif "cicids_arrays" in globals():

    baseline_cicids_arrays = (
        cicids_arrays
    )

else:

    raise NameError(
        "Official CICIDS2017 arrays are "
        "unavailable. Rerun the official "
        "CICIDS2017 preprocessing cell."
    )

# ------------------------------------------------------------
# 3. Exact expected shapes and class distributions
# ------------------------------------------------------------

expected_baseline_summary = {
    "DAPT2020": {
        "train": {
            "shape": (29_147, 8, 67),
            "benign": 20_251,
            "attack": 8_896,
        },
        "val": {
            "shape": (6_067, 8, 67),
            "benign": 4_162,
            "attack": 1_905,
        },
        "test": {
            "shape": (6_694, 8, 67),
            "benign": 4_788,
            "attack": 1_906,
        },
    },
    "CICIDS2017": {
        "train": {
            "shape": (243_814, 8, 72),
            "benign": 195_449,
            "attack": 48_365,
        },
        "val": {
            "shape": (54_458, 8, 72),
            "benign": 41_966,
            "attack": 12_492,
        },
        "test": {
            "shape": (54_499, 8, 72),
            "benign": 41_965,
            "attack": 12_534,
        },
    },
}

baseline_datasets = {
    "DAPT2020": baseline_dapt_arrays,
    "CICIDS2017": baseline_cicids_arrays,
}

baseline_verification_rows = []

for dataset_name, arrays in (
    baseline_datasets.items()
):

    for split_name in [
        "train",
        "val",
        "test",
    ]:

        sequence_x, sequence_y = (
            arrays[split_name]
        )

        expected = (
            expected_baseline_summary[
                dataset_name
            ][split_name]
        )

        assert (
            sequence_x.shape
            == expected["shape"]
        ), (
            f"{dataset_name} {split_name}: "
            f"expected {expected['shape']}, "
            f"found {sequence_x.shape}"
        )

        observed_benign = int(
            (sequence_y == 0).sum()
        )

        observed_attack = int(
            (sequence_y == 1).sum()
        )

        assert (
            observed_benign
            == expected["benign"]
        )

        assert (
            observed_attack
            == expected["attack"]
        )

        assert np.isfinite(
            sequence_x
        ).all(), (
            f"Non-finite input values in "
            f"{dataset_name} {split_name}."
        )

        assert np.isfinite(
            sequence_y
        ).all()

        baseline_verification_rows.append({
            "Dataset": dataset_name,
            "Split": split_name,
            "Sequences": len(sequence_y),
            "Benign": observed_benign,
            "Attack": observed_attack,
            "Shape": sequence_x.shape,
            "Input dtype": str(
                sequence_x.dtype
            ),
        })

print("Official baseline arrays:")

display(
    pd.DataFrame(
        baseline_verification_rows
    )
)

# ------------------------------------------------------------
# 4. Verify reusable training functions
# ------------------------------------------------------------

required_baseline_functions = [
    "make_loader",
    "select_threshold",
    "compute_metrics",
    "set_seed",
]

function_rows = []

for function_name in (
    required_baseline_functions
):

    assert function_name in globals(), (
        f"Required function "
        f"'{function_name}' is unavailable."
    )

    function_rows.append({
        "Function": function_name,
        "Signature": str(
            inspect.signature(
                globals()[function_name]
            )
        ),
    })

print("\nReusable training functions:")

display(
    pd.DataFrame(function_rows)
)

# ------------------------------------------------------------
# 5. Inspect KAN and SSM components
# ------------------------------------------------------------

component_rows = []

for component_name in [
    "SparseKAN",
    "DSSSM",
]:

    if component_name not in globals():

        component_rows.append({
            "Component": component_name,
            "Available": False,
            "Constructor": "Unavailable",
            "Forward": "Unavailable",
        })

        continue

    component_class = globals()[
        component_name
    ]

    try:
        constructor_signature = str(
            inspect.signature(
                component_class
            )
        )
    except Exception as error:
        constructor_signature = str(error)

    try:
        forward_signature = str(
            inspect.signature(
                component_class.forward
            )
        )
    except Exception as error:
        forward_signature = str(error)

    component_rows.append({
        "Component": component_name,
        "Available": True,
        "Constructor": (
            constructor_signature
        ),
        "Forward": forward_signature,
    })

print("\nReusable component interfaces:")

display(
    pd.DataFrame(component_rows)
)

# ------------------------------------------------------------
# 6. Device verification
# ------------------------------------------------------------

baseline_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nBaseline device:", baseline_device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    torch.cuda.empty_cache()

print(
    "\nOfficial DAPT2020 and "
    "CICIDS2017 arrays verified."
)

Official baseline arrays:


,Dataset,Split,Sequences,Benign,Attack,Shape,Input dtype
0,DAPT2020,train,29147,20251,8896,"(29147, 8, 67)",float32
1,DAPT2020,val,6067,4162,1905,"(6067, 8, 67)",float32
2,DAPT2020,test,6694,4788,1906,"(6694, 8, 67)",float32
3,CICIDS2017,train,243814,195449,48365,"(243814, 8, 72)",float32
4,CICIDS2017,val,54458,41966,12492,"(54458, 8, 72)",float32
5,CICIDS2017,test,54499,41965,12534,"(54499, 8, 72)",float32



Reusable training functions:


,Function,Signature
0,make_loader,"(x, y, cfg: 'Config', shuffle: 'bool') -> 'Dat..."
1,select_threshold,"(y: 'np.ndarray', p: 'np.ndarray') -> 'float'"
2,compute_metrics,"(y: 'np.ndarray', p: 'np.ndarray', threshold: ..."
3,set_seed,"(seed: 'int', deterministic: 'bool' = True) ->..."



Reusable component interfaces:


,Component,Available,Constructor,Forward
0,SparseKAN,True,"(in_dim: 'int', out_dim: 'int', num_basis: 'in...","(self, x: 'torch.Tensor', threshold: 'torch.Te..."
1,DSSSM,True,"(in_dim: 'int', state_dim: 'int', dropout: 'fl...","(self, x: 'torch.Tensor', delta: 'torch.Tensor..."



Baseline device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Official DAPT2020 and CICIDS2017 arrays verified.


In [76]:
# ============================================================
# Baseline Experiment — Cell 2
# Define and verify the six neural baseline architectures
# ============================================================

import math
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

# ------------------------------------------------------------
# 1. Common output wrapper
# ------------------------------------------------------------

class BaselineOutputMixin:

    def format_output(
        self,
        logits,
        embedding,
        return_aux=False,
    ):

        if not return_aux:
            return logits

        auxiliary = {
            "embedding": embedding
        }

        return logits, auxiliary


# ------------------------------------------------------------
# 2. Multilayer perceptron
# ------------------------------------------------------------

class MLPBaseline(
    nn.Module,
    BaselineOutputMixin,
):

    def __init__(
        self,
        input_dim,
        sequence_length=8,
        hidden_dim=128,
        dropout=0.2,
    ):

        super().__init__()

        flattened_dim = (
            input_dim
            * sequence_length
        )

        self.encoder = nn.Sequential(
            nn.Linear(
                flattened_dim,
                256,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                256,
                hidden_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(
            hidden_dim,
            2,
        )

    def forward(
        self,
        x,
        return_aux=False,
    ):

        embedding = self.encoder(
            x.flatten(start_dim=1)
        )

        logits = self.classifier(
            embedding
        )

        return self.format_output(
            logits,
            embedding,
            return_aux,
        )


# ------------------------------------------------------------
# 3. One-dimensional CNN
# ------------------------------------------------------------

class CNNBaseline(
    nn.Module,
    BaselineOutputMixin,
):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        dropout=0.2,
    ):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(
                input_dim,
                128,
                kernel_size=3,
                padding=1,
            ),
            nn.GELU(),
            nn.BatchNorm1d(128),
            nn.Conv1d(
                128,
                hidden_dim,
                kernel_size=3,
                padding=1,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(
            hidden_dim,
            2,
        )

    def forward(
        self,
        x,
        return_aux=False,
    ):

        # B x T x D -> B x D x T
        hidden = self.encoder(
            x.transpose(1, 2)
        )

        embedding = hidden.mean(dim=2)

        logits = self.classifier(
            embedding
        )

        return self.format_output(
            logits,
            embedding,
            return_aux,
        )


# ------------------------------------------------------------
# 4. Bidirectional LSTM
# ------------------------------------------------------------

class BiLSTMBaseline(
    nn.Module,
    BaselineOutputMixin,
):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        dropout=0.2,
    ):

        super().__init__()

        recurrent_hidden = (
            hidden_dim // 2
        )

        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim,
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=recurrent_hidden,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=True,
        )

        self.output_dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            hidden_dim,
            2,
        )

    def forward(
        self,
        x,
        return_aux=False,
    ):

        projected = self.input_projection(
            x
        )

        sequence_output, _ = self.lstm(
            projected
        )

        embedding = sequence_output.mean(
            dim=1
        )

        embedding = self.output_dropout(
            embedding
        )

        logits = self.classifier(
            embedding
        )

        return self.format_output(
            logits,
            embedding,
            return_aux,
        )


# ------------------------------------------------------------
# 5. Transformer encoder
# ------------------------------------------------------------

class TransformerBaseline(
    nn.Module,
    BaselineOutputMixin,
):

    def __init__(
        self,
        input_dim,
        sequence_length=8,
        hidden_dim=128,
        number_of_heads=4,
        number_of_layers=2,
        dropout=0.2,
    ):

        super().__init__()

        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim,
        )

        self.position_embedding = (
            nn.Parameter(
                torch.zeros(
                    1,
                    sequence_length,
                    hidden_dim,
                )
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=number_of_heads,
                dim_feedforward=256,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=number_of_layers,
        )

        self.output_normalization = (
            nn.LayerNorm(hidden_dim)
        )

        self.classifier = nn.Linear(
            hidden_dim,
            2,
        )

    def forward(
        self,
        x,
        return_aux=False,
    ):

        hidden = self.input_projection(
            x
        )

        hidden = (
            hidden
            + self.position_embedding[
                :,
                :hidden.shape[1],
            ]
        )

        hidden = self.encoder(hidden)

        embedding = (
            self.output_normalization(
                hidden.mean(dim=1)
            )
        )

        logits = self.classifier(
            embedding
        )

        return self.format_output(
            logits,
            embedding,
            return_aux,
        )


# ------------------------------------------------------------
# 6. Standard RBF Kolmogorov-Arnold Network
#
# This baseline uses dense learnable basis mappings.
# It does not use:
#   - adaptive sparsity
#   - controller-generated thresholds
#   - SwiGLU
#   - DSSSM
#   - adaptive fusion
# ------------------------------------------------------------

class DenseKANLayer(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim,
        number_of_basis=8,
        basis_min=-3.0,
        basis_max=3.0,
    ):

        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.number_of_basis = (
            number_of_basis
        )

        basis_centers = torch.linspace(
            basis_min,
            basis_max,
            number_of_basis,
        )

        self.register_buffer(
            "basis_centers",
            basis_centers,
        )

        center_spacing = (
            basis_centers[1]
            - basis_centers[0]
            if number_of_basis > 1
            else torch.tensor(1.0)
        )

        self.log_basis_width = (
            nn.Parameter(
                torch.log(
                    torch.full(
                        (
                            input_dim,
                            number_of_basis,
                        ),
                        float(
                            center_spacing
                        ),
                    )
                )
            )
        )

        self.coefficients = nn.Parameter(
            torch.empty(
                input_dim,
                number_of_basis,
                output_dim,
            )
        )

        self.base_weight = nn.Parameter(
            torch.empty(
                input_dim,
                output_dim,
            )
        )

        self.bias = nn.Parameter(
            torch.zeros(output_dim)
        )

        nn.init.xavier_uniform_(
            self.coefficients.view(
                input_dim
                * number_of_basis,
                output_dim,
            )
        )

        nn.init.xavier_uniform_(
            self.base_weight
        )

    def forward(self, x):

        # x: B x T x D
        expanded_x = x.unsqueeze(-1)

        centers = (
            self.basis_centers
            .view(
                1,
                1,
                1,
                self.number_of_basis,
            )
        )

        widths = (
            F.softplus(
                self.log_basis_width
            )
            .view(
                1,
                1,
                self.input_dim,
                self.number_of_basis,
            )
            + 1e-4
        )

        basis = torch.exp(
            -0.5
            * (
                (
                    expanded_x
                    - centers
                )
                / widths
            ).square()
        )

        basis_output = torch.einsum(
            "btdk,dko->bto",
            basis,
            self.coefficients,
        )

        base_output = torch.einsum(
            "btd,do->bto",
            F.silu(x),
            self.base_weight,
        )

        return (
            basis_output
            + base_output
            + self.bias
        )


class KANBaseline(
    nn.Module,
    BaselineOutputMixin,
):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        number_of_basis=8,
        dropout=0.2,
    ):

        super().__init__()

        self.kan_layer = DenseKANLayer(
            input_dim=input_dim,
            output_dim=hidden_dim,
            number_of_basis=(
                number_of_basis
            ),
        )

        self.output_network = nn.Sequential(
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(
            hidden_dim,
            2,
        )

    def forward(
        self,
        x,
        return_aux=False,
    ):

        hidden = self.kan_layer(x)

        hidden = self.output_network(
            hidden
        )

        embedding = hidden.mean(dim=1)

        logits = self.classifier(
            embedding
        )

        return self.format_output(
            logits,
            embedding,
            return_aux,
        )


# ------------------------------------------------------------
# 7. Selective state-space baseline
#
# A single-branch diagonal selective SSM.
# It excludes EDyT, SKAN, SwiGLU, DSSSM dual gating,
# the adaptive controller, and adaptive fusion.
# ------------------------------------------------------------

class SelectiveSSMBaseline(
    nn.Module,
    BaselineOutputMixin,
):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        state_dim=96,
        dropout=0.2,
    ):

        super().__init__()

        self.state_dim = state_dim

        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim,
        )

        self.delta_projection = nn.Linear(
            hidden_dim,
            state_dim,
        )

        self.input_to_state = nn.Linear(
            hidden_dim,
            state_dim,
        )

        self.log_decay = nn.Parameter(
            torch.zeros(state_dim)
        )

        self.state_to_output = nn.Linear(
            state_dim,
            hidden_dim,
        )

        self.skip_projection = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

        self.output_network = nn.Sequential(
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(
            hidden_dim,
            2,
        )

    def forward(
        self,
        x,
        return_aux=False,
    ):

        projected = self.input_projection(
            x
        )

        batch_size = projected.shape[0]

        state = torch.zeros(
            batch_size,
            self.state_dim,
            dtype=projected.dtype,
            device=projected.device,
        )

        decay_rate = F.softplus(
            self.log_decay
        ).unsqueeze(0)

        sequence_outputs = []

        for time_index in range(
            projected.shape[1]
        ):

            current_input = projected[
                :,
                time_index,
            ]

            delta = F.softplus(
                self.delta_projection(
                    current_input
                )
            ) + 1e-4

            transition = torch.exp(
                -decay_rate
                * delta
            )

            candidate_state = torch.tanh(
                self.input_to_state(
                    current_input
                )
            )

            state = (
                transition
                * state
                + (
                    1.0
                    - transition
                )
                * candidate_state
            )

            current_output = (
                self.state_to_output(
                    state
                )
                + self.skip_projection(
                    current_input
                )
            )

            sequence_outputs.append(
                current_output
            )

        sequence_output = torch.stack(
            sequence_outputs,
            dim=1,
        )

        sequence_output = (
            self.output_network(
                sequence_output
            )
        )

        embedding = (
            sequence_output.mean(dim=1)
        )

        logits = self.classifier(
            embedding
        )

        return self.format_output(
            logits,
            embedding,
            return_aux,
        )


# ------------------------------------------------------------
# 8. Baseline model factory
# ------------------------------------------------------------

neural_baseline_names = (
    "mlp",
    "cnn",
    "bilstm",
    "transformer",
    "kan",
    "selective_ssm",
)

def build_neural_baseline(
    baseline_name,
    input_dim,
    sequence_length=8,
):

    baseline_name = (
        baseline_name
        .strip()
        .lower()
    )

    if baseline_name == "mlp":

        return MLPBaseline(
            input_dim=input_dim,
            sequence_length=sequence_length,
        )

    if baseline_name == "cnn":

        return CNNBaseline(
            input_dim=input_dim,
        )

    if baseline_name == "bilstm":

        return BiLSTMBaseline(
            input_dim=input_dim,
        )

    if baseline_name == "transformer":

        return TransformerBaseline(
            input_dim=input_dim,
            sequence_length=sequence_length,
        )

    if baseline_name == "kan":

        return KANBaseline(
            input_dim=input_dim,
        )

    if baseline_name == "selective_ssm":

        return SelectiveSSMBaseline(
            input_dim=input_dim,
        )

    raise ValueError(
        f"Unknown neural baseline: "
        f"{baseline_name}"
    )


# ------------------------------------------------------------
# 9. Forward-pass verification on both feature dimensions
# ------------------------------------------------------------

baseline_forward_rows = []

for dataset_name, input_dim in [
    ("DAPT2020", 67),
    ("CICIDS2017", 72),
]:

    probe_x = torch.randn(
        4,
        8,
        input_dim,
        device=baseline_device,
    )

    for baseline_name in (
        neural_baseline_names
    ):

        model = build_neural_baseline(
            baseline_name=baseline_name,
            input_dim=input_dim,
            sequence_length=8,
        ).to(baseline_device)

        model.eval()

        with torch.no_grad():

            logits, auxiliary = model(
                probe_x,
                return_aux=True,
            )

        assert logits.shape == (4, 2)

        assert (
            auxiliary["embedding"].shape[0]
            == 4
        )

        assert torch.isfinite(
            logits
        ).all()

        assert torch.isfinite(
            auxiliary["embedding"]
        ).all()

        parameter_count = sum(
            parameter.numel()
            for parameter
            in model.parameters()
            if parameter.requires_grad
        )

        baseline_forward_rows.append({
            "Dataset": dataset_name,
            "Baseline": baseline_name,
            "Parameters": parameter_count,
            "Logits": tuple(
                logits.shape
            ),
            "Embedding": tuple(
                auxiliary[
                    "embedding"
                ].shape
            ),
        })

        del model

    del probe_x

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(
    "Neural baseline forward-pass audit:"
)

display(
    pd.DataFrame(
        baseline_forward_rows
    )
)

print(
    "\nAll six neural baseline "
    "forward tests passed."
)

<CONDA_ROOT>\envs\hlcda\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Neural baseline forward-pass audit:


,Dataset,Baseline,Parameters,Logits,Embedding
0,DAPT2020,mlp,170626,"(4, 2)","(4, 128)"
1,DAPT2020,cnn,75650,"(4, 2)","(4, 128)"
2,DAPT2020,bilstm,207618,"(4, 2)","(4, 128)"
3,DAPT2020,transformer,275202,"(4, 2)","(4, 128)"
4,DAPT2020,kan,94618,"(4, 2)","(4, 128)"
5,DAPT2020,selective_ssm,62754,"(4, 2)","(4, 128)"
6,CICIDS2017,mlp,180866,"(4, 2)","(4, 128)"
7,CICIDS2017,cnn,77570,"(4, 2)","(4, 128)"
8,CICIDS2017,bilstm,208258,"(4, 2)","(4, 128)"
9,CICIDS2017,transformer,275842,"(4, 2)","(4, 128)"



All six neural baseline forward tests passed.


In [77]:
# ============================================================
# Baseline Experiment — Cell 3
# Common neural-baseline training pipeline + MLP pilot
# ============================================================

from dataclasses import replace, asdict
from pathlib import Path
import time
import json
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Baseline prediction function
# ------------------------------------------------------------

@torch.no_grad()
def predict_neural_baseline(
    model,
    loader,
    device,
):

    model.eval()

    true_rows = []
    probability_rows = []

    for sequence_x, sequence_y in loader:

        sequence_x = sequence_x.to(
            device,
            non_blocking=True,
        )

        logits = model(sequence_x)

        probability = torch.softmax(
            logits,
            dim=-1,
        )[:, 1]

        if not torch.isfinite(
            probability
        ).all():

            raise ValueError(
                "Baseline prediction contains "
                "non-finite probabilities."
            )

        true_rows.append(
            sequence_y.numpy()
        )

        probability_rows.append(
            probability
            .detach()
            .cpu()
            .numpy()
        )

    return (
        np.concatenate(true_rows),
        np.concatenate(
            probability_rows
        ),
    )


# ------------------------------------------------------------
# 2. JSON helper
# ------------------------------------------------------------

def save_baseline_json(
    data,
    output_path,
):

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    serializable_data = {}

    for key, value in data.items():

        if isinstance(
            value,
            np.generic,
        ):
            serializable_data[key] = (
                value.item()
            )
        else:
            serializable_data[key] = value

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            serializable_data,
            file,
            indent=2,
        )


# ------------------------------------------------------------
# 3. Train one neural baseline for one seed
# ------------------------------------------------------------

def train_neural_baseline_seed(
    cfg,
    dataset_name,
    baseline_name,
    seed,
    arrays,
    device,
    experiment_root,
):

    set_seed(
        seed,
        cfg.deterministic,
    )

    run_directory = (
        Path(experiment_root)
        / dataset_name
        / baseline_name
        / f"seed_{seed}"
    )

    run_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    train_x, train_y = arrays["train"]
    val_x, val_y = arrays["val"]
    test_x, test_y = arrays["test"]

    train_loader = make_loader(
        train_x,
        train_y,
        cfg,
        shuffle=True,
    )

    validation_loader = make_loader(
        val_x,
        val_y,
        cfg,
        shuffle=False,
    )

    test_loader = make_loader(
        test_x,
        test_y,
        cfg,
        shuffle=False,
    )

    model = build_neural_baseline(
        baseline_name=baseline_name,
        input_dim=train_x.shape[-1],
        sequence_length=(
            train_x.shape[1]
        ),
    ).to(device)

    parameter_count = sum(
        parameter.numel()
        for parameter
        in model.parameters()
        if parameter.requires_grad
    )

    class_counts = np.bincount(
        train_y,
        minlength=2,
    )

    class_weights = torch.tensor(
        len(train_y)
        / np.maximum(
            2 * class_counts,
            1,
        ),
        dtype=torch.float32,
        device=device,
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=cfg.epochs,
        )
    )

    amp_enabled = (
        cfg.use_amp
        and device.type == "cuda"
    )

    gradient_scaler = (
        torch.amp.GradScaler(
            "cuda",
            enabled=amp_enabled,
        )
    )

    optimizer_information = {
        "dataset": dataset_name,
        "baseline": baseline_name,
        "seed": seed,
        "optimizer": "AdamW",
        "initial_learning_rate": (
            cfg.learning_rate
        ),
        "weight_decay": (
            cfg.weight_decay
        ),
        "scheduler": (
            "CosineAnnealingLR"
        ),
        "scheduler_T_max": cfg.epochs,
        "gradient_clip": (
            cfg.gradient_clip
        ),
        "mixed_precision": (
            amp_enabled
        ),
        "parameters": (
            parameter_count
        ),
    }

    save_baseline_json(
        optimizer_information,
        run_directory
        / "optimizer_settings.json",
    )

    best_validation_auprc = -np.inf
    best_epoch = 0
    stale_epochs = 0
    history_rows = []

    training_start_time = (
        time.perf_counter()
    )

    for epoch in range(
        1,
        cfg.epochs + 1,
    ):

        epoch_start_time = (
            time.perf_counter()
        )

        model.train()

        running_loss = 0.0
        observed_samples = 0

        for sequence_x, sequence_y in (
            train_loader
        ):

            sequence_x = sequence_x.to(
                device,
                non_blocking=True,
            )

            sequence_y = sequence_y.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):

                logits = model(sequence_x)

                loss = criterion(
                    logits,
                    sequence_y,
                )

            if not torch.isfinite(loss):

                raise ValueError(
                    f"Non-finite loss for "
                    f"{dataset_name}, "
                    f"{baseline_name}, "
                    f"seed {seed}, "
                    f"epoch {epoch}."
                )

            gradient_scaler.scale(
                loss
            ).backward()

            gradient_scaler.unscale_(
                optimizer
            )

            nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.gradient_clip,
            )

            gradient_scaler.step(
                optimizer
            )

            gradient_scaler.update()

            running_loss += (
                float(
                    loss.detach()
                )
                * len(sequence_y)
            )

            observed_samples += len(
                sequence_y
            )

        scheduler.step()

        validation_true, validation_prob = (
            predict_neural_baseline(
                model,
                validation_loader,
                device,
            )
        )

        validation_threshold = (
            select_threshold(
                validation_true,
                validation_prob,
            )
        )

        validation_metrics = (
            compute_metrics(
                validation_true,
                validation_prob,
                validation_threshold,
            )
        )

        epoch_runtime = (
            time.perf_counter()
            - epoch_start_time
        )

        epoch_row = {
            "epoch": epoch,
            "train_loss": (
                running_loss
                / observed_samples
            ),
            "learning_rate": (
                optimizer
                .param_groups[0]["lr"]
            ),
            "epoch_runtime_seconds": (
                epoch_runtime
            ),
            **validation_metrics,
        }

        history_rows.append(
            epoch_row
        )

        current_auprc = (
            validation_metrics[
                "auprc"
            ]
        )

        if (
            current_auprc
            > best_validation_auprc
            + 1e-5
        ):

            best_validation_auprc = (
                current_auprc
            )

            best_epoch = epoch
            stale_epochs = 0

            torch.save(
                {
                    "model": (
                        model.state_dict()
                    ),
                    "dataset": (
                        dataset_name
                    ),
                    "baseline": (
                        baseline_name
                    ),
                    "seed": seed,
                    "input_dim": (
                        train_x.shape[-1]
                    ),
                    "sequence_length": (
                        train_x.shape[1]
                    ),
                    "config": asdict(cfg),
                },
                run_directory
                / "best_model.pt",
            )

        else:

            stale_epochs += 1

        print({
            "dataset": dataset_name,
            "baseline": baseline_name,
            "seed": seed,
            "epoch": epoch,
            "train_loss": round(
                epoch_row[
                    "train_loss"
                ],
                6,
            ),
            "val_auprc": round(
                current_auprc,
                6,
            ),
            "val_f1": round(
                validation_metrics[
                    "f1"
                ],
                6,
            ),
            "seconds": round(
                epoch_runtime,
                2,
            ),
        })

        if stale_epochs >= cfg.patience:
            break

    training_runtime_seconds = (
        time.perf_counter()
        - training_start_time
    )

    history_frame = pd.DataFrame(
        history_rows
    )

    history_frame.to_csv(
        run_directory
        / "history.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Restore checkpoint selected by validation AUPRC
    # --------------------------------------------------------

    checkpoint = torch.load(
        run_directory
        / "best_model.pt",
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    # Threshold is selected on validation data only
    validation_true, validation_prob = (
        predict_neural_baseline(
            model,
            validation_loader,
            device,
        )
    )

    final_threshold = select_threshold(
        validation_true,
        validation_prob,
    )

    # Final test evaluation
    test_true, test_prob = (
        predict_neural_baseline(
            model,
            test_loader,
            device,
        )
    )

    test_metrics = compute_metrics(
        test_true,
        test_prob,
        final_threshold,
    )

    test_metrics.update({
        "dataset": dataset_name,
        "method": baseline_name,
        "seed": seed,
        "best_epoch": best_epoch,
        "epochs_completed": int(
            len(history_frame)
        ),
        "parameters": parameter_count,
        "runtime_seconds": (
            training_runtime_seconds
        ),
        "runtime_minutes": (
            training_runtime_seconds
            / 60.0
        ),
        "numerical_precision": (
            "AMP"
            if amp_enabled
            else "FP32"
        ),
    })

    pd.DataFrame({
        "label": test_true,
        "probability_attack": (
            test_prob
        ),
    }).to_csv(
        run_directory
        / "test_predictions.csv",
        index=False,
    )

    save_baseline_json(
        test_metrics,
        run_directory
        / "test_metrics.json",
    )

    del (
        model,
        train_loader,
        validation_loader,
        test_loader,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return test_metrics


# ------------------------------------------------------------
# 4. Five-epoch DAPT2020 MLP pilot
# ------------------------------------------------------------

baseline_output_root = (
    PROJECT_ROOT
    / "outputs"
    / "baseline_experiments"
)

baseline_pilot_root = (
    PROJECT_ROOT
    / "outputs"
    / "baseline_pilot"
)

dapt_baseline_pilot_cfg = replace(
    dapt_cfg,
    output_dir=str(
        baseline_pilot_root
        / "DAPT2020"
    ),
    epochs=5,
    patience=5,
    seeds=(13,),
)

print(
    "Pilot output:",
    baseline_pilot_root
)

print(
    "Pilot model: MLP"
)

print(
    "Pilot dataset: DAPT2020"
)

print(
    "Pilot epochs:",
    dapt_baseline_pilot_cfg.epochs
)

print(
    "Device:",
    baseline_device
)

dapt_mlp_pilot_result = (
    train_neural_baseline_seed(
        cfg=dapt_baseline_pilot_cfg,
        dataset_name="DAPT2020",
        baseline_name="mlp",
        seed=13,
        arrays=baseline_dapt_arrays,
        device=baseline_device,
        experiment_root=(
            baseline_pilot_root
        ),
    )
)

print("\nDAPT2020 MLP pilot result:")

display(
    pd.DataFrame(
        [dapt_mlp_pilot_result]
    )
)

Pilot output: <PROJECT_ROOT>\outputs\baseline_pilot
Pilot model: MLP
Pilot dataset: DAPT2020
Pilot epochs: 5
Device: cuda
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 1, 'train_loss': 0.276918, 'val_auprc': 0.921536, 'val_f1': 0.911213, 'seconds': 0.63}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 2, 'train_loss': 0.14261, 'val_auprc': 0.930102, 'val_f1': 0.925804, 'seconds': 0.48}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 3, 'train_loss': 0.128911, 'val_auprc': 0.931338, 'val_f1': 0.92736, 'seconds': 0.45}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 4, 'train_loss': 0.122405, 'val_auprc': 0.932375, 'val_f1': 0.928371, 'seconds': 0.39}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 5, 'train_loss': 0.119162, 'val_auprc': 0.932662, 'val_f1': 0.928826, 'seconds': 0.38}

DAPT2020 MLP pilot result:


,threshold,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,auroc,auprc,...,tp,dataset,method,seed,best_epoch,epochs_completed,parameters,runtime_seconds,runtime_minutes,numerical_precision
0,0.734473,0.959516,0.960173,0.902511,0.9617,0.958647,0.931166,0.903446,0.991428,0.975474,...,1833,DAPT2020,mlp,13,5,5,170626,2.336529,0.038942,AMP


In [78]:
# ============================================================
# Baseline Experiment — Cell 4
# Official DAPT2020 neural baselines: six models × five seeds
# ============================================================

from dataclasses import replace
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import json
import gc

# ------------------------------------------------------------
# 1. Official DAPT2020 baseline configuration
# ------------------------------------------------------------

official_baseline_seeds = (
    13,
    27,
    41,
    55,
    69,
)

dapt_baseline_output = (
    PROJECT_ROOT
    / "outputs"
    / "baseline_experiments"
)

dapt_official_baseline_cfg = replace(
    dapt_cfg,
    output_dir=str(
        dapt_baseline_output
        / "DAPT2020"
    ),
    epochs=60,
    patience=10,
    seeds=official_baseline_seeds,
)

print(
    "Official baseline output:",
    dapt_baseline_output
)

print(
    "Dataset: DAPT2020"
)

print(
    "Neural baselines:",
    neural_baseline_names
)

print(
    "Seeds:",
    official_baseline_seeds
)

print(
    "Epoch limit:",
    dapt_official_baseline_cfg.epochs
)

print(
    "Patience:",
    dapt_official_baseline_cfg.patience
)

print(
    "Device:",
    baseline_device
)

# ------------------------------------------------------------
# 2. Verify official DAPT2020 arrays
# ------------------------------------------------------------

assert baseline_dapt_arrays[
    "train"
][0].shape == (
    29_147,
    8,
    67,
)

assert baseline_dapt_arrays[
    "val"
][0].shape == (
    6_067,
    8,
    67,
)

assert baseline_dapt_arrays[
    "test"
][0].shape == (
    6_694,
    8,
    67,
)

print(
    "Official DAPT2020 arrays verified."
)

# ------------------------------------------------------------
# 3. Numerical-error detector
# ------------------------------------------------------------

def is_baseline_nonfinite_error(
    error,
):

    error_message = str(
        error
    ).lower()

    numerical_terms = (
        "nan",
        "infinity",
        "infinite",
        "non-finite",
        "not finite",
    )

    return any(
        term in error_message
        for term in numerical_terms
    )

# ------------------------------------------------------------
# 4. Train with resume support
# ------------------------------------------------------------

dapt_neural_baseline_rows = []

for baseline_name in (
    neural_baseline_names
):

    for seed in (
        official_baseline_seeds
    ):

        run_directory = (
            dapt_baseline_output
            / "DAPT2020"
            / baseline_name
            / f"seed_{seed}"
        )

        metrics_path = (
            run_directory
            / "test_metrics.json"
        )

        # ----------------------------------------------------
        # Resume completed run
        # ----------------------------------------------------

        if metrics_path.exists():

            with open(
                metrics_path,
                "r",
                encoding="utf-8",
            ) as file:
                metrics = json.load(file)

            dapt_neural_baseline_rows.append(
                metrics
            )

            print(
                "Loaded completed run:",
                baseline_name,
                seed,
            )

            continue

        print(
            "\nTraining official DAPT2020 "
            f"baseline: {baseline_name}, "
            f"seed {seed}"
        )

        # ----------------------------------------------------
        # First attempt: configured AMP mode
        # ----------------------------------------------------

        try:

            metrics = (
                train_neural_baseline_seed(
                    cfg=(
                        dapt_official_baseline_cfg
                    ),
                    dataset_name="DAPT2020",
                    baseline_name=(
                        baseline_name
                    ),
                    seed=seed,
                    arrays=(
                        baseline_dapt_arrays
                    ),
                    device=baseline_device,
                    experiment_root=(
                        dapt_baseline_output
                    ),
                )
            )

        except (
            ValueError,
            RuntimeError,
        ) as error:

            if not is_baseline_nonfinite_error(
                error
            ):
                raise

            print(
                "Non-finite values detected "
                f"for {baseline_name}, "
                f"seed {seed}."
            )

            print(
                "Retrying this run in FP32."
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()

            fp32_cfg = replace(
                dapt_official_baseline_cfg,
                use_amp=False,
            )

            metrics = (
                train_neural_baseline_seed(
                    cfg=fp32_cfg,
                    dataset_name="DAPT2020",
                    baseline_name=(
                        baseline_name
                    ),
                    seed=seed,
                    arrays=(
                        baseline_dapt_arrays
                    ),
                    device=baseline_device,
                    experiment_root=(
                        dapt_baseline_output
                    ),
                )
            )

            metrics[
                "numerical_precision"
            ] = "FP32 fallback"

            with open(
                metrics_path,
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    metrics,
                    file,
                    indent=2,
                )

        # ----------------------------------------------------
        # Verify returned results
        # ----------------------------------------------------

        for metric_name in [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "mcc",
            "auroc",
            "auprc",
        ]:

            assert np.isfinite(
                metrics[metric_name]
            ), (
                f"Non-finite {metric_name} "
                f"for {baseline_name}, "
                f"seed {seed}."
            )

        dapt_neural_baseline_rows.append(
            metrics
        )

        print(
            "\nCompleted result:",
            {
                "method": baseline_name,
                "seed": seed,
                "accuracy": (
                    metrics["accuracy"]
                ),
                "precision": (
                    metrics["precision"]
                ),
                "recall": (
                    metrics["recall"]
                ),
                "f1": metrics["f1"],
                "mcc": metrics["mcc"],
                "auroc": (
                    metrics["auroc"]
                ),
                "auprc": (
                    metrics["auprc"]
                ),
                "best_epoch": (
                    metrics["best_epoch"]
                ),
                "runtime_minutes": (
                    metrics[
                        "runtime_minutes"
                    ]
                ),
            },
        )

        # Save progress after every run
        pd.DataFrame(
            dapt_neural_baseline_rows
        ).to_csv(
            dapt_baseline_output
            / "DAPT2020"
            / (
                "neural_baseline_"
                "seed_results_partial.csv"
            ),
            index=False,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5. Final neural-baseline seed-level results
# ------------------------------------------------------------

dapt_neural_baseline_results = (
    pd.DataFrame(
        dapt_neural_baseline_rows
    )
)

assert len(
    dapt_neural_baseline_results
) == (
    len(neural_baseline_names)
    * len(official_baseline_seeds)
)

dapt_neural_baseline_results.to_csv(
    dapt_baseline_output
    / "DAPT2020"
    / "neural_baseline_seed_results.csv",
    index=False,
)

print(
    "\nAll official DAPT2020 "
    "neural baselines completed."
)

display(
    dapt_neural_baseline_results[
        [
            "method",
            "seed",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "best_epoch",
            "parameters",
            "runtime_minutes",
            "numerical_precision",
        ]
    ]
)

# ------------------------------------------------------------
# 6. Preliminary mean results
# ------------------------------------------------------------

dapt_neural_baseline_means = (
    dapt_neural_baseline_results
    .groupby("method")[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "parameters",
            "runtime_minutes",
        ]
    ]
    .mean()
    .sort_values(
        by="f1",
        ascending=False,
    )
)

print(
    "\nDAPT2020 neural-baseline "
    "mean results:"
)

display(
    dapt_neural_baseline_means
)

Official baseline output: <PROJECT_ROOT>\outputs\baseline_experiments
Dataset: DAPT2020
Neural baselines: ('mlp', 'cnn', 'bilstm', 'transformer', 'kan', 'selective_ssm')
Seeds: (13, 27, 41, 55, 69)
Epoch limit: 60
Patience: 10
Device: cuda
Official DAPT2020 arrays verified.

Training official DAPT2020 baseline: mlp, seed 13
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 1, 'train_loss': 0.276918, 'val_auprc': 0.921536, 'val_f1': 0.911213, 'seconds': 0.51}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 2, 'train_loss': 0.142171, 'val_auprc': 0.930271, 'val_f1': 0.926348, 'seconds': 0.37}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 3, 'train_loss': 0.127698, 'val_auprc': 0.931838, 'val_f1': 0.927203, 'seconds': 0.34}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 4, 'train_loss': 0.120484, 'val_auprc': 0.934294, 'val_f1': 0.930043, 'seconds': 0.35}
{'dataset': 'DAPT2020', 'baseline': 'mlp', 'seed': 13, 'epoch': 5, 'tr

<CONDA_ROOT>\envs\hlcda\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 13, 'epoch': 1, 'train_loss': 0.176976, 'val_auprc': 0.97831, 'val_f1': 0.928315, 'seconds': 0.87}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 13, 'epoch': 2, 'train_loss': 0.10823, 'val_auprc': 0.987397, 'val_f1': 0.941718, 'seconds': 0.87}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 13, 'epoch': 3, 'train_loss': 0.093238, 'val_auprc': 0.989755, 'val_f1': 0.94247, 'seconds': 0.77}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 13, 'epoch': 4, 'train_loss': 0.08498, 'val_auprc': 0.990432, 'val_f1': 0.948969, 'seconds': 0.85}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 13, 'epoch': 5, 'train_loss': 0.080685, 'val_auprc': 0.991147, 'val_f1': 0.95176, 'seconds': 0.85}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 13, 'epoch': 6, 'train_loss': 0.076668, 'val_auprc': 0.991741, 'val_f1': 0.953673, 'seconds': 0.81}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 

<CONDA_ROOT>\envs\hlcda\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 27, 'epoch': 1, 'train_loss': 0.170325, 'val_auprc': 0.976987, 'val_f1': 0.930101, 'seconds': 0.74}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 27, 'epoch': 2, 'train_loss': 0.107448, 'val_auprc': 0.985047, 'val_f1': 0.935981, 'seconds': 0.73}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 27, 'epoch': 3, 'train_loss': 0.096309, 'val_auprc': 0.987988, 'val_f1': 0.943444, 'seconds': 0.75}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 27, 'epoch': 4, 'train_loss': 0.086465, 'val_auprc': 0.990967, 'val_f1': 0.95134, 'seconds': 0.72}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 27, 'epoch': 5, 'train_loss': 0.083494, 'val_auprc': 0.991287, 'val_f1': 0.951958, 'seconds': 0.73}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 27, 'epoch': 6, 'train_loss': 0.078521, 'val_auprc': 0.992245, 'val_f1': 0.952258, 'seconds': 0.72}
{'dataset': 'DAPT2020', 'baseline': 'transforme

...

<CONDA_ROOT>\envs\hlcda\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 69, 'epoch': 1, 'train_loss': 0.176849, 'val_auprc': 0.981085, 'val_f1': 0.932541, 'seconds': 0.74}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 69, 'epoch': 2, 'train_loss': 0.107996, 'val_auprc': 0.983953, 'val_f1': 0.934413, 'seconds': 0.76}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 69, 'epoch': 3, 'train_loss': 0.096618, 'val_auprc': 0.988198, 'val_f1': 0.945848, 'seconds': 0.76}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 69, 'epoch': 4, 'train_loss': 0.086248, 'val_auprc': 0.988111, 'val_f1': 0.941788, 'seconds': 0.73}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 69, 'epoch': 5, 'train_loss': 0.083727, 'val_auprc': 0.990422, 'val_f1': 0.950582, 'seconds': 0.74}
{'dataset': 'DAPT2020', 'baseline': 'transformer', 'seed': 69, 'epoch': 6, 'train_loss': 0.080367, 'val_auprc': 0.991398, 'val_f1': 0.947691, 'seconds': 0.72}
{'dataset': 'DAPT2020', 'baseline': 'transform

,method,seed,accuracy,precision,recall,f1,mcc,auroc,auprc,best_epoch,parameters,runtime_minutes,numerical_precision
0,mlp,13,0.953242,0.878385,0.970094,0.921965,0.890936,0.987128,0.963859,51,170626,0.381878,AMP
1,mlp,27,0.955781,0.875466,0.984785,0.926914,0.898557,0.990422,0.973637,50,170626,0.386396,AMP
2,mlp,41,0.957275,0.877446,0.987933,0.929418,0.902188,0.992512,0.979927,48,170626,0.412292,AMP
3,mlp,55,0.951300,0.871590,0.972193,0.919147,0.887152,0.992583,0.981505,41,170626,0.386301,AMP
4,mlp,69,0.952644,0.875650,0.971668,0.921164,0.889906,0.984231,0.953682,50,170626,0.420335,AMP
5,cnn,13,0.967882,0.911836,0.982162,0.945693,0.924220,0.995827,0.983886,32,75650,0.593317,AMP
6,cnn,27,0.965491,0.903226,0.984260,0.942004,0.919222,0.995549,0.986078,35,75650,0.929502,AMP
7,cnn,41,0.949507,0.857991,0.985834,0.917480,0.885769,0.993197,0.979244,55,75650,0.651002,AMP
8,cnn,55,0.974305,0.945988,0.964848,0.955325,0.937384,0.996207,0.985853,45,75650,0.385763,AMP
9,cnn,69,0.937407,0.829128,0.982686,0.899400,0.860940,0.986759,0.964677,56,75650,0.449681,AMP



DAPT2020 neural-baseline mean results:


,accuracy,precision,recall,f1,mcc,auroc,auprc,parameters,runtime_minutes
method,,,,,,,,,
transformer,0.986137,0.975116,0.976285,0.975673,0.966006,0.998678,0.997397,275202.0,0.746498
bilstm,0.978309,0.951685,0.973347,0.962354,0.947281,0.997913,0.995322,207618.0,0.735771
selective_ssm,0.968300,0.914419,0.980693,0.946312,0.925090,0.995043,0.984505,62754.0,1.417724
kan,0.965013,0.902562,0.983421,0.941229,0.918128,0.993306,0.979159,94618.0,0.883112
cnn,0.958918,0.889634,0.979958,0.931980,0.905507,0.993508,0.979948,75650.0,0.601853
mlp,0.954048,0.875708,0.977335,0.923721,0.893748,0.989375,0.970522,170626.0,0.397440


In [79]:
# ============================================================
# Baseline Experiment — Cell 5
# Official DAPT2020 Random Forest baseline: five seeds
# ============================================================

from pathlib import Path
import time
import json
import joblib
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestClassifier
)

# ------------------------------------------------------------
# 1. Sequence pooling for Random Forest
#
# Each sequence is represented using:
#   mean, standard deviation, minimum, and maximum
# for every traffic feature.
# ------------------------------------------------------------

def pool_sequences_for_rf(
    sequence_x,
):

    sequence_mean = sequence_x.mean(
        axis=1
    )

    sequence_std = sequence_x.std(
        axis=1
    )

    sequence_minimum = sequence_x.min(
        axis=1
    )

    sequence_maximum = sequence_x.max(
        axis=1
    )

    pooled_features = np.concatenate(
        [
            sequence_mean,
            sequence_std,
            sequence_minimum,
            sequence_maximum,
        ],
        axis=1,
    ).astype(np.float32)

    assert np.isfinite(
        pooled_features
    ).all()

    return pooled_features


# ------------------------------------------------------------
# 2. Prepare pooled official DAPT2020 partitions
# ------------------------------------------------------------

print(
    "Pooling official DAPT2020 "
    "sequences for Random Forest..."
)

dapt_rf_arrays = {}

for split_name in [
    "train",
    "val",
    "test",
]:

    sequence_x, sequence_y = (
        baseline_dapt_arrays[
            split_name
        ]
    )

    pooled_x = (
        pool_sequences_for_rf(
            sequence_x
        )
    )

    dapt_rf_arrays[
        split_name
    ] = (
        pooled_x,
        sequence_y.copy(),
    )

    print(
        split_name,
        pooled_x.shape,
        sequence_y.shape,
    )

assert dapt_rf_arrays[
    "train"
][0].shape == (
    29_147,
    268,
)

assert dapt_rf_arrays[
    "val"
][0].shape == (
    6_067,
    268,
)

assert dapt_rf_arrays[
    "test"
][0].shape == (
    6_694,
    268,
)

# ------------------------------------------------------------
# 3. Train one Random Forest seed
# ------------------------------------------------------------

def train_random_forest_seed(
    dataset_name,
    seed,
    pooled_arrays,
    experiment_root,
):

    run_directory = (
        Path(experiment_root)
        / dataset_name
        / "rf"
        / f"seed_{seed}"
    )

    run_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = (
        run_directory
        / "test_metrics.json"
    )

    if metrics_path.exists():

        with open(
            metrics_path,
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    train_x, train_y = (
        pooled_arrays["train"]
    )

    val_x, val_y = (
        pooled_arrays["val"]
    )

    test_x, test_y = (
        pooled_arrays["test"]
    )

    classifier = (
        RandomForestClassifier(
            n_estimators=500,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            bootstrap=True,
            class_weight=(
                "balanced_subsample"
            ),
            n_jobs=-1,
            random_state=seed,
        )
    )

    training_start = (
        time.perf_counter()
    )

    classifier.fit(
        train_x,
        train_y,
    )

    training_runtime = (
        time.perf_counter()
        - training_start
    )

    validation_probability = (
        classifier.predict_proba(
            val_x
        )[:, 1]
    )

    threshold = select_threshold(
        val_y,
        validation_probability,
    )

    test_probability = (
        classifier.predict_proba(
            test_x
        )[:, 1]
    )

    metrics = compute_metrics(
        test_y,
        test_probability,
        threshold,
    )

    total_tree_nodes = int(
        sum(
            estimator.tree_.node_count
            for estimator
            in classifier.estimators_
        )
    )

    total_tree_leaves = int(
        sum(
            estimator.tree_.n_leaves
            for estimator
            in classifier.estimators_
        )
    )

    metrics.update({
        "dataset": dataset_name,
        "method": "rf",
        "seed": seed,
        "best_epoch": np.nan,
        "epochs_completed": np.nan,
        # RF does not have neural trainable parameters
        "parameters": np.nan,
        "number_of_trees": 500,
        "total_tree_nodes": (
            total_tree_nodes
        ),
        "total_tree_leaves": (
            total_tree_leaves
        ),
        "pooled_feature_dimensions": (
            train_x.shape[1]
        ),
        "runtime_seconds": (
            training_runtime
        ),
        "runtime_minutes": (
            training_runtime
            / 60.0
        ),
        "numerical_precision": (
            "FP32 / scikit-learn"
        ),
    })

    pd.DataFrame({
        "label": test_y,
        "probability_attack": (
            test_probability
        ),
    }).to_csv(
        run_directory
        / "test_predictions.csv",
        index=False,
    )

    save_baseline_json(
        metrics,
        metrics_path,
    )

    joblib.dump(
        classifier,
        run_directory
        / "random_forest.joblib",
        compress=3,
    )

    del classifier

    gc.collect()

    return metrics


# ------------------------------------------------------------
# 4. Official five-seed Random Forest experiment
# ------------------------------------------------------------

dapt_rf_rows = []

for seed in official_baseline_seeds:

    print(
        "\nTraining DAPT2020 "
        f"Random Forest, seed {seed}"
    )

    rf_metrics = (
        train_random_forest_seed(
            dataset_name="DAPT2020",
            seed=seed,
            pooled_arrays=(
                dapt_rf_arrays
            ),
            experiment_root=(
                dapt_baseline_output
            ),
        )
    )

    dapt_rf_rows.append(
        rf_metrics
    )

    print({
        "seed": seed,
        "accuracy": (
            rf_metrics["accuracy"]
        ),
        "precision": (
            rf_metrics["precision"]
        ),
        "recall": (
            rf_metrics["recall"]
        ),
        "f1": rf_metrics["f1"],
        "mcc": rf_metrics["mcc"],
        "auroc": (
            rf_metrics["auroc"]
        ),
        "auprc": (
            rf_metrics["auprc"]
        ),
        "runtime_minutes": (
            rf_metrics[
                "runtime_minutes"
            ]
        ),
    })

dapt_rf_results = pd.DataFrame(
    dapt_rf_rows
)

assert len(dapt_rf_results) == 5

dapt_rf_results.to_csv(
    dapt_baseline_output
    / "DAPT2020"
    / "rf_seed_results.csv",
    index=False,
)

print(
    "\nDAPT2020 Random Forest "
    "seed-level results:"
)

display(
    dapt_rf_results[
        [
            "seed",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "number_of_trees",
            "total_tree_nodes",
            "runtime_minutes",
        ]
    ]
)

print(
    "\nDAPT2020 Random Forest "
    "mean ± standard deviation:"
)

rf_reported_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "mcc",
    "auroc",
    "auprc",
    "runtime_minutes",
]

dapt_rf_summary = pd.DataFrame({
    "mean": (
        dapt_rf_results[
            rf_reported_metrics
        ].mean()
    ),
    "std": (
        dapt_rf_results[
            rf_reported_metrics
        ].std(ddof=1)
    ),
})

display(dapt_rf_summary)

Pooling official DAPT2020 sequences for Random Forest...
train (29147, 268) (29147,)
val (6067, 268) (6067,)
test (6694, 268) (6694,)

Training DAPT2020 Random Forest, seed 13
{'seed': 13, 'accuracy': 0.9898416492381237, 'precision': 0.9781477627471384, 'recall': 0.9863588667366212, 'f1': 0.9822361546499477, 'mcc': 0.97514006295581, 'auroc': 0.9987572770681513, 'auprc': 0.9977595998491379, 'runtime_minutes': 0.053653063333573905}

Training DAPT2020 Random Forest, seed 27
{'seed': 27, 'accuracy': 0.9899910367493278, 'precision': 0.9821709491347667, 'recall': 0.9826862539349422, 'f1': 0.982428533962759, 'mcc': 0.9754312111840958, 'auroc': 0.9988558971756079, 'auprc': 0.9977612120392267, 'runtime_minutes': 0.053700560000046}

Training DAPT2020 Random Forest, seed 41
{'seed': 41, 'accuracy': 0.9895428742157155, 'precision': 0.9786235662148071, 'recall': 0.9847848898216159, 'f1': 0.981694560669456, 'mcc': 0.9743849493111567, 'auroc': 0.9988903046353205, 'auprc': 0.9978234839711505, 'runtime

,seed,accuracy,precision,recall,f1,mcc,auroc,auprc,number_of_trees,total_tree_nodes,runtime_minutes
0,13,0.989842,0.978148,0.986359,0.982236,0.975140,0.998757,0.997760,500,471624,0.053653
1,27,0.989991,0.982171,0.982686,0.982429,0.975431,0.998856,0.997761,500,468356,0.053701
2,41,0.989543,0.978624,0.984785,0.981695,0.974385,0.998890,0.997823,500,470030,0.055284
3,55,0.990439,0.987302,0.979014,0.983140,0.976485,0.998884,0.997817,500,472520,0.077453
4,69,0.990439,0.983211,0.983211,0.983211,0.976528,0.998704,0.997677,500,468468,0.079240



DAPT2020 Random Forest mean ± standard deviation:


,mean,std
accuracy,0.990051,0.000390
precision,0.981891,0.003734
recall,0.983211,0.002751
f1,0.982542,0.000638
mcc,0.975594,0.000916
auroc,0.998818,0.000083
auprc,0.997768,0.000059
runtime_minutes,0.063866,0.013250


In [80]:
# ============================================================
# Baseline Experiment — Cell 6
# Official CICIDS2017 neural baselines: six models × five seeds
# ============================================================

from dataclasses import replace
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import json
import gc

# ------------------------------------------------------------
# 1. Official CICIDS2017 baseline configuration
# ------------------------------------------------------------

cicids_baseline_output = (
    PROJECT_ROOT
    / "outputs"
    / "baseline_experiments"
)

cicids_official_baseline_cfg = replace(
    cicids_cfg,
    output_dir=str(
        cicids_baseline_output
        / "CICIDS2017"
    ),
    epochs=60,
    patience=10,
    seeds=official_baseline_seeds,
)

print(
    "Official baseline output:",
    cicids_baseline_output
)

print("Dataset: CICIDS2017")

print(
    "Neural baselines:",
    neural_baseline_names
)

print(
    "Seeds:",
    official_baseline_seeds
)

print(
    "Epoch limit:",
    cicids_official_baseline_cfg.epochs
)

print(
    "Patience:",
    cicids_official_baseline_cfg.patience
)

print(
    "Device:",
    baseline_device
)

# ------------------------------------------------------------
# 2. Verify exact official arrays
# ------------------------------------------------------------

assert baseline_cicids_arrays[
    "train"
][0].shape == (
    243_814,
    8,
    72,
)

assert baseline_cicids_arrays[
    "val"
][0].shape == (
    54_458,
    8,
    72,
)

assert baseline_cicids_arrays[
    "test"
][0].shape == (
    54_499,
    8,
    72,
)

assert int(
    (
        baseline_cicids_arrays[
            "test"
        ][1] == 1
    ).sum()
) == 12_534

print(
    "Official CICIDS2017 arrays verified."
)

# ------------------------------------------------------------
# 3. Train with resume and FP32 fallback
# ------------------------------------------------------------

cicids_neural_baseline_rows = []

for baseline_name in (
    neural_baseline_names
):

    for seed in (
        official_baseline_seeds
    ):

        run_directory = (
            cicids_baseline_output
            / "CICIDS2017"
            / baseline_name
            / f"seed_{seed}"
        )

        metrics_path = (
            run_directory
            / "test_metrics.json"
        )

        # ----------------------------------------------------
        # Resume completed run
        # ----------------------------------------------------

        if metrics_path.exists():

            with open(
                metrics_path,
                "r",
                encoding="utf-8",
            ) as file:
                metrics = json.load(file)

            cicids_neural_baseline_rows.append(
                metrics
            )

            print(
                "Loaded completed run:",
                baseline_name,
                seed,
            )

            continue

        print(
            "\nTraining official CICIDS2017 "
            f"baseline: {baseline_name}, "
            f"seed {seed}"
        )

        # ----------------------------------------------------
        # First attempt: AMP
        # ----------------------------------------------------

        try:

            metrics = (
                train_neural_baseline_seed(
                    cfg=(
                        cicids_official_baseline_cfg
                    ),
                    dataset_name=(
                        "CICIDS2017"
                    ),
                    baseline_name=(
                        baseline_name
                    ),
                    seed=seed,
                    arrays=(
                        baseline_cicids_arrays
                    ),
                    device=baseline_device,
                    experiment_root=(
                        cicids_baseline_output
                    ),
                )
            )

        except (
            ValueError,
            RuntimeError,
        ) as error:

            if not is_baseline_nonfinite_error(
                error
            ):
                raise

            print(
                "Non-finite values detected "
                f"for {baseline_name}, "
                f"seed {seed}."
            )

            print(
                "Retrying this run in FP32."
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()

            fp32_cfg = replace(
                cicids_official_baseline_cfg,
                use_amp=False,
            )

            metrics = (
                train_neural_baseline_seed(
                    cfg=fp32_cfg,
                    dataset_name=(
                        "CICIDS2017"
                    ),
                    baseline_name=(
                        baseline_name
                    ),
                    seed=seed,
                    arrays=(
                        baseline_cicids_arrays
                    ),
                    device=baseline_device,
                    experiment_root=(
                        cicids_baseline_output
                    ),
                )
            )

            metrics[
                "numerical_precision"
            ] = "FP32 fallback"

            with open(
                metrics_path,
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    metrics,
                    file,
                    indent=2,
                )

        # ----------------------------------------------------
        # Verify metrics
        # ----------------------------------------------------

        for metric_name in [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "mcc",
            "auroc",
            "auprc",
        ]:

            assert np.isfinite(
                metrics[metric_name]
            ), (
                f"Non-finite {metric_name} "
                f"for {baseline_name}, "
                f"seed {seed}."
            )

        cicids_neural_baseline_rows.append(
            metrics
        )

        print(
            "\nCompleted result:",
            {
                "method": baseline_name,
                "seed": seed,
                "accuracy": (
                    metrics["accuracy"]
                ),
                "precision": (
                    metrics["precision"]
                ),
                "recall": (
                    metrics["recall"]
                ),
                "f1": metrics["f1"],
                "mcc": metrics["mcc"],
                "auroc": (
                    metrics["auroc"]
                ),
                "auprc": (
                    metrics["auprc"]
                ),
                "best_epoch": (
                    metrics["best_epoch"]
                ),
                "runtime_minutes": (
                    metrics[
                        "runtime_minutes"
                    ]
                ),
                "precision_mode": (
                    metrics[
                        "numerical_precision"
                    ]
                ),
            },
        )

        # Save progress after every run
        pd.DataFrame(
            cicids_neural_baseline_rows
        ).to_csv(
            cicids_baseline_output
            / "CICIDS2017"
            / (
                "neural_baseline_"
                "seed_results_partial.csv"
            ),
            index=False,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Final seed-level results
# ------------------------------------------------------------

cicids_neural_baseline_results = (
    pd.DataFrame(
        cicids_neural_baseline_rows
    )
)

expected_neural_runs = (
    len(neural_baseline_names)
    * len(official_baseline_seeds)
)

assert len(
    cicids_neural_baseline_results
) == expected_neural_runs, (
    f"Expected {expected_neural_runs} "
    "CICIDS2017 neural runs, found "
    f"{len(cicids_neural_baseline_results)}."
)

cicids_neural_baseline_results.to_csv(
    cicids_baseline_output
    / "CICIDS2017"
    / "neural_baseline_seed_results.csv",
    index=False,
)

print(
    "\nAll official CICIDS2017 "
    "neural baselines completed."
)

display(
    cicids_neural_baseline_results[
        [
            "method",
            "seed",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "best_epoch",
            "parameters",
            "runtime_minutes",
            "numerical_precision",
        ]
    ]
)

# ------------------------------------------------------------
# 5. Preliminary mean results
# ------------------------------------------------------------

cicids_neural_baseline_means = (
    cicids_neural_baseline_results
    .groupby("method")[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "parameters",
            "runtime_minutes",
        ]
    ]
    .mean()
    .sort_values(
        by="f1",
        ascending=False,
    )
)

print(
    "\nCICIDS2017 neural-baseline "
    "mean results:"
)

display(
    cicids_neural_baseline_means
)

Official baseline output: <PROJECT_ROOT>\outputs\baseline_experiments
Dataset: CICIDS2017
Neural baselines: ('mlp', 'cnn', 'bilstm', 'transformer', 'kan', 'selective_ssm')
Seeds: (13, 27, 41, 55, 69)
Epoch limit: 60
Patience: 10
Device: cuda
Official CICIDS2017 arrays verified.

Training official CICIDS2017 baseline: mlp, seed 13
{'dataset': 'CICIDS2017', 'baseline': 'mlp', 'seed': 13, 'epoch': 1, 'train_loss': 0.154761, 'val_auprc': 0.960365, 'val_f1': 0.907229, 'seconds': 3.35}
{'dataset': 'CICIDS2017', 'baseline': 'mlp', 'seed': 13, 'epoch': 2, 'train_loss': 0.106345, 'val_auprc': 0.988363, 'val_f1': 0.948188, 'seconds': 3.2}
{'dataset': 'CICIDS2017', 'baseline': 'mlp', 'seed': 13, 'epoch': 3, 'train_loss': 0.096384, 'val_auprc': 0.989394, 'val_f1': 0.953099, 'seconds': 3.07}
{'dataset': 'CICIDS2017', 'baseline': 'mlp', 'seed': 13, 'epoch': 4, 'train_loss': 0.087989, 'val_auprc': 0.990317, 'val_f1': 0.954964, 'seconds': 3.22}
{'dataset': 'CICIDS2017', 'baseline': 'mlp', 'seed': 13, 

<CONDA_ROOT>\envs\hlcda\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 13, 'epoch': 1, 'train_loss': 0.092329, 'val_auprc': 0.950818, 'val_f1': 0.91442, 'seconds': 7.16}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 13, 'epoch': 2, 'train_loss': 0.061967, 'val_auprc': 0.981971, 'val_f1': 0.929687, 'seconds': 7.61}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 13, 'epoch': 3, 'train_loss': 0.054449, 'val_auprc': 0.994324, 'val_f1': 0.972584, 'seconds': 7.47}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 13, 'epoch': 4, 'train_loss': 0.049556, 'val_auprc': 0.92803, 'val_f1': 0.892985, 'seconds': 7.65}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 13, 'epoch': 5, 'train_loss': 0.045696, 'val_auprc': 0.99555, 'val_f1': 0.971529, 'seconds': 8.03}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 13, 'epoch': 6, 'train_loss': 0.043075, 'val_auprc': 0.996117, 'val_f1': 0.974683, 'seconds': 7.57}
{'dataset': 'CICIDS2017', 'baseline':

<CONDA_ROOT>\envs\hlcda\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 27, 'epoch': 1, 'train_loss': 0.090257, 'val_auprc': 0.989996, 'val_f1': 0.963321, 'seconds': 7.45}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 27, 'epoch': 2, 'train_loss': 0.061867, 'val_auprc': 0.970227, 'val_f1': 0.902642, 'seconds': 7.78}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 27, 'epoch': 3, 'train_loss': 0.053982, 'val_auprc': 0.994755, 'val_f1': 0.973621, 'seconds': 7.41}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 27, 'epoch': 4, 'train_loss': 0.049101, 'val_auprc': 0.994138, 'val_f1': 0.970074, 'seconds': 7.76}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 27, 'epoch': 5, 'train_loss': 0.046328, 'val_auprc': 0.997224, 'val_f1': 0.980554, 'seconds': 7.29}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 27, 'epoch': 6, 'train_loss': 0.042988, 'val_auprc': 0.997546, 'val_f1': 0.982654, 'seconds': 7.43}
{'dataset': 'CICIDS2017', 'baselin

...

<CONDA_ROOT>\envs\hlcda\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 69, 'epoch': 1, 'train_loss': 0.090779, 'val_auprc': 0.879214, 'val_f1': 0.847793, 'seconds': 7.53}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 69, 'epoch': 2, 'train_loss': 0.062083, 'val_auprc': 0.924707, 'val_f1': 0.874482, 'seconds': 7.46}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 69, 'epoch': 3, 'train_loss': 0.05416, 'val_auprc': 0.983284, 'val_f1': 0.941583, 'seconds': 7.37}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 69, 'epoch': 4, 'train_loss': 0.049309, 'val_auprc': 0.977435, 'val_f1': 0.939553, 'seconds': 7.98}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 69, 'epoch': 5, 'train_loss': 0.046732, 'val_auprc': 0.986869, 'val_f1': 0.954545, 'seconds': 7.49}
{'dataset': 'CICIDS2017', 'baseline': 'transformer', 'seed': 69, 'epoch': 6, 'train_loss': 0.043288, 'val_auprc': 0.988599, 'val_f1': 0.958661, 'seconds': 7.31}
{'dataset': 'CICIDS2017', 'baseline

,method,seed,accuracy,precision,recall,f1,mcc,auroc,auprc,best_epoch,parameters,runtime_minutes,numerical_precision
0,mlp,13,0.912806,0.987228,0.629009,0.768421,0.745611,0.993713,0.979353,4,180866,0.746372,AMP
1,mlp,27,0.903650,0.995779,0.583533,0.735852,0.718205,0.993698,0.980199,3,180866,0.686526,AMP
2,mlp,41,0.910659,0.991914,0.616563,0.760443,0.739460,0.992833,0.977224,4,180866,0.749594,AMP
3,mlp,55,0.926531,0.983231,0.692357,0.812547,0.786844,0.993909,0.979996,4,180866,0.739892,AMP
4,mlp,69,0.897484,0.996143,0.556407,0.714001,0.698973,0.994036,0.981054,4,180866,0.737371,AMP
5,cnn,13,0.952935,0.987482,0.805569,0.887297,0.865043,0.997450,0.991667,52,77570,3.743814,AMP
6,cnn,27,0.957999,0.993925,0.822403,0.900065,0.880060,0.998354,0.994806,23,77570,2.236287,AMP
7,cnn,41,0.951210,0.990756,0.795277,0.882319,0.860195,0.997829,0.993037,36,77570,3.033010,AMP
8,cnn,55,0.966862,0.992834,0.862135,0.922880,0.905554,0.998495,0.995008,54,77570,3.940284,AMP
9,cnn,69,0.928641,0.989802,0.696904,0.817922,0.793654,0.997237,0.990684,27,77570,2.502882,AMP



CICIDS2017 neural-baseline mean results:


,accuracy,precision,recall,f1,mcc,auroc,auprc,parameters,runtime_minutes
method,,,,,,,,,
transformer,0.977640,0.993103,0.909079,0.949215,0.936433,0.998621,0.996040,275842.0,5.455005
bilstm,0.955874,0.993749,0.813308,0.893841,0.873795,0.998380,0.994861,208258.0,4.230020
cnn,0.951529,0.990960,0.796458,0.882097,0.860901,0.997873,0.993040,77570.0,3.091256
selective_ssm,0.924035,0.985573,0.679895,0.803898,0.779437,0.993480,0.978736,63394.0,2.287612
mlp,0.910226,0.990859,0.615574,0.758253,0.737819,0.993638,0.979565,180866.0,0.731951
kan,0.884710,0.912719,0.553678,0.686390,0.651948,0.979284,0.920067,100418.0,0.886136


In [81]:
# ============================================================
# Baseline Experiment — Cell 7
# Official CICIDS2017 Random Forest baseline: five seeds
# ============================================================

from pathlib import Path
import time
import json
import joblib
import gc

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Pool official CICIDS2017 sequences
# ------------------------------------------------------------

print(
    "Pooling official CICIDS2017 "
    "sequences for Random Forest..."
)

cicids_rf_arrays = {}

for split_name in [
    "train",
    "val",
    "test",
]:

    sequence_x, sequence_y = (
        baseline_cicids_arrays[
            split_name
        ]
    )

    pooled_x = (
        pool_sequences_for_rf(
            sequence_x
        )
    )

    cicids_rf_arrays[
        split_name
    ] = (
        pooled_x,
        sequence_y.copy(),
    )

    print(
        split_name,
        pooled_x.shape,
        sequence_y.shape,
    )

assert cicids_rf_arrays[
    "train"
][0].shape == (
    243_814,
    288,
)

assert cicids_rf_arrays[
    "val"
][0].shape == (
    54_458,
    288,
)

assert cicids_rf_arrays[
    "test"
][0].shape == (
    54_499,
    288,
)

# ------------------------------------------------------------
# 2. CICIDS2017 Random Forest training function
#
# Only the representative seed-13 model is retained to avoid
# saving five potentially large forest files.
# ------------------------------------------------------------

def train_cicids_random_forest_seed(
    seed,
    pooled_arrays,
    experiment_root,
    save_model=False,
):

    run_directory = (
        Path(experiment_root)
        / "CICIDS2017"
        / "rf"
        / f"seed_{seed}"
    )

    run_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = (
        run_directory
        / "test_metrics.json"
    )

    # Resume completed result
    if metrics_path.exists():

        with open(
            metrics_path,
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    train_x, train_y = (
        pooled_arrays["train"]
    )

    val_x, val_y = (
        pooled_arrays["val"]
    )

    test_x, test_y = (
        pooled_arrays["test"]
    )

    classifier = (
        RandomForestClassifier(
            n_estimators=500,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            bootstrap=True,
            class_weight=(
                "balanced_subsample"
            ),
            n_jobs=-1,
            random_state=seed,
        )
    )

    training_start = (
        time.perf_counter()
    )

    classifier.fit(
        train_x,
        train_y,
    )

    training_runtime = (
        time.perf_counter()
        - training_start
    )

    validation_probability = (
        classifier.predict_proba(
            val_x
        )[:, 1]
    )

    threshold = select_threshold(
        val_y,
        validation_probability,
    )

    test_probability = (
        classifier.predict_proba(
            test_x
        )[:, 1]
    )

    metrics = compute_metrics(
        test_y,
        test_probability,
        threshold,
    )

    total_tree_nodes = int(
        sum(
            estimator.tree_.node_count
            for estimator
            in classifier.estimators_
        )
    )

    total_tree_leaves = int(
        sum(
            estimator.tree_.n_leaves
            for estimator
            in classifier.estimators_
        )
    )

    metrics.update({
        "dataset": "CICIDS2017",
        "method": "rf",
        "seed": seed,
        "best_epoch": np.nan,
        "epochs_completed": np.nan,
        "parameters": np.nan,
        "number_of_trees": 500,
        "total_tree_nodes": (
            total_tree_nodes
        ),
        "total_tree_leaves": (
            total_tree_leaves
        ),
        "pooled_feature_dimensions": (
            train_x.shape[1]
        ),
        "runtime_seconds": (
            training_runtime
        ),
        "runtime_minutes": (
            training_runtime
            / 60.0
        ),
        "numerical_precision": (
            "FP32 / scikit-learn"
        ),
    })

    pd.DataFrame({
        "label": test_y,
        "probability_attack": (
            test_probability
        ),
    }).to_csv(
        run_directory
        / "test_predictions.csv",
        index=False,
    )

    save_baseline_json(
        metrics,
        metrics_path,
    )

    # Retain only the predetermined representative model
    if save_model:

        joblib.dump(
            classifier,
            run_directory
            / "random_forest.joblib",
            compress=3,
        )

    del classifier

    gc.collect()

    return metrics


# ------------------------------------------------------------
# 3. Official five-seed CICIDS2017 RF evaluation
# ------------------------------------------------------------

cicids_rf_rows = []

for seed in official_baseline_seeds:

    print(
        "\nTraining CICIDS2017 "
        f"Random Forest, seed {seed}"
    )

    rf_metrics = (
        train_cicids_random_forest_seed(
            seed=seed,
            pooled_arrays=(
                cicids_rf_arrays
            ),
            experiment_root=(
                cicids_baseline_output
            ),
            save_model=(seed == 13),
        )
    )

    cicids_rf_rows.append(
        rf_metrics
    )

    print({
        "seed": seed,
        "accuracy": (
            rf_metrics["accuracy"]
        ),
        "precision": (
            rf_metrics["precision"]
        ),
        "recall": (
            rf_metrics["recall"]
        ),
        "f1": rf_metrics["f1"],
        "mcc": rf_metrics["mcc"],
        "auroc": (
            rf_metrics["auroc"]
        ),
        "auprc": (
            rf_metrics["auprc"]
        ),
        "runtime_minutes": (
            rf_metrics[
                "runtime_minutes"
            ]
        ),
    })

    # Save partial progress
    pd.DataFrame(
        cicids_rf_rows
    ).to_csv(
        cicids_baseline_output
        / "CICIDS2017"
        / "rf_seed_results_partial.csv",
        index=False,
    )

cicids_rf_results = pd.DataFrame(
    cicids_rf_rows
)

assert len(cicids_rf_results) == 5

cicids_rf_results.to_csv(
    cicids_baseline_output
    / "CICIDS2017"
    / "rf_seed_results.csv",
    index=False,
)

print(
    "\nCICIDS2017 Random Forest "
    "seed-level results:"
)

display(
    cicids_rf_results[
        [
            "seed",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
            "number_of_trees",
            "total_tree_nodes",
            "runtime_minutes",
        ]
    ]
)

print(
    "\nCICIDS2017 Random Forest "
    "mean ± standard deviation:"
)

cicids_rf_summary = pd.DataFrame({
    "mean": (
        cicids_rf_results[
            rf_reported_metrics
        ].mean()
    ),
    "std": (
        cicids_rf_results[
            rf_reported_metrics
        ].std(ddof=1)
    ),
})

display(cicids_rf_summary)

# Release pooled RF arrays after completion
del cicids_rf_arrays

gc.collect()

print(
    "\nOfficial CICIDS2017 "
    "Random Forest completed."
)

Pooling official CICIDS2017 sequences for Random Forest...
train (243814, 288) (243814,)
val (54458, 288) (54458,)
test (54499, 288) (54499,)

Training CICIDS2017 Random Forest, seed 13
{'seed': 13, 'accuracy': 0.9969357235912586, 'precision': 0.9963875732519868, 'recall': 0.99026647518749, 'f1': 0.9933175943339603, 'mcc': 0.9913376108106782, 'auroc': 0.9993087530619206, 'auprc': 0.998626681757973, 'runtime_minutes': 1.2718003049997302}

Training CICIDS2017 Random Forest, seed 27
{'seed': 27, 'accuracy': 0.9967705829464761, 'precision': 0.9954297626683771, 'recall': 0.9905058241582895, 'f1': 0.9929616891945933, 'mcc': 0.9908713973718124, 'auroc': 0.999337461820279, 'auprc': 0.9986441558276747, 'runtime_minutes': 1.37628484833355}

Training CICIDS2017 Random Forest, seed 41
{'seed': 41, 'accuracy': 0.9968439787886016, 'precision': 0.9965456298200515, 'recall': 0.9897079942556247, 'f1': 0.9931150428308382, 'mcc': 0.9910777291983063, 'auroc': 0.9993204880912885, 'auprc': 0.998661348924909

,seed,accuracy,precision,recall,f1,mcc,auroc,auprc,number_of_trees,total_tree_nodes,runtime_minutes
0,13,0.996936,0.996388,0.990266,0.993318,0.991338,0.999309,0.998627,500,2606718,1.271800
1,27,0.996771,0.995430,0.990506,0.992962,0.990871,0.999337,0.998644,500,2607908,1.376285
2,41,0.996844,0.996546,0.989708,0.993115,0.991078,0.999320,0.998661,500,2623226,1.379580
3,55,0.996752,0.996066,0.989788,0.992917,0.990818,0.999240,0.998580,500,2618932,2.251434
4,69,0.996771,0.996385,0.989548,0.992955,0.990870,0.999250,0.998588,500,2619902,2.229855



CICIDS2017 Random Forest mean ± standard deviation:


,mean,std
accuracy,0.996815,0.000076
precision,0.996163,0.000445
recall,0.989963,0.000404
f1,0.993053,0.000166
mcc,0.990995,0.000216
auroc,0.999291,0.000044
auprc,0.998620,0.000035
runtime_minutes,1.701791,0.493869



Official CICIDS2017 Random Forest completed.


In [82]:
# ============================================================
# Baseline Experiment — Cell 8
# Final aggregation, runtime calculation, and LaTeX-ready data
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Output directories
# ------------------------------------------------------------

baseline_results_root = (
    PROJECT_ROOT
    / "outputs"
    / "baseline_experiments"
)

final_comparison_output = (
    PROJECT_ROOT
    / "outputs"
    / "final_comparison"
)

final_comparison_output.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 2. Load all 70 baseline runs
# ------------------------------------------------------------

baseline_result_paths = {
    "DAPT2020 neural": (
        baseline_results_root
        / "DAPT2020"
        / "neural_baseline_seed_results.csv"
    ),
    "DAPT2020 RF": (
        baseline_results_root
        / "DAPT2020"
        / "rf_seed_results.csv"
    ),
    "CICIDS2017 neural": (
        baseline_results_root
        / "CICIDS2017"
        / "neural_baseline_seed_results.csv"
    ),
    "CICIDS2017 RF": (
        baseline_results_root
        / "CICIDS2017"
        / "rf_seed_results.csv"
    ),
}

for result_name, result_path in (
    baseline_result_paths.items()
):

    assert result_path.exists(), (
        f"Missing {result_name}: "
        f"{result_path}"
    )

dapt_neural_rows = pd.read_csv(
    baseline_result_paths[
        "DAPT2020 neural"
    ]
)

dapt_rf_rows = pd.read_csv(
    baseline_result_paths[
        "DAPT2020 RF"
    ]
)

cicids_neural_rows = pd.read_csv(
    baseline_result_paths[
        "CICIDS2017 neural"
    ]
)

cicids_rf_rows = pd.read_csv(
    baseline_result_paths[
        "CICIDS2017 RF"
    ]
)

baseline_seed_results = pd.concat(
    [
        dapt_neural_rows,
        dapt_rf_rows,
        cicids_neural_rows,
        cicids_rf_rows,
    ],
    ignore_index=True,
    sort=False,
)

baseline_seed_results[
    "seed"
] = baseline_seed_results[
    "seed"
].astype(int)

assert len(
    baseline_seed_results
) == 70

assert (
    baseline_seed_results
    .groupby(
        [
            "dataset",
            "method",
        ]
    )
    .size()
    .eq(5)
    .all()
)

print(
    "Verified baseline runs:",
    len(baseline_seed_results)
)

# ------------------------------------------------------------
# 3. Load ten full KAMBA++ runs
# ------------------------------------------------------------

full_model_paths = {
    "DAPT2020": (
        PROJECT_ROOT
        / "outputs"
        / "DAPT2020"
        / "all_seed_results.csv"
    ),
    "CICIDS2017": (
        PROJECT_ROOT
        / "outputs"
        / "CICIDS2017"
        / "all_seed_results.csv"
    ),
}

full_model_frames = []

for dataset_name, result_path in (
    full_model_paths.items()
):

    assert result_path.exists(), (
        f"Missing full-model results: "
        f"{result_path}"
    )

    current_frame = pd.read_csv(
        result_path
    )

    current_frame["dataset"] = (
        dataset_name
    )

    current_frame["method"] = (
        "kamba++"
    )

    current_frame["seed"] = (
        current_frame["seed"]
        .astype(int)
    )

    assert len(current_frame) == 5

    full_model_frames.append(
        current_frame
    )

full_model_seed_results = pd.concat(
    full_model_frames,
    ignore_index=True,
    sort=False,
)

assert len(
    full_model_seed_results
) == 10

# ------------------------------------------------------------
# 4. Combine baseline and KAMBA++ results
# ------------------------------------------------------------

comparison_seed_results = pd.concat(
    [
        baseline_seed_results,
        full_model_seed_results,
    ],
    ignore_index=True,
    sort=False,
)

method_order = [
    "rf",
    "mlp",
    "cnn",
    "bilstm",
    "transformer",
    "kan",
    "selective_ssm",
    "kamba++",
]

method_display_names = {
    "rf": "RF",
    "mlp": "MLP",
    "cnn": "CNN",
    "bilstm": "BiLSTM",
    "transformer": "Transformer",
    "kan": "KAN",
    "selective_ssm": (
        "Selective SSM"
    ),
    "kamba++": "KAMBA++",
}

comparison_seed_results[
    "method"
] = pd.Categorical(
    comparison_seed_results[
        "method"
    ],
    categories=method_order,
    ordered=True,
)

comparison_seed_results = (
    comparison_seed_results
    .sort_values(
        by=[
            "dataset",
            "method",
            "seed",
        ]
    )
    .reset_index(drop=True)
)

assert len(
    comparison_seed_results
) == 80

assert (
    comparison_seed_results
    .groupby(
        [
            "dataset",
            "method",
        ],
        observed=False,
    )
    .size()
    .eq(5)
    .all()
)

# ------------------------------------------------------------
# 5. Calculate mean and standard deviation
# ------------------------------------------------------------

comparison_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "mcc",
    "auroc",
    "auprc",
]

comparison_summary_rows = []

for dataset_name in [
    "DAPT2020",
    "CICIDS2017",
]:

    for method_name in method_order:

        current_rows = (
            comparison_seed_results[
                (
                    comparison_seed_results[
                        "dataset"
                    ] == dataset_name
                )
                & (
                    comparison_seed_results[
                        "method"
                    ].astype(str)
                    == method_name
                )
            ]
        )

        assert len(current_rows) == 5

        summary_row = {
            "dataset": dataset_name,
            "method": method_name,
            "display_name": (
                method_display_names[
                    method_name
                ]
            ),
            "runs": len(current_rows),
        }

        for metric_name in (
            comparison_metrics
        ):

            summary_row[
                f"{metric_name}_mean"
            ] = current_rows[
                metric_name
            ].mean()

            summary_row[
                f"{metric_name}_std"
            ] = current_rows[
                metric_name
            ].std(ddof=1)

        if (
            current_rows[
                "parameters"
            ].notna().any()
        ):

            summary_row[
                "parameters"
            ] = current_rows[
                "parameters"
            ].mean()

        else:

            summary_row[
                "parameters"
            ] = np.nan

        comparison_summary_rows.append(
            summary_row
        )

comparison_mean_std = pd.DataFrame(
    comparison_summary_rows
)

# ------------------------------------------------------------
# 6. Create LaTeX-ready mean ± std values
# ------------------------------------------------------------

latex_comparison_rows = []

for _, row in (
    comparison_mean_std.iterrows()
):

    latex_row = {
        "dataset": row["dataset"],
        "method": row["display_name"],
    }

    for metric_name in [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "mcc",
        "auroc",
        "auprc",
    ]:

        latex_row[metric_name] = (
            f"{row[f'{metric_name}_mean']:.4f}"
            r"\(\pm\)"
            f"{row[f'{metric_name}_std']:.4f}"
        )

    if np.isfinite(
        row["parameters"]
    ):

        latex_row["parameters"] = int(
            round(row["parameters"])
        )

    else:

        latex_row["parameters"] = (
            "500 trees"
        )

    latex_comparison_rows.append(
        latex_row
    )

latex_comparison_table = pd.DataFrame(
    latex_comparison_rows
)

# ------------------------------------------------------------
# 7. Identify metric winners
# ------------------------------------------------------------

winner_rows = []

for dataset_name in [
    "DAPT2020",
    "CICIDS2017",
]:

    current_dataset = (
        comparison_mean_std[
            comparison_mean_std[
                "dataset"
            ] == dataset_name
        ]
    )

    for metric_name in [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "mcc",
        "auroc",
        "auprc",
    ]:

        mean_column = (
            f"{metric_name}_mean"
        )

        winner_index = (
            current_dataset[
                mean_column
            ].idxmax()
        )

        winning_row = (
            current_dataset.loc[
                winner_index
            ]
        )

        winner_rows.append({
            "dataset": dataset_name,
            "metric": metric_name,
            "winner": (
                winning_row[
                    "display_name"
                ]
            ),
            "mean": (
                winning_row[
                    mean_column
                ]
            ),
        })

metric_winners = pd.DataFrame(
    winner_rows
)

# ------------------------------------------------------------
# 8. Differences relative to KAMBA++
# ------------------------------------------------------------

difference_rows = []

for dataset_name in [
    "DAPT2020",
    "CICIDS2017",
]:

    dataset_summary = (
        comparison_mean_std[
            comparison_mean_std[
                "dataset"
            ] == dataset_name
        ]
    )

    kamba_row = (
        dataset_summary[
            dataset_summary[
                "method"
            ] == "kamba++"
        ].iloc[0]
    )

    for _, method_row in (
        dataset_summary.iterrows()
    ):

        if (
            method_row["method"]
            == "kamba++"
        ):
            continue

        difference_rows.append({
            "dataset": dataset_name,
            "method": (
                method_row[
                    "display_name"
                ]
            ),
            "accuracy_difference_pp": (
                100
                * (
                    method_row[
                        "accuracy_mean"
                    ]
                    - kamba_row[
                        "accuracy_mean"
                    ]
                )
            ),
            "f1_difference_pp": (
                100
                * (
                    method_row[
                        "f1_mean"
                    ]
                    - kamba_row[
                        "f1_mean"
                    ]
                )
            ),
            "mcc_difference_pp": (
                100
                * (
                    method_row[
                        "mcc_mean"
                    ]
                    - kamba_row[
                        "mcc_mean"
                    ]
                )
            ),
            "auprc_difference_pp": (
                100
                * (
                    method_row[
                        "auprc_mean"
                    ]
                    - kamba_row[
                        "auprc_mean"
                    ]
                )
            ),
        })

differences_from_kamba = pd.DataFrame(
    difference_rows
)

# ------------------------------------------------------------
# 9. Calculate baseline runtime
# ------------------------------------------------------------

baseline_runtime_by_dataset = (
    baseline_seed_results
    .groupby("dataset")[
        "runtime_minutes"
    ]
    .agg([
        "count",
        "sum",
        "mean",
        "std",
    ])
    .reset_index()
)

baseline_runtime_by_dataset[
    "runtime_hours"
] = (
    baseline_runtime_by_dataset[
        "sum"
    ]
    / 60.0
)

baseline_runtime_by_method = (
    baseline_seed_results
    .groupby(
        [
            "dataset",
            "method",
        ]
    )[
        "runtime_minutes"
    ]
    .agg([
        "count",
        "sum",
        "mean",
        "std",
    ])
    .reset_index()
)

# ------------------------------------------------------------
# 10. Verified previous experiment runtime
#
# These values were reconstructed from the saved official
# history/configuration timestamps before baseline training.
# ------------------------------------------------------------

previous_runtime = {
    "DAPT2020": {
        "full_runs": 5,
        "full_hours": 0.240402,
        "ablation_runs": 30,
        "ablation_hours": 1.405683,
    },
    "CICIDS2017": {
        "full_runs": 5,
        "full_hours": 1.378728,
        "ablation_runs": 30,
        "ablation_hours": 7.008504,
    },
}

runtime_rows = []

for dataset_name in [
    "DAPT2020",
    "CICIDS2017",
]:

    baseline_row = (
        baseline_runtime_by_dataset[
            baseline_runtime_by_dataset[
                "dataset"
            ] == dataset_name
        ].iloc[0]
    )

    baseline_hours = float(
        baseline_row[
            "runtime_hours"
        ]
    )

    full_hours = (
        previous_runtime[
            dataset_name
        ]["full_hours"]
    )

    ablation_hours = (
        previous_runtime[
            dataset_name
        ]["ablation_hours"]
    )

    total_hours = (
        baseline_hours
        + full_hours
        + ablation_hours
    )

    runtime_rows.extend([
        {
            "dataset": dataset_name,
            "experiment": (
                "Baseline comparison"
            ),
            "runs": 35,
            "runtime_hours": (
                baseline_hours
            ),
        },
        {
            "dataset": dataset_name,
            "experiment": (
                "Full KAMBA++ evaluation"
            ),
            "runs": 5,
            "runtime_hours": (
                full_hours
            ),
        },
        {
            "dataset": dataset_name,
            "experiment": (
                "Ablation study"
            ),
            "runs": 30,
            "runtime_hours": (
                ablation_hours
            ),
        },
        {
            "dataset": dataset_name,
            "experiment": "Dataset total",
            "runs": 70,
            "runtime_hours": (
                total_hours
            ),
        },
    ])

runtime_summary = pd.DataFrame(
    runtime_rows
)

overall_runtime_hours = (
    runtime_summary[
        runtime_summary[
            "experiment"
        ] == "Dataset total"
    ][
        "runtime_hours"
    ].sum()
)

overall_runtime_row = pd.DataFrame([
    {
        "dataset": "Overall",
        "experiment": (
            "Three experiments"
        ),
        "runs": 140,
        "runtime_hours": (
            overall_runtime_hours
        ),
    }
])

runtime_summary = pd.concat(
    [
        runtime_summary,
        overall_runtime_row,
    ],
    ignore_index=True,
)

# ------------------------------------------------------------
# 11. Save final files
# ------------------------------------------------------------

comparison_seed_results.to_csv(
    final_comparison_output
    / "all_comparison_seed_results.csv",
    index=False,
)

comparison_mean_std.to_csv(
    final_comparison_output
    / "comparison_mean_std.csv",
    index=False,
)

latex_comparison_table.to_csv(
    final_comparison_output
    / "comparison_latex_ready.csv",
    index=False,
)

metric_winners.to_csv(
    final_comparison_output
    / "metric_winners.csv",
    index=False,
)

differences_from_kamba.to_csv(
    final_comparison_output
    / "differences_from_kamba.csv",
    index=False,
)

baseline_runtime_by_method.to_csv(
    final_comparison_output
    / "baseline_runtime_by_method.csv",
    index=False,
)

runtime_summary.to_csv(
    final_comparison_output
    / "three_experiment_runtime.csv",
    index=False,
)

# ------------------------------------------------------------
# 12. Display final results
# ------------------------------------------------------------

print(
    "\nFinal comparison mean results:"
)

display(
    comparison_mean_std[
        [
            "dataset",
            "display_name",
            "accuracy_mean",
            "precision_mean",
            "recall_mean",
            "f1_mean",
            "mcc_mean",
            "auroc_mean",
            "auprc_mean",
            "parameters",
        ]
    ]
)

print(
    "\nLaTeX-ready comparison table:"
)

display(
    latex_comparison_table
)

print("\nMetric winners:")

display(metric_winners)

print(
    "\nDifferences from KAMBA++ "
    "(percentage points):"
)

display(differences_from_kamba)

print(
    "\nBaseline runtime by method:"
)

display(baseline_runtime_by_method)

print(
    "\nRuntime of all three experiments:"
)

display(runtime_summary)

print(
    "\nTotal evaluated runs:",
    140
)

print(
    "Total experimental runtime:",
    f"{overall_runtime_hours:.3f} hours"
)

print("\nSaved to:")

print(final_comparison_output)

Verified baseline runs: 70

Final comparison mean results:


,dataset,display_name,accuracy_mean,precision_mean,recall_mean,f1_mean,mcc_mean,auroc_mean,auprc_mean,parameters
0,DAPT2020,RF,0.990051,0.981891,0.983211,0.982542,0.975594,0.998818,0.997768,NaN
1,DAPT2020,MLP,0.954048,0.875708,0.977335,0.923721,0.893748,0.989375,0.970522,170626.0
2,DAPT2020,CNN,0.958918,0.889634,0.979958,0.931980,0.905507,0.993508,0.979948,75650.0
3,DAPT2020,BiLSTM,0.978309,0.951685,0.973347,0.962354,0.947281,0.997913,0.995322,207618.0
4,DAPT2020,Transformer,0.986137,0.975116,0.976285,0.975673,0.966006,0.998678,0.997397,275202.0
5,DAPT2020,KAN,0.965013,0.902562,0.983421,0.941229,0.918128,0.993306,0.979159,94618.0
6,DAPT2020,Selective SSM,0.968300,0.914419,0.980693,0.946312,0.925090,0.995043,0.984505,62754.0
7,DAPT2020,KAMBA++,0.978877,0.950589,0.977020,0.963515,0.948940,0.997911,0.995048,375721.0
8,CICIDS2017,RF,0.996815,0.996163,0.989963,0.993053,0.990995,0.999291,0.998620,NaN
9,CICIDS2017,MLP,0.910226,0.990859,0.615574,0.758253,0.737819,0.993638,0.979565,180866.0



LaTeX-ready comparison table:


,dataset,method,accuracy,precision,recall,f1,mcc,auroc,auprc,parameters
0,DAPT2020,RF,0.9901\(\pm\)0.0004,0.9819\(\pm\)0.0037,0.9832\(\pm\)0.0028,0.9825\(\pm\)0.0006,0.9756\(\pm\)0.0009,0.9988\(\pm\)0.0001,0.9978\(\pm\)0.0001,500 trees
1,DAPT2020,MLP,0.9540\(\pm\)0.0024,0.8757\(\pm\)0.0026,0.9773\(\pm\)0.0083,0.9237\(\pm\)0.0043,0.8937\(\pm\)0.0063,0.9894\(\pm\)0.0036,0.9705\(\pm\)0.0117,170626
2,DAPT2020,CNN,0.9589\(\pm\)0.0151,0.8896\(\pm\)0.0461,0.9800\(\pm\)0.0086,0.9320\(\pm\)0.0229,0.9055\(\pm\)0.0313,0.9935\(\pm\)0.0040,0.9799\(\pm\)0.0090,75650
3,DAPT2020,BiLSTM,0.9783\(\pm\)0.0056,0.9517\(\pm\)0.0131,0.9733\(\pm\)0.0106,0.9624\(\pm\)0.0097,0.9473\(\pm\)0.0136,0.9979\(\pm\)0.0005,0.9953\(\pm\)0.0012,207618
4,DAPT2020,Transformer,0.9861\(\pm\)0.0013,0.9751\(\pm\)0.0066,0.9763\(\pm\)0.0058,0.9757\(\pm\)0.0023,0.9660\(\pm\)0.0032,0.9987\(\pm\)0.0003,0.9974\(\pm\)0.0005,275202
5,DAPT2020,KAN,0.9650\(\pm\)0.0035,0.9026\(\pm\)0.0106,0.9834\(\pm\)0.0013,0.9412\(\pm\)0.0055,0.9181\(\pm\)0.0075,0.9933\(\pm\)0.0010,0.9792\(\pm\)0.0026,94618
6,DAPT2020,Selective SSM,0.9683\(\pm\)0.0029,0.9144\(\pm\)0.0126,0.9807\(\pm\)0.0085,0.9463\(\pm\)0.0045,0.9251\(\pm\)0.0061,0.9950\(\pm\)0.0008,0.9845\(\pm\)0.0030,62754
7,DAPT2020,KAMBA++,0.9789\(\pm\)0.0061,0.9506\(\pm\)0.0207,0.9770\(\pm\)0.0041,0.9635\(\pm\)0.0102,0.9489\(\pm\)0.0142,0.9979\(\pm\)0.0005,0.9950\(\pm\)0.0015,375721
8,CICIDS2017,RF,0.9968\(\pm\)0.0001,0.9962\(\pm\)0.0004,0.9900\(\pm\)0.0004,0.9931\(\pm\)0.0002,0.9910\(\pm\)0.0002,0.9993\(\pm\)0.0000,0.9986\(\pm\)0.0000,500 trees
9,CICIDS2017,MLP,0.9102\(\pm\)0.0109,0.9909\(\pm\)0.0056,0.6156\(\pm\)0.0515,0.7583\(\pm\)0.0371,0.7378\(\pm\)0.0330,0.9936\(\pm\)0.0005,0.9796\(\pm\)0.0014,180866



Metric winners:


...


Baseline runtime by method:


,dataset,method,count,sum,mean,std
0,CICIDS2017,bilstm,5,21.150102,4.230020,0.833489
1,CICIDS2017,cnn,5,15.456278,3.091256,0.746195
2,CICIDS2017,kan,5,4.430680,0.886136,0.019601
3,CICIDS2017,mlp,5,3.659754,0.731951,0.025861
4,CICIDS2017,rf,5,8.508955,1.701791,0.493869
5,CICIDS2017,selective_ssm,5,11.438061,2.287612,0.034602
6,CICIDS2017,transformer,5,27.275024,5.455005,1.710187
7,DAPT2020,bilstm,5,3.678857,0.735771,0.016954
8,DAPT2020,cnn,5,3.009266,0.601853,0.211952
9,DAPT2020,kan,5,4.415562,0.883112,0.377668



Runtime of all three experiments:


,dataset,experiment,runs,runtime_hours
0,DAPT2020,Baseline comparison,35,0.403855
1,DAPT2020,Full KAMBA++ evaluation,5,0.240402
2,DAPT2020,Ablation study,30,1.405683
3,DAPT2020,Dataset total,70,2.049940
4,CICIDS2017,Baseline comparison,35,1.531981
5,CICIDS2017,Full KAMBA++ evaluation,5,1.378728
6,CICIDS2017,Ablation study,30,7.008504
7,CICIDS2017,Dataset total,70,9.919213
8,Overall,Three experiments,140,11.969153



Total evaluated runs: 140
Total experimental runtime: 11.969 hours

Saved to:
<PROJECT_ROOT>\outputs\final_comparison
